In [ ]:
# ==============================================================================
# CELL 1 — Reload final 9E files for downstream validation
# New separate downstream notebook / new runtime
#
# Purpose:
#   1. Mount Google Drive.
#   2. Locate all final 9E output files.
#   3. Load observed / learned 9E / baseline molecule tables.
#   4. Load Step 4 metadata and denoised AnnData.
#   5. Check row counts and required columns.
#   6. Prepare common dictionaries/variables for later validation cells.
#
# GPU not needed.
# ==============================================================================

import os
import json
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy import sparse

print("=" * 100)
print("CELL 1 — Reload final 9E files for downstream validation")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Mount Drive
# ------------------------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------------------
# 1. Paths and run name
# ------------------------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/diffusion"
STEP4_EXPORT_DIR = os.path.join(BASE_DIR, "step4_exports")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "latest_run")

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

print(f"BASE_DIR        : {BASE_DIR}")
print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"RUN_NAME        : {RUN_NAME}")

# ------------------------------------------------------------------------------
# 2. Final file paths
# ------------------------------------------------------------------------------

paths = {
    # Step 4 / preprocessing files
    "step4_config": os.path.join(STEP4_EXPORT_DIR, "step4_config.json"),
    "denoised_adata": os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad"),
    "was_corrected": os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy"),
    "cell_data": os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"),
    "original_molecules": os.path.join(STEP4_EXPORT_DIR, "molecules.parquet"),

    # Step 5 processed observed molecule table
    "step5_mol_processed": os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet"),

    # Learned 9E outputs
    "learned_imputed": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_imputed_records.parquet"
    ),
    "learned_completed": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_completed_molecule_table.parquet"
    ),
    "imputation_targets": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_imputation_targets.csv"
    ),
    "count_reconciliation": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_count_reconciliation.csv"
    ),
    "imputation_summary": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_imputation_summary.csv"
    ),

    # Empirical baseline outputs
    "gene_emp_imputed": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_gene_emp_imputed_records.parquet"
    ),
    "ct_gene_emp_imputed": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
    ),
    "spatial_knn_emp_imputed": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"
    ),
    "baseline_summary": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_baseline_imputation_summary.csv"
    ),

    # Final recovery evaluation / previous downstream outputs
    "final_large_recovery_metrics": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_final_large_recovery_metrics.csv"
    ),
    "final_large_recovery_report": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_final_large_recovery_report.txt"
    ),
    "downstream_clustering_metrics": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
    ),
    "downstream_count_metrics": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
    ),
}

# ------------------------------------------------------------------------------
# 3. Verify files exist
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Checking required files")
print("=" * 100)

required_keys = [
    "step4_config",
    "denoised_adata",
    "was_corrected",
    "cell_data",
    "original_molecules",
    "step5_mol_processed",
    "learned_imputed",
    "learned_completed",
    "imputation_targets",
    "count_reconciliation",
    "imputation_summary",
    "gene_emp_imputed",
    "ct_gene_emp_imputed",
    "spatial_knn_emp_imputed",
    "baseline_summary",
    "final_large_recovery_metrics",
]

missing = []

for key in required_keys:
    path = paths[key]
    exists = os.path.exists(path)

    if exists:
        size_gb = os.path.getsize(path) / 1e9
        print(f"  ✓ {key:32s} {size_gb:8.3f} GB  {path}")
    else:
        print(f"  ✗ {key:32s} MISSING  {path}")
        missing.append(key)

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("\nAll required files exist.")

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    """Convert sparse matrix to dense numpy array if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_parquet_nrows(path):
    """Read Parquet metadata row count without loading full file."""
    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        return int(pf.metadata.num_rows), int(pf.metadata.num_columns)
    except Exception as e:
        print(f"Could not read parquet metadata for {path}: {e}")
        return None, None


def require_columns(df, required_cols, name):
    """Check if a dataframe has required columns."""
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"{name} is missing required columns: {missing_cols}")
    print(f"  ✓ {name}: required columns present")


# ------------------------------------------------------------------------------
# 5. Fast Parquet row-count check before loading
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Fast Parquet row-count checks")
print("=" * 100)

expected_parquet_rows = {
    "step5_mol_processed": 23_171_103,
    "learned_imputed": 4_638_214,
    "learned_completed": 27_809_317,
    "gene_emp_imputed": 4_638_214,
    "ct_gene_emp_imputed": 4_638_214,
    "spatial_knn_emp_imputed": 4_638_214,
}

row_check_records = []

for key, expected_rows in expected_parquet_rows.items():
    path = paths[key]
    n_rows, n_cols = get_parquet_nrows(path)

    ok = (n_rows == expected_rows)

    row_check_records.append({
        "file_key": key,
        "expected_rows": expected_rows,
        "actual_rows": n_rows,
        "n_cols": n_cols,
        "ok": ok,
        "path": path,
    })

    print(
        f"{'✓' if ok else '✗'} {key:24s} "
        f"expected={expected_rows:,} actual={n_rows:,} cols={n_cols}"
    )

row_check_df = pd.DataFrame(row_check_records)

if not row_check_df["ok"].all():
    print("\nWARNING: Some row counts do not match expected values.")
    display(row_check_df)
    raise RuntimeError("Row-count check failed. Verify files before continuing.")

print("\nAll important Parquet row counts match expected values.")

# ------------------------------------------------------------------------------
# 6. Load Step 4 config and AnnData
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading Step 4 config and AnnData")
print("=" * 100)

with open(paths["step4_config"], "r") as f:
    step4_config = json.load(f)

shared_genes = list(step4_config["shared_genes"])
n_cells_step4 = int(step4_config["n_cells"])
n_genes_step4 = int(step4_config["n_genes"])
cell_type_column = step4_config.get("cell_type_column", "cell_type")

print(f"Shared genes: {len(shared_genes):,}")
print(f"Step4 cells : {n_cells_step4:,}")
print(f"Step4 genes : {n_genes_step4:,}")
print(f"Cell type column from config: {cell_type_column}")
print(f"First 10 genes: {shared_genes[:10]}")

# Load AnnData.
try:
    import anndata as ad
except Exception:
    !pip install -q anndata
    import anndata as ad

denoised_adata = ad.read_h5ad(paths["denoised_adata"])

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"denoised_adata.obs columns: {list(denoised_adata.obs.columns)[:15]}")
print(f"denoised_adata.var_names first 10: {list(denoised_adata.var_names[:10])}")

# Get Step4 matrices.
X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

# Prefer corrected raw layer if present, otherwise use molecule-derived matrix later.
if "raw" in denoised_adata.layers:
    X_raw_from_adata = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
else:
    X_raw_from_adata = None
    print("WARNING: denoised_adata.layers['raw'] not found.")

was_corrected = np.load(paths["was_corrected"])

print(f"X_denoised shape: {X_denoised.shape}")
if X_raw_from_adata is not None:
    print(f"X_raw_from_adata shape: {X_raw_from_adata.shape}")
print(f"was_corrected shape: {was_corrected.shape}")

if X_denoised.shape != (n_cells_step4, n_genes_step4):
    raise ValueError(f"X_denoised shape mismatch: {X_denoised.shape}")

if was_corrected.shape != X_denoised.shape:
    raise ValueError(f"was_corrected shape mismatch: {was_corrected.shape} vs {X_denoised.shape}")

# Cell IDs.
if "cell_id" in denoised_adata.obs.columns:
    cell_ids_step4 = denoised_adata.obs["cell_id"].astype(int).values
else:
    # Fallback: obs names may be cell IDs.
    cell_ids_step4 = denoised_adata.obs_names.astype(int).values

print(f"cell_ids_step4 length: {len(cell_ids_step4):,}")
print(f"first 10 cell_ids_step4: {cell_ids_step4[:10]}")

# Cell type labels.
if cell_type_column in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs[cell_type_column].astype(str).values
elif "cell_type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["cell_type"].astype(str).values
elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["Assigned_Xenium_Cell_Type"].astype(str).values
else:
    raise KeyError("Could not find cell type column in denoised_adata.obs.")

unique_cell_types = sorted(pd.unique(cell_type_labels).tolist())

print(f"Unique cell types: {len(unique_cell_types)}")
print(unique_cell_types)

# ------------------------------------------------------------------------------
# 7. Load main molecule tables
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading final molecule tables")
print("=" * 100)

# Observed processed molecule table from Step 5 before imputation.
print("\nLoading observed processed molecule table:")
print(paths["step5_mol_processed"])
mol_observed = pd.read_parquet(paths["step5_mol_processed"])

print(f"mol_observed shape: {mol_observed.shape}")
print("mol_observed status counts:")
display(mol_observed["status"].value_counts(dropna=False).reset_index())

# Learned 9E imputed molecules.
print("\nLoading learned 9E imputed molecule table:")
print(paths["learned_imputed"])
learned_imputed_df = pd.read_parquet(paths["learned_imputed"])

print(f"learned_imputed_df shape: {learned_imputed_df.shape}")

# Completed learned 9E molecule table.
# This is large but useful for some downstream analyses.
print("\nLoading learned 9E completed molecule table:")
print(paths["learned_completed"])
learned_completed_mol = pd.read_parquet(paths["learned_completed"])

print(f"learned_completed_mol shape: {learned_completed_mol.shape}")
print("learned_completed_mol status counts:")
display(learned_completed_mol["status"].value_counts(dropna=False).reset_index())

# Baseline imputed molecule tables.
print("\nLoading baseline imputed molecule tables...")

gene_emp_imputed_df = pd.read_parquet(paths["gene_emp_imputed"])
ct_gene_emp_imputed_df = pd.read_parquet(paths["ct_gene_emp_imputed"])
spatial_knn_emp_imputed_df = pd.read_parquet(paths["spatial_knn_emp_imputed"])

baseline_imputed_dfs = {
    "gene_emp": gene_emp_imputed_df,
    "ct_gene_emp": ct_gene_emp_imputed_df,
    "spatial_knn_emp": spatial_knn_emp_imputed_df,
}

print(f"gene_emp_imputed_df shape       : {gene_emp_imputed_df.shape}")
print(f"ct_gene_emp_imputed_df shape    : {ct_gene_emp_imputed_df.shape}")
print(f"spatial_knn_emp_imputed_df shape: {spatial_knn_emp_imputed_df.shape}")

# ------------------------------------------------------------------------------
# 8. Load CSV summaries / previous metrics
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Loading summary and metric files")
print("=" * 100)

imputation_targets_df = pd.read_csv(paths["imputation_targets"])
count_reconciliation_df = pd.read_csv(paths["count_reconciliation"])
imputation_summary_df = pd.read_csv(paths["imputation_summary"])
baseline_summary_df = pd.read_csv(paths["baseline_summary"])
final_large_recovery_metrics_df = pd.read_csv(paths["final_large_recovery_metrics"])

print(f"imputation_targets_df shape        : {imputation_targets_df.shape}")
print(f"count_reconciliation_df shape      : {count_reconciliation_df.shape}")
print(f"imputation_summary_df shape        : {imputation_summary_df.shape}")
print(f"baseline_summary_df shape          : {baseline_summary_df.shape}")
print(f"final_large_recovery_metrics shape : {final_large_recovery_metrics_df.shape}")

display(imputation_summary_df)
display(baseline_summary_df)
display(final_large_recovery_metrics_df)

# Optional previously completed downstream metrics.
if os.path.exists(paths["downstream_clustering_metrics"]):
    downstream_clustering_df = pd.read_csv(paths["downstream_clustering_metrics"])
    print("\nLoaded previous downstream clustering metrics:")
    display(downstream_clustering_df)
else:
    downstream_clustering_df = None
    print("\nPrevious downstream clustering metrics not found.")

if os.path.exists(paths["downstream_count_metrics"]):
    downstream_count_metrics_df = pd.read_csv(paths["downstream_count_metrics"])
    print("\nLoaded previous downstream count metrics:")
    display(downstream_count_metrics_df)
else:
    downstream_count_metrics_df = None
    print("\nPrevious downstream count metrics not found.")

# ------------------------------------------------------------------------------
# 9. Required column checks
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Checking required columns")
print("=" * 100)

common_molecule_cols = [
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "is_imputed",
]

imputed_extra_cols = [
    "imputation_model",
    "imputation_confidence",
]

require_columns(mol_observed, common_molecule_cols, "mol_observed")
require_columns(learned_imputed_df, common_molecule_cols + imputed_extra_cols, "learned_imputed_df")
require_columns(learned_completed_mol, common_molecule_cols, "learned_completed_mol")
require_columns(gene_emp_imputed_df, common_molecule_cols, "gene_emp_imputed_df")
require_columns(ct_gene_emp_imputed_df, common_molecule_cols, "ct_gene_emp_imputed_df")
require_columns(spatial_knn_emp_imputed_df, common_molecule_cols, "spatial_knn_emp_imputed_df")

# ------------------------------------------------------------------------------
# 10. Count sanity checks
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Count sanity checks")
print("=" * 100)

n_observed = int((learned_completed_mol["status"].astype(str) == "observed").sum())
n_imputed = int((learned_completed_mol["status"].astype(str) == "imputed").sum())
n_completed = len(learned_completed_mol)

print(f"Observed in completed table: {n_observed:,}")
print(f"Imputed in completed table : {n_imputed:,}")
print(f"Completed table total      : {n_completed:,}")
print(f"Observed + imputed         : {n_observed + n_imputed:,}")

if n_observed != 23_171_103:
    raise RuntimeError(f"Unexpected observed count: {n_observed:,}")

if n_imputed != 4_638_214:
    raise RuntimeError(f"Unexpected learned imputed count: {n_imputed:,}")

if n_completed != 27_809_317:
    raise RuntimeError(f"Unexpected completed molecule count: {n_completed:,}")

# Count reconciliation.
if "diff" in count_reconciliation_df.columns:
    n_bad = int((count_reconciliation_df["diff"] != 0).sum())
    max_abs_diff = int(count_reconciliation_df["diff"].abs().max())
    print(f"Count reconciliation mismatched pairs: {n_bad:,}")
    print(f"Count reconciliation max abs diff    : {max_abs_diff}")

    if n_bad != 0 or max_abs_diff != 0:
        raise RuntimeError("Count reconciliation failed.")
else:
    raise KeyError("count_reconciliation_df does not contain 'diff' column.")

print("Count sanity checks passed.")

# ------------------------------------------------------------------------------
# 11. Build useful maps for later cells
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Preparing common lookup maps for later validation cells")
print("=" * 100)

gene_to_col = {str(g): i for i, g in enumerate(shared_genes)}
col_to_gene = {i: str(g) for i, g in enumerate(shared_genes)}

step4_cell_to_row = {int(cid): i for i, cid in enumerate(cell_ids_step4)}
row_to_cell_id = {i: int(cid) for i, cid in enumerate(cell_ids_step4)}

cell_type_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = np.array([cell_type_to_idx[str(ct)] for ct in cell_type_labels], dtype=np.int64)

print(f"gene_to_col entries      : {len(gene_to_col):,}")
print(f"step4_cell_to_row entries: {len(step4_cell_to_row):,}")
print(f"cell types               : {len(cell_type_to_idx):,}")

# ------------------------------------------------------------------------------
# 12. Basic coordinate range sanity checks
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Basic coordinate range sanity checks")
print("=" * 100)

def coordinate_sanity(df, name):
    print(f"\n{name}")
    print("-" * 80)
    print(f"Rows: {len(df):,}")

    for col in ["r_norm", "theta", "z_rel", "p_nuclear"]:
        vals = pd.to_numeric(df[col], errors="coerce")
        print(
            f"{col:10s}: "
            f"min={vals.min(): .4f}, "
            f"max={vals.max(): .4f}, "
            f"mean={vals.mean(): .4f}, "
            f"missing={vals.isna().sum():,}"
        )

coordinate_sanity(learned_imputed_df, "Learned 9E imputed")
coordinate_sanity(gene_emp_imputed_df, "Gene empirical imputed")
coordinate_sanity(ct_gene_emp_imputed_df, "Cell-type gene empirical imputed")
coordinate_sanity(spatial_knn_emp_imputed_df, "Spatial-kNN empirical imputed")

# ------------------------------------------------------------------------------
# 13. Save reload manifest for this downstream notebook
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Saving reload manifest")
print("=" * 100)

reload_manifest = {
    "run_name": RUN_NAME,
    "base_dir": BASE_DIR,
    "step4_export_dir": STEP4_EXPORT_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "paths": paths,
    "n_cells": int(len(cell_ids_step4)),
    "n_genes": int(len(shared_genes)),
    "n_cell_types": int(len(unique_cell_types)),
    "n_observed_molecules": int(n_observed),
    "n_learned_imputed_molecules": int(len(learned_imputed_df)),
    "n_completed_molecules": int(len(learned_completed_mol)),
    "n_gene_emp_imputed_molecules": int(len(gene_emp_imputed_df)),
    "n_ct_gene_emp_imputed_molecules": int(len(ct_gene_emp_imputed_df)),
    "n_spatial_knn_emp_imputed_molecules": int(len(spatial_knn_emp_imputed_df)),
    "count_reconciliation_mismatched_pairs": int(n_bad),
    "count_reconciliation_max_abs_diff": int(max_abs_diff),
}

RELOAD_MANIFEST_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_new_downstream_reload_manifest.json"
)

with open(RELOAD_MANIFEST_PATH, "w") as f:
    json.dump(reload_manifest, f, indent=2)

print(f"Saved reload manifest:")
print(f"  {RELOAD_MANIFEST_PATH}")

# ------------------------------------------------------------------------------
# 14. Final status
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 1 COMPLETE — final 9E files loaded successfully")
print("=" * 100)

print("""
Available main variables for next cells:

  mol_observed
  learned_imputed_df
  learned_completed_mol
  gene_emp_imputed_df
  ct_gene_emp_imputed_df
  spatial_knn_emp_imputed_df
  baseline_imputed_dfs

  denoised_adata
  X_denoised
  X_raw_from_adata
  was_corrected
  shared_genes
  cell_ids_step4
  cell_type_labels
  unique_cell_types

  imputation_targets_df
  count_reconciliation_df
  imputation_summary_df
  baseline_summary_df
  final_large_recovery_metrics_df

  gene_to_col
  step4_cell_to_row
  cell_type_to_idx
""")

gc.collect()

===========COUNT-LEVEL VALIDATION============

In [ ]:
# ==============================================================================
# CELL 2 — Statistical power / analyzable cell-gene pair gain
#
# Biological question:
#   Does 9E imputation make more cell-gene pairs usable for downstream analyses?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 100)
print("CELL 2 — Statistical power / analyzable cell-gene pair gain")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from Cell 1
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "X_raw_from_adata",
    "learned_imputed_df",
    "shared_genes",
    "cell_ids_step4",
    "cell_type_labels",
    "gene_to_col",
    "step4_cell_to_row",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from Cell 1: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

POWER_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_power_gain_summary.csv"
)

POWER_BY_GENE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_power_gain_by_gene.csv"
)

POWER_BY_CELLTYPE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_power_gain_by_celltype.csv"
)

# ------------------------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def build_completed_count_matrix_from_imputed(X_raw, imputed_df):
    """
    Builds:
        X_completed = X_raw + imputed molecule counts per (cell_id, gene_id)

    This uses only the cell_id and gene_id of imputed molecules.
    """
    X_completed = np.asarray(X_raw, dtype=np.float32).copy()

    required_cols = ["cell_id", "gene_id"]
    missing = [c for c in required_cols if c not in imputed_df.columns]
    if missing:
        raise KeyError(f"learned_imputed_df missing columns: {missing}")

    counts = (
        imputed_df[["cell_id", "gene_id"]]
        .copy()
        .assign(
            cell_id=lambda d: d["cell_id"].astype(int),
            gene_id=lambda d: d["gene_id"].astype(str)
        )
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="n_imputed")
    )

    rr = counts["cell_id"].map(step4_cell_to_row)
    cc = counts["gene_id"].map(gene_to_col)

    ok = rr.notna() & cc.notna()

    if int(ok.sum()) < len(counts):
        print(f"WARNING: dropped {len(counts) - int(ok.sum()):,} unmapped imputed count rows.")

    X_completed[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy()
    ] += counts.loc[ok, "n_imputed"].to_numpy(dtype=np.float32)

    return X_completed


def analyzable_summary(X, thresholds, label):
    """
    Count how many cell-gene entries are analyzable at each threshold.
    """
    rows = []

    total_pairs = X.shape[0] * X.shape[1]

    for t in thresholds:
        n_pairs = int((X >= t).sum())
        frac_pairs = n_pairs / total_pairs

        rows.append({
            "dataset": label,
            "threshold_min_count": int(t),
            "analyzable_cell_gene_pairs": n_pairs,
            "fraction_of_all_cell_gene_pairs": frac_pairs,
            "total_cell_gene_pairs": int(total_pairs),
        })

    return rows


def analyzable_by_gene(X_raw, X_completed, thresholds):
    """
    For each gene, count how many cells become analyzable.
    """
    rows = []

    for j, gene in enumerate(shared_genes):
        raw_gene_counts = X_raw[:, j]
        comp_gene_counts = X_completed[:, j]

        for t in thresholds:
            raw_n = int((raw_gene_counts >= t).sum())
            comp_n = int((comp_gene_counts >= t).sum())
            gain = comp_n - raw_n

            rows.append({
                "gene_id": str(gene),
                "threshold_min_count": int(t),
                "raw_analyzable_cells": raw_n,
                "completed_analyzable_cells": comp_n,
                "gain_cells": gain,
                "gain_fraction_of_all_cells": gain / X_raw.shape[0],
                "raw_fraction_cells": raw_n / X_raw.shape[0],
                "completed_fraction_cells": comp_n / X_raw.shape[0],
            })

    return pd.DataFrame(rows)


def analyzable_by_celltype(X_raw, X_completed, thresholds):
    """
    For each cell type, count how many cell-gene pairs become analyzable.
    """
    rows = []

    ct_series = pd.Series(cell_type_labels.astype(str))
    unique_cts = sorted(ct_series.unique())

    for ct in unique_cts:
        mask = (ct_series.values == ct)
        n_cells_ct = int(mask.sum())

        Xr = X_raw[mask, :]
        Xc = X_completed[mask, :]

        total_pairs_ct = Xr.shape[0] * Xr.shape[1]

        for t in thresholds:
            raw_n = int((Xr >= t).sum())
            comp_n = int((Xc >= t).sum())
            gain = comp_n - raw_n

            rows.append({
                "cell_type": ct,
                "n_cells": n_cells_ct,
                "threshold_min_count": int(t),
                "raw_analyzable_cell_gene_pairs": raw_n,
                "completed_analyzable_cell_gene_pairs": comp_n,
                "gain_cell_gene_pairs": gain,
                "gain_fraction_of_celltype_pairs": gain / total_pairs_ct if total_pairs_ct > 0 else np.nan,
                "raw_fraction_celltype_pairs": raw_n / total_pairs_ct if total_pairs_ct > 0 else np.nan,
                "completed_fraction_celltype_pairs": comp_n / total_pairs_ct if total_pairs_ct > 0 else np.nan,
            })

    return pd.DataFrame(rows)

# ------------------------------------------------------------------------------
# 3. Build raw and completed count matrices
# ------------------------------------------------------------------------------

print("\nBuilding raw and learned 9E completed count matrices...")

X_raw_counts = ensure_dense(X_raw_from_adata).astype(np.float32)

X_completed_9E = build_completed_count_matrix_from_imputed(
    X_raw=X_raw_counts,
    imputed_df=learned_imputed_df,
)

print(f"X_raw_counts shape   : {X_raw_counts.shape}")
print(f"X_completed_9E shape : {X_completed_9E.shape}")
print(f"Raw total counts     : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Completed total counts: {X_completed_9E.sum(dtype=np.float64):,.0f}")
print(f"Added counts         : {(X_completed_9E - X_raw_counts).sum(dtype=np.float64):,.0f}")

# ------------------------------------------------------------------------------
# 4. Overall analyzable pair gain
# ------------------------------------------------------------------------------

thresholds = [1, 2, 3, 5, 8, 10]

print("\nComputing overall analyzable pair gain...")

summary_rows = []
summary_rows.extend(analyzable_summary(X_raw_counts, thresholds, "Raw observed counts"))
summary_rows.extend(analyzable_summary(X_completed_9E, thresholds, "Learned 9E completed counts"))

power_summary_df = pd.DataFrame(summary_rows)

# Add paired gain table.
gain_rows = []

for t in thresholds:
    raw_row = power_summary_df[
        (power_summary_df["dataset"] == "Raw observed counts")
        & (power_summary_df["threshold_min_count"] == t)
    ].iloc[0]

    comp_row = power_summary_df[
        (power_summary_df["dataset"] == "Learned 9E completed counts")
        & (power_summary_df["threshold_min_count"] == t)
    ].iloc[0]

    raw_n = int(raw_row["analyzable_cell_gene_pairs"])
    comp_n = int(comp_row["analyzable_cell_gene_pairs"])
    gain = comp_n - raw_n

    gain_rows.append({
        "threshold_min_count": int(t),
        "raw_analyzable_pairs": raw_n,
        "completed_analyzable_pairs": comp_n,
        "gain_pairs": gain,
        "relative_gain_percent": 100.0 * gain / max(raw_n, 1),
        "raw_fraction_all_pairs": float(raw_row["fraction_of_all_cell_gene_pairs"]),
        "completed_fraction_all_pairs": float(comp_row["fraction_of_all_cell_gene_pairs"]),
    })

power_gain_summary_df = pd.DataFrame(gain_rows)

print("\nOverall analyzable pair gain:")
display(power_gain_summary_df)

# ------------------------------------------------------------------------------
# 5. By-gene analyzable gain
# ------------------------------------------------------------------------------

print("\nComputing by-gene analyzable gain...")

power_by_gene_df = analyzable_by_gene(
    X_raw=X_raw_counts,
    X_completed=X_completed_9E,
    thresholds=thresholds,
)

print("\nTop genes by gain at threshold >=5:")
display(
    power_by_gene_df[power_by_gene_df["threshold_min_count"] == 5]
    .sort_values("gain_cells", ascending=False)
    .head(20)
)

# ------------------------------------------------------------------------------
# 6. By-cell-type analyzable gain
# ------------------------------------------------------------------------------

print("\nComputing by-cell-type analyzable gain...")

power_by_celltype_df = analyzable_by_celltype(
    X_raw=X_raw_counts,
    X_completed=X_completed_9E,
    thresholds=thresholds,
)

print("\nCell-type gains at threshold >=5:")
display(
    power_by_celltype_df[power_by_celltype_df["threshold_min_count"] == 5]
    .sort_values("gain_cell_gene_pairs", ascending=False)
)

# ------------------------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------------------------

power_gain_summary_df.to_csv(POWER_SUMMARY_PATH, index=False)
power_by_gene_df.to_csv(POWER_BY_GENE_PATH, index=False)
power_by_celltype_df.to_csv(POWER_BY_CELLTYPE_PATH, index=False)

print("\nSaved statistical power / analyzable-pair outputs:")
print(f"  Overall summary : {POWER_SUMMARY_PATH}")
print(f"  By gene         : {POWER_BY_GENE_PATH}")
print(f"  By cell type    : {POWER_BY_CELLTYPE_PATH}")

print("\n" + "=" * 100)
print("CELL 2 COMPLETE — statistical power / analyzable pair gain")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 3 — Marker-based cell-type signal strengthening
#
# Biological question:
#   Do known marker genes become clearer in their expected cell types after
#   learned 9E imputation?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import roc_auc_score

print("=" * 100)
print("CELL 3 — Marker-based cell-type signal strengthening")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables from Cell 1/2
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "shared_genes",
    "cell_type_labels",
    "unique_cell_types",
    "gene_to_col",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

if "X_raw_counts" not in globals():
    if "X_raw_from_adata" not in globals():
        raise NameError("Missing X_raw_counts and X_raw_from_adata.")
    X_raw_counts = X_raw_from_adata.toarray() if sparse.issparse(X_raw_from_adata) else np.asarray(X_raw_from_adata)
    X_raw_counts = X_raw_counts.astype(np.float32)

if "X_completed_9E" not in globals():
    if "learned_imputed_df" not in globals():
        raise NameError("Missing X_completed_9E and learned_imputed_df.")
    print("X_completed_9E not found; rebuilding from learned_imputed_df.")

    X_completed_9E = X_raw_counts.astype(np.float32).copy()

    counts = (
        learned_imputed_df[["cell_id", "gene_id"]]
        .copy()
        .assign(
            cell_id=lambda d: d["cell_id"].astype(int),
            gene_id=lambda d: d["gene_id"].astype(str)
        )
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="n_imputed")
    )

    rr = counts["cell_id"].map(step4_cell_to_row)
    cc = counts["gene_id"].map(gene_to_col)
    ok = rr.notna() & cc.notna()

    X_completed_9E[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy()
    ] += counts.loc[ok, "n_imputed"].to_numpy(dtype=np.float32)

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

MARKER_SIGNAL_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_signal_strengthening.csv"
)

MARKER_SIGNAL_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_signal_strengthening_summary.csv"
)

# ------------------------------------------------------------------------------
# 2. Marker dictionary
# ------------------------------------------------------------------------------

# This dictionary is intentionally broad.
# The code will automatically keep only genes that exist in your 313-gene panel
# and cell types that exist in your data.

marker_sets = {
    "B_Cells": [
        "MS4A1", "CD79A", "CD79B", "BANK1", "CD19", "CD22", "TNFRSF17"
    ],

    "CD4+_T_Cells": [
        "CD3D", "CD3E", "CD4", "IL7R", "CCR7", "TCF7", "LTB"
    ],

    "CD8+_T_Cells": [
        "CD3D", "CD3E", "CD8A", "CD8B", "GZMB", "NKG7", "PRF1"
    ],

    "Macrophages_1": [
        "CD68", "CD163", "LYZ", "TYROBP", "FCGR3A", "LST1", "MNDA"
    ],

    "Macrophages_2": [
        "CD68", "CD163", "LYZ", "TYROBP", "FCGR3A", "LST1", "MNDA"
    ],

    "Endothelial": [
        "PECAM1", "VWF", "KDR", "CLDN5", "RAMP2", "NOSTRIN", "CAV1"
    ],

    "Stromal": [
        "LUM", "DCN", "COL1A1", "COL1A2", "POSTN", "DPT", "FBLN1", "CXCL12"
    ],

    "DCIS_1": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "DCIS_2": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "Invasive_Tumor": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "Prolif_Invasive_Tumor": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "MKI67", "TOP2A", "CCND1"
    ],

    "IRF7+_DCs": [
        "IRF7", "ITGAX", "LAMP3", "CLEC10A", "FCER1A"
    ],

    "LAMP3+_DCs": [
        "LAMP3", "ITGAX", "IRF7", "CLEC10A", "FCER1A"
    ],
}

available_genes = set(str(g) for g in shared_genes)
available_cell_types = set(str(ct) for ct in unique_cell_types)

filtered_marker_sets = {}

for ct, genes in marker_sets.items():
    if ct not in available_cell_types:
        continue

    genes_present = [g for g in genes if g in available_genes]

    if len(genes_present) > 0:
        filtered_marker_sets[ct] = genes_present

print("\nFiltered marker sets present in this dataset:")
for ct, genes in filtered_marker_sets.items():
    print(f"  {ct:25s}: {genes}")

if len(filtered_marker_sets) == 0:
    raise RuntimeError("No marker genes from marker_sets were found in this dataset.")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def log1p_normalize_counts(X):
    """
    Library-size normalize then log1p transform.
    This makes marker expression more comparable across cells.
    """
    X = np.asarray(X, dtype=np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm)

    return X_log.astype(np.float32)


def safe_auroc(y_true, scores):
    """
    AUROC can fail if y_true has only one class.
    """
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, scores))
    except Exception:
        return np.nan


def cohens_d(x_pos, x_neg):
    """
    Cohen's d effect size.
    """
    x_pos = np.asarray(x_pos, dtype=np.float32)
    x_neg = np.asarray(x_neg, dtype=np.float32)

    n1 = len(x_pos)
    n0 = len(x_neg)

    if n1 < 2 or n0 < 2:
        return np.nan

    m1 = np.mean(x_pos)
    m0 = np.mean(x_neg)

    v1 = np.var(x_pos, ddof=1)
    v0 = np.var(x_neg, ddof=1)

    pooled = np.sqrt(((n1 - 1) * v1 + (n0 - 1) * v0) / max(n1 + n0 - 2, 1))

    if pooled <= 1e-8:
        return np.nan

    return float((m1 - m0) / pooled)


def marker_signal_for_matrix(X_log, X_raw_counts_for_detection, dataset_name):
    """
    Compute marker signal metrics for one matrix.
    X_log: normalized log expression matrix.
    X_raw_counts_for_detection: unnormalized count matrix for detection rate.
    """
    rows = []

    ct_labels = np.asarray(cell_type_labels).astype(str)

    for expected_ct, marker_genes in filtered_marker_sets.items():
        expected_mask = (ct_labels == expected_ct)
        background_mask = ~expected_mask

        n_expected = int(expected_mask.sum())
        n_background = int(background_mask.sum())

        if n_expected == 0 or n_background == 0:
            continue

        y_true = expected_mask.astype(int)

        for gene in marker_genes:
            j = gene_to_col[gene]

            expr = X_log[:, j]
            raw_counts_gene = X_raw_counts_for_detection[:, j]

            expected_expr = expr[expected_mask]
            background_expr = expr[background_mask]

            expected_counts = raw_counts_gene[expected_mask]
            background_counts = raw_counts_gene[background_mask]

            mean_expected = float(np.mean(expected_expr))
            mean_background = float(np.mean(background_expr))

            detection_expected = float((expected_counts > 0).mean())
            detection_background = float((background_counts > 0).mean())

            # Signal-to-background ratio on original normalized-log scale is not ideal,
            # so use both ratio and difference.
            signal_diff = mean_expected - mean_background
            signal_ratio = (mean_expected + 1e-6) / (mean_background + 1e-6)

            # log2 fold-change based on mean normalized-log values.
            log2_fc_like = float(np.log2((mean_expected + 1e-6) / (mean_background + 1e-6)))

            auc = safe_auroc(y_true, expr)
            d = cohens_d(expected_expr, background_expr)

            rows.append({
                "dataset": dataset_name,
                "expected_cell_type": expected_ct,
                "gene_id": gene,
                "n_expected_cells": n_expected,
                "n_background_cells": n_background,

                "mean_expr_expected": mean_expected,
                "mean_expr_background": mean_background,
                "signal_diff_expected_minus_background": signal_diff,
                "signal_ratio_expected_over_background": signal_ratio,
                "log2fc_like_expected_vs_background": log2_fc_like,

                "detection_rate_expected": detection_expected,
                "detection_rate_background": detection_background,
                "detection_diff_expected_minus_background": detection_expected - detection_background,

                "AUROC_expected_vs_background": auc,
                "cohens_d_expected_vs_background": d,
            })

    return pd.DataFrame(rows)

# ------------------------------------------------------------------------------
# 4. Normalize raw and completed matrices
# ------------------------------------------------------------------------------

print("\nNormalizing raw and completed count matrices...")

X_raw_log = log1p_normalize_counts(X_raw_counts)
X_completed_log = log1p_normalize_counts(X_completed_9E)

print(f"X_raw_log shape       : {X_raw_log.shape}")
print(f"X_completed_log shape : {X_completed_log.shape}")

# ------------------------------------------------------------------------------
# 5. Compute marker signal metrics
# ------------------------------------------------------------------------------

print("\nComputing marker signal metrics...")

raw_marker_df = marker_signal_for_matrix(
    X_log=X_raw_log,
    X_raw_counts_for_detection=X_raw_counts,
    dataset_name="Raw observed counts",
)

completed_marker_df = marker_signal_for_matrix(
    X_log=X_completed_log,
    X_raw_counts_for_detection=X_completed_9E,
    dataset_name="Learned 9E completed counts",
)

marker_signal_df = pd.concat([raw_marker_df, completed_marker_df], ignore_index=True)

print(f"Raw marker rows      : {len(raw_marker_df):,}")
print(f"Completed marker rows: {len(completed_marker_df):,}")

display(marker_signal_df.head(20))

# ------------------------------------------------------------------------------
# 6. Build raw-vs-completed gain table
# ------------------------------------------------------------------------------

print("\nComputing raw-vs-completed marker signal gains...")

merge_keys = ["expected_cell_type", "gene_id"]

raw_renamed = raw_marker_df.rename(columns={
    c: f"raw_{c}" for c in raw_marker_df.columns if c not in merge_keys
})

completed_renamed = completed_marker_df.rename(columns={
    c: f"completed_{c}" for c in completed_marker_df.columns if c not in merge_keys
})

marker_gain_df = raw_renamed.merge(
    completed_renamed,
    on=merge_keys,
    how="inner",
)

# Add improvement columns.
metric_pairs = [
    "mean_expr_expected",
    "mean_expr_background",
    "signal_diff_expected_minus_background",
    "signal_ratio_expected_over_background",
    "log2fc_like_expected_vs_background",
    "detection_rate_expected",
    "detection_rate_background",
    "detection_diff_expected_minus_background",
    "AUROC_expected_vs_background",
    "cohens_d_expected_vs_background",
]

for m in metric_pairs:
    raw_col = f"raw_{m}"
    comp_col = f"completed_{m}"

    if raw_col in marker_gain_df.columns and comp_col in marker_gain_df.columns:
        marker_gain_df[f"delta_{m}"] = marker_gain_df[comp_col] - marker_gain_df[raw_col]

# A simple combined improvement flag.
marker_gain_df["improved_AUROC"] = marker_gain_df["delta_AUROC_expected_vs_background"] > 0
marker_gain_df["improved_signal_diff"] = marker_gain_df["delta_signal_diff_expected_minus_background"] > 0
marker_gain_df["improved_detection_diff"] = marker_gain_df["delta_detection_diff_expected_minus_background"] > 0

print("\nTop marker improvements by AUROC gain:")
display(
    marker_gain_df
    .sort_values("delta_AUROC_expected_vs_background", ascending=False)
    .head(20)
)

print("\nTop marker improvements by signal-difference gain:")
display(
    marker_gain_df
    .sort_values("delta_signal_diff_expected_minus_background", ascending=False)
    .head(20)
)

# ------------------------------------------------------------------------------
# 7. Summary statistics
# ------------------------------------------------------------------------------

summary = {
    "n_marker_tests": int(len(marker_gain_df)),

    "mean_delta_AUROC": float(marker_gain_df["delta_AUROC_expected_vs_background"].mean()),
    "median_delta_AUROC": float(marker_gain_df["delta_AUROC_expected_vs_background"].median()),
    "fraction_markers_AUROC_improved": float(marker_gain_df["improved_AUROC"].mean()),

    "mean_delta_signal_diff": float(marker_gain_df["delta_signal_diff_expected_minus_background"].mean()),
    "median_delta_signal_diff": float(marker_gain_df["delta_signal_diff_expected_minus_background"].median()),
    "fraction_markers_signal_diff_improved": float(marker_gain_df["improved_signal_diff"].mean()),

    "mean_delta_detection_diff": float(marker_gain_df["delta_detection_diff_expected_minus_background"].mean()),
    "median_delta_detection_diff": float(marker_gain_df["delta_detection_diff_expected_minus_background"].median()),
    "fraction_markers_detection_diff_improved": float(marker_gain_df["improved_detection_diff"].mean()),
}

marker_signal_summary_df = pd.DataFrame([summary])

print("\nMarker signal strengthening summary:")
display(marker_signal_summary_df)

# ------------------------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------------------------

marker_gain_df.to_csv(MARKER_SIGNAL_PATH, index=False)
marker_signal_summary_df.to_csv(MARKER_SIGNAL_SUMMARY_PATH, index=False)

print("\nSaved marker-based signal strengthening outputs:")
print(f"  Marker gain table : {MARKER_SIGNAL_PATH}")
print(f"  Summary table     : {MARKER_SIGNAL_SUMMARY_PATH}")

print("\n" + "=" * 100)
print("CELL 3 COMPLETE — marker-based cell-type signal strengthening")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 3B — Marker signal gain heatmaps
#
# Biological question:
#   Which marker genes/cell types show stronger biological signal after 9E
#   imputation compared with raw observed counts?
#
# Heatmaps:
#   1. AUROC gain:
#        AUROC_completed - AUROC_raw
#   2. Signal-difference gain:
#        signal_diff_completed - signal_diff_raw
#
# Rows    = marker genes
# Columns = expected cell types
# Color   = improvement after imputation
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 3B — Marker signal gain heatmaps")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables / paths
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

MARKER_SIGNAL_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_signal_strengthening.csv"
)

HEATMAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_marker_signal_heatmaps"
)

os.makedirs(HEATMAP_DIR, exist_ok=True)

AUROC_HEATMAP_PNG = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_marker_AUROC_gain_heatmap.png"
)

SIGNAL_DIFF_HEATMAP_PNG = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_marker_signal_diff_gain_heatmap.png"
)

AUROC_MATRIX_CSV = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_marker_AUROC_gain_matrix.csv"
)

SIGNAL_DIFF_MATRIX_CSV = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_marker_signal_diff_gain_matrix.csv"
)

HEATMAP_SUMMARY_CSV = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_marker_signal_heatmap_summary.csv"
)

print(f"HEATMAP_DIR: {HEATMAP_DIR}")

# ------------------------------------------------------------------------------
# 1. Load marker_gain_df if not already in memory
# ------------------------------------------------------------------------------

if "marker_gain_df" not in globals():
    if not os.path.exists(MARKER_SIGNAL_PATH):
        raise FileNotFoundError(
            f"marker_gain_df not in memory and marker signal CSV not found:\n{MARKER_SIGNAL_PATH}\n"
            "Run CELL 3 first."
        )

    print(f"Loading marker_gain_df from:")
    print(f"  {MARKER_SIGNAL_PATH}")

    marker_gain_df = pd.read_csv(MARKER_SIGNAL_PATH)

else:
    print("Using marker_gain_df from memory.")

print(f"marker_gain_df shape: {marker_gain_df.shape}")
print("Columns:")
print(list(marker_gain_df.columns))

required_cols = [
    "expected_cell_type",
    "gene_id",
    "delta_AUROC_expected_vs_background",
    "delta_signal_diff_expected_minus_background",
]

missing_cols = [c for c in required_cols if c not in marker_gain_df.columns]

if missing_cols:
    raise KeyError(f"marker_gain_df is missing required columns: {missing_cols}")

# ------------------------------------------------------------------------------
# 2. Prepare heatmap matrices
# ------------------------------------------------------------------------------

df = marker_gain_df.copy()

df["expected_cell_type"] = df["expected_cell_type"].astype(str)
df["gene_id"] = df["gene_id"].astype(str)

# If the same gene/cell-type pair appears more than once, average it.
auroc_matrix = (
    df
    .pivot_table(
        index="gene_id",
        columns="expected_cell_type",
        values="delta_AUROC_expected_vs_background",
        aggfunc="mean"
    )
)

signal_diff_matrix = (
    df
    .pivot_table(
        index="gene_id",
        columns="expected_cell_type",
        values="delta_signal_diff_expected_minus_background",
        aggfunc="mean"
    )
)

# Order genes by strongest absolute AUROC gain, so the most informative genes appear first.
gene_order = (
    auroc_matrix
    .abs()
    .max(axis=1)
    .sort_values(ascending=False)
    .index
)

# Order cell types alphabetically for readability.
celltype_order = sorted(auroc_matrix.columns.tolist())

auroc_matrix = auroc_matrix.loc[gene_order, celltype_order]
signal_diff_matrix = signal_diff_matrix.reindex(index=gene_order, columns=celltype_order)

print("\nAUROC gain matrix shape:")
print(auroc_matrix.shape)

print("\nSignal-difference gain matrix shape:")
print(signal_diff_matrix.shape)

auroc_matrix.to_csv(AUROC_MATRIX_CSV)
signal_diff_matrix.to_csv(SIGNAL_DIFF_MATRIX_CSV)

print("\nSaved heatmap matrices:")
print(f"  AUROC gain matrix      : {AUROC_MATRIX_CSV}")
print(f"  Signal-diff gain matrix: {SIGNAL_DIFF_MATRIX_CSV}")

# ------------------------------------------------------------------------------
# 3. Heatmap plotting helper
# ------------------------------------------------------------------------------

def plot_gain_heatmap(matrix, title, colorbar_label, out_path, annotate=True):
    """
    Plot a diverging heatmap centered at 0.

    Positive value:
      marker signal improved after imputation.

    Negative value:
      marker signal weakened after imputation.
    """

    data = matrix.copy()

    n_rows, n_cols = data.shape

    # Make figure size adaptive.
    fig_w = max(10, 0.75 * n_cols + 4)
    fig_h = max(8, 0.35 * n_rows + 3)

    arr = data.values.astype(float)

    # Symmetric color range around zero.
    finite_vals = arr[np.isfinite(arr)]

    if len(finite_vals) == 0:
        raise ValueError("No finite values available for heatmap.")

    vmax = np.nanpercentile(np.abs(finite_vals), 95)

    if vmax <= 0 or not np.isfinite(vmax):
        vmax = np.nanmax(np.abs(finite_vals))

    if vmax <= 0 or not np.isfinite(vmax):
        vmax = 1.0

    vmin = -vmax

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(
        arr,
        aspect="auto",
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(title, fontsize=14, pad=16)
    ax.set_xlabel("Expected cell type", fontsize=12)
    ax.set_ylabel("Marker gene", fontsize=12)

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(data.columns, rotation=45, ha="right", fontsize=9)

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(data.index, fontsize=9)

    # Grid lines.
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Optional numeric annotations.
    if annotate and n_rows <= 60 and n_cols <= 20:
        for i in range(n_rows):
            for j in range(n_cols):
                val = arr[i, j]

                if not np.isfinite(val):
                    continue

                txt_color = "black" if abs(val) < 0.65 * vmax else "white"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color=txt_color,
                )

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label(colorbar_label, fontsize=11)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap: {out_path}")

# ------------------------------------------------------------------------------
# 4. Plot AUROC gain heatmap
# ------------------------------------------------------------------------------

plot_gain_heatmap(
    matrix=auroc_matrix,
    title="Marker signal gain after 9E imputation: AUROC completed − AUROC raw",
    colorbar_label="ΔAUROC",
    out_path=AUROC_HEATMAP_PNG,
    annotate=True,
)

# ------------------------------------------------------------------------------
# 5. Plot signal-difference gain heatmap
# ------------------------------------------------------------------------------

plot_gain_heatmap(
    matrix=signal_diff_matrix,
    title="Marker signal gain after 9E imputation: signal difference completed − raw",
    colorbar_label="Δ signal difference",
    out_path=SIGNAL_DIFF_HEATMAP_PNG,
    annotate=True,
)

# ------------------------------------------------------------------------------
# 6. Summary table: how many marker/cell-type pairs improved?
# ------------------------------------------------------------------------------

summary_rows = []

for metric_name, matrix in [
    ("delta_AUROC_expected_vs_background", auroc_matrix),
    ("delta_signal_diff_expected_minus_background", signal_diff_matrix),
]:
    vals = matrix.values.astype(float)
    finite_vals = vals[np.isfinite(vals)]

    n_total = len(finite_vals)
    n_improved = int((finite_vals > 0).sum())
    n_weakened = int((finite_vals < 0).sum())
    n_unchanged = int((finite_vals == 0).sum())

    summary_rows.append({
        "metric": metric_name,
        "n_marker_celltype_pairs": n_total,
        "n_improved_positive_gain": n_improved,
        "n_weakened_negative_gain": n_weakened,
        "n_unchanged_zero_gain": n_unchanged,
        "fraction_improved": n_improved / n_total if n_total > 0 else np.nan,
        "mean_gain": float(np.mean(finite_vals)) if n_total > 0 else np.nan,
        "median_gain": float(np.median(finite_vals)) if n_total > 0 else np.nan,
        "max_gain": float(np.max(finite_vals)) if n_total > 0 else np.nan,
        "min_gain": float(np.min(finite_vals)) if n_total > 0 else np.nan,
    })

heatmap_summary_df = pd.DataFrame(summary_rows)
heatmap_summary_df.to_csv(HEATMAP_SUMMARY_CSV, index=False)

print("\nMarker heatmap summary:")
display(heatmap_summary_df)

print(f"\nSaved heatmap summary:")
print(f"  {HEATMAP_SUMMARY_CSV}")

# ------------------------------------------------------------------------------
# 7. Top improved and weakened marker/cell-type pairs
# ------------------------------------------------------------------------------

top_auroc_improved = (
    df
    .sort_values("delta_AUROC_expected_vs_background", ascending=False)
    [["expected_cell_type", "gene_id", "delta_AUROC_expected_vs_background",
      "raw_AUROC_expected_vs_background", "completed_AUROC_expected_vs_background"]]
    .head(20)
)

top_auroc_weakened = (
    df
    .sort_values("delta_AUROC_expected_vs_background", ascending=True)
    [["expected_cell_type", "gene_id", "delta_AUROC_expected_vs_background",
      "raw_AUROC_expected_vs_background", "completed_AUROC_expected_vs_background"]]
    .head(20)
)

print("\nTop 20 marker/cell-type pairs improved by AUROC:")
display(top_auroc_improved)

print("\nTop 20 marker/cell-type pairs weakened by AUROC:")
display(top_auroc_weakened)

TOP_AUROC_IMPROVED_PATH = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_top_AUROC_marker_improvements.csv"
)

TOP_AUROC_WEAKENED_PATH = os.path.join(
    HEATMAP_DIR,
    f"{RUN_NAME}_top_AUROC_marker_weakened.csv"
)

top_auroc_improved.to_csv(TOP_AUROC_IMPROVED_PATH, index=False)
top_auroc_weakened.to_csv(TOP_AUROC_WEAKENED_PATH, index=False)

print("\nSaved top improvement/weakened tables:")
print(f"  Improved: {TOP_AUROC_IMPROVED_PATH}")
print(f"  Weakened: {TOP_AUROC_WEAKENED_PATH}")

print("\n" + "=" * 100)
print("CELL 3B COMPLETE — Marker signal gain heatmaps")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 4 — Marker-only cell-type classification
#
# Biological question:
#   Do known marker genes classify cell types better after learned 9E imputation?
#
# Main comparison:
#   Raw observed counts vs Learned 9E completed counts
#
# Optional comparison:
#   Step4 denoised counts, if X_denoised is available.
#
# Method:
#   1. Use only known marker genes present in the 313-gene panel.
#   2. Normalize counts using library-size normalization + log1p.
#   3. Train/test split cells using the same split for all datasets.
#   4. Train a simple logistic-regression classifier.
#   5. Compare accuracy, balanced accuracy, macro-F1, weighted-F1.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("=" * 100)
print("CELL 4 — Marker-only cell-type classification")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "shared_genes",
    "cell_type_labels",
    "unique_cell_types",
    "gene_to_col",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

if "X_raw_counts" not in globals():
    if "X_raw_from_adata" not in globals():
        raise NameError("Missing X_raw_counts and X_raw_from_adata.")
    X_raw_counts = X_raw_from_adata.toarray() if sparse.issparse(X_raw_from_adata) else np.asarray(X_raw_from_adata)
    X_raw_counts = X_raw_counts.astype(np.float32)

if "X_completed_9E" not in globals():
    raise NameError(
        "Missing X_completed_9E. Run Cell 2 first, or rebuild completed count matrix before this cell."
    )

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

MARKER_CLASSIFICATION_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_only_classification_summary.csv"
)

MARKER_CLASSIFICATION_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_only_classification_report.txt"
)

MARKER_CLASSIFICATION_CONFUSION_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_only_classification_confusion_matrices.csv"
)

MARKER_GENE_LIST_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_countlevel_marker_only_genes_used.csv"
)

# ------------------------------------------------------------------------------
# 2. Marker dictionary
# ------------------------------------------------------------------------------

# Broad marker dictionary. The cell automatically keeps only genes present
# in your measured 313-gene panel and cell types present in your data.

marker_sets = {
    "B_Cells": [
        "MS4A1", "CD79A", "CD79B", "BANK1", "CD19", "CD22", "TNFRSF17"
    ],

    "CD4+_T_Cells": [
        "CD3D", "CD3E", "CD4", "IL7R", "CCR7", "TCF7", "LTB"
    ],

    "CD8+_T_Cells": [
        "CD3D", "CD3E", "CD8A", "CD8B", "GZMB", "NKG7", "PRF1"
    ],

    "Macrophages_1": [
        "CD68", "CD163", "LYZ", "TYROBP", "FCGR3A", "LST1", "MNDA"
    ],

    "Macrophages_2": [
        "CD68", "CD163", "LYZ", "TYROBP", "FCGR3A", "LST1", "MNDA"
    ],

    "Endothelial": [
        "PECAM1", "VWF", "KDR", "CLDN5", "RAMP2", "NOSTRIN", "CAV1"
    ],

    "Stromal": [
        "LUM", "DCN", "COL1A1", "COL1A2", "POSTN", "DPT", "FBLN1", "CXCL12"
    ],

    "DCIS_1": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "DCIS_2": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "Invasive_Tumor": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1", "KRT19"
    ],

    "Prolif_Invasive_Tumor": [
        "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2", "GATA3", "FOXA1",
        "MKI67", "TOP2A", "CCND1"
    ],

    "IRF7+_DCs": [
        "IRF7", "ITGAX", "LAMP3", "CLEC10A", "FCER1A"
    ],

    "LAMP3+_DCs": [
        "LAMP3", "ITGAX", "IRF7", "CLEC10A", "FCER1A"
    ],
}

available_genes = set(str(g) for g in shared_genes)
available_cell_types = set(str(ct) for ct in unique_cell_types)

filtered_marker_sets = {}

for ct, genes in marker_sets.items():
    if ct not in available_cell_types:
        continue

    genes_present = [g for g in genes if g in available_genes]

    if len(genes_present) > 0:
        filtered_marker_sets[ct] = genes_present

marker_genes_used = sorted(set(g for genes in filtered_marker_sets.values() for g in genes))
marker_gene_indices = [gene_to_col[g] for g in marker_genes_used]

print("\nMarker sets used:")
for ct, genes in filtered_marker_sets.items():
    print(f"  {ct:25s}: {genes}")

print(f"\nTotal unique marker genes used: {len(marker_genes_used)}")
print(marker_genes_used)

if len(marker_genes_used) < 5:
    raise RuntimeError(
        "Too few marker genes found in this panel for marker-only classification."
    )

marker_gene_df = pd.DataFrame({
    "gene_id": marker_genes_used,
    "gene_col": marker_gene_indices,
})

marker_gene_df.to_csv(MARKER_GENE_LIST_PATH, index=False)

print(f"\nSaved marker gene list:")
print(f"  {MARKER_GENE_LIST_PATH}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def normalize_log_marker_matrix(X, marker_indices):
    """
    Library-size normalize using the whole count matrix, then select marker genes.
    This avoids normalizing only by marker genes, which could distort cells where
    marker-gene counts are low.
    """
    X = ensure_dense(X).astype(np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm)

    return X_log[:, marker_indices].astype(np.float32)


def evaluate_marker_classifier(dataset_name, X_counts, y_labels, train_idx, test_idx):
    """
    Train and evaluate marker-only logistic regression classifier.
    """

    print("\n" + "-" * 100)
    print(f"Evaluating marker-only classifier: {dataset_name}")
    print("-" * 100)

    X_marker = normalize_log_marker_matrix(X_counts, marker_gene_indices)

    X_train = X_marker[train_idx]
    X_test = X_marker[test_idx]

    y_train = y_labels[train_idx]
    y_test = y_labels[test_idx]

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
            multi_class="auto",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Accuracy          : {acc:.4f}")
    print(f"Balanced accuracy : {bal_acc:.4f}")
    print(f"Macro-F1          : {macro_f1:.4f}")
    print(f"Weighted-F1       : {weighted_f1:.4f}")

    report_dict = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report_dict).T.reset_index().rename(columns={"index": "label"})
    report_df.insert(0, "dataset", dataset_name)

    labels_sorted = sorted(pd.unique(y_labels).tolist())

    cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in labels_sorted],
        columns=[f"pred_{x}" for x in labels_sorted],
    )

    cm_long = (
        cm_df
        .reset_index()
        .melt(id_vars="index", var_name="predicted_label", value_name="n_cells")
        .rename(columns={"index": "true_label"})
    )
    cm_long.insert(0, "dataset", dataset_name)

    summary = {
        "dataset": dataset_name,
        "n_cells_total": int(len(y_labels)),
        "n_train_cells": int(len(train_idx)),
        "n_test_cells": int(len(test_idx)),
        "n_marker_genes": int(len(marker_genes_used)),
        "n_cell_types": int(len(np.unique(y_labels))),
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
    }

    del X_marker, X_train, X_test
    gc.collect()

    return summary, report_df, cm_long


# ------------------------------------------------------------------------------
# 4. Train/test split
# ------------------------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.25

y_labels = np.asarray(cell_type_labels).astype(str)

# Keep only cell types with enough cells for stratified train/test split.
cell_type_counts = pd.Series(y_labels).value_counts()
eligible_cell_types = cell_type_counts[cell_type_counts >= 10].index.tolist()

eligible_mask = np.isin(y_labels, eligible_cell_types)

eligible_indices = np.where(eligible_mask)[0]
eligible_labels = y_labels[eligible_indices]

print("\nCell type counts:")
display(cell_type_counts.reset_index().rename(columns={"index": "cell_type", "count": "n_cells"}))

print(f"\nEligible cells for classification: {len(eligible_indices):,} / {len(y_labels):,}")
print(f"Eligible cell types: {len(eligible_cell_types):,}")

train_local_idx, test_local_idx = train_test_split(
    np.arange(len(eligible_indices)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=eligible_labels,
)

train_idx = eligible_indices[train_local_idx]
test_idx = eligible_indices[test_local_idx]

print(f"Train cells: {len(train_idx):,}")
print(f"Test cells : {len(test_idx):,}")

# ------------------------------------------------------------------------------
# 5. Evaluate raw / completed / optional Step4
# ------------------------------------------------------------------------------

classification_summaries = []
classification_reports = []
confusion_matrices = []

datasets_to_evaluate = {
    "Raw observed counts": X_raw_counts,
    "Learned 9E completed counts": X_completed_9E,
}

if "X_denoised" in globals():
    datasets_to_evaluate["Step4 denoised counts"] = X_denoised

for dataset_name, X_counts in datasets_to_evaluate.items():
    summary, report_df, cm_long = evaluate_marker_classifier(
        dataset_name=dataset_name,
        X_counts=X_counts,
        y_labels=y_labels,
        train_idx=train_idx,
        test_idx=test_idx,
    )

    classification_summaries.append(summary)
    classification_reports.append(report_df)
    confusion_matrices.append(cm_long)

classification_summary_df = pd.DataFrame(classification_summaries)
classification_report_df = pd.concat(classification_reports, ignore_index=True)
classification_confusion_df = pd.concat(confusion_matrices, ignore_index=True)

# Put datasets in a clean order.
dataset_order = [
    "Raw observed counts",
    "Step4 denoised counts",
    "Learned 9E completed counts",
]

classification_summary_df["dataset"] = pd.Categorical(
    classification_summary_df["dataset"],
    categories=dataset_order,
    ordered=True,
)

classification_summary_df = classification_summary_df.sort_values("dataset").reset_index(drop=True)

print("\n" + "=" * 100)
print("MARKER-ONLY CLASSIFICATION SUMMARY")
print("=" * 100)
display(classification_summary_df)

# ------------------------------------------------------------------------------
# 6. Raw-vs-completed improvement table
# ------------------------------------------------------------------------------

raw_row = classification_summary_df[
    classification_summary_df["dataset"].astype(str) == "Raw observed counts"
]

completed_row = classification_summary_df[
    classification_summary_df["dataset"].astype(str) == "Learned 9E completed counts"
]

if len(raw_row) == 1 and len(completed_row) == 1:
    raw_row = raw_row.iloc[0]
    completed_row = completed_row.iloc[0]

    improvement_summary = {
        "comparison": "Learned 9E completed counts - Raw observed counts",
        "delta_accuracy": float(completed_row["accuracy"] - raw_row["accuracy"]),
        "delta_balanced_accuracy": float(completed_row["balanced_accuracy"] - raw_row["balanced_accuracy"]),
        "delta_macro_f1": float(completed_row["macro_f1"] - raw_row["macro_f1"]),
        "delta_weighted_f1": float(completed_row["weighted_f1"] - raw_row["weighted_f1"]),
        "relative_accuracy_gain_percent": float(
            100.0 * (completed_row["accuracy"] - raw_row["accuracy"]) / max(raw_row["accuracy"], 1e-8)
        ),
        "relative_balanced_accuracy_gain_percent": float(
            100.0 * (completed_row["balanced_accuracy"] - raw_row["balanced_accuracy"]) / max(raw_row["balanced_accuracy"], 1e-8)
        ),
        "relative_macro_f1_gain_percent": float(
            100.0 * (completed_row["macro_f1"] - raw_row["macro_f1"]) / max(raw_row["macro_f1"], 1e-8)
        ),
        "relative_weighted_f1_gain_percent": float(
            100.0 * (completed_row["weighted_f1"] - raw_row["weighted_f1"]) / max(raw_row["weighted_f1"], 1e-8)
        ),
    }

    marker_classification_improvement_df = pd.DataFrame([improvement_summary])

    print("\nRaw vs learned 9E completed improvement:")
    display(marker_classification_improvement_df)

else:
    marker_classification_improvement_df = pd.DataFrame()
    print("WARNING: Could not compute Raw-vs-Completed improvement table.")

# ------------------------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------------------------

classification_summary_df.to_csv(MARKER_CLASSIFICATION_SUMMARY_PATH, index=False)
classification_report_df.to_csv(
    MARKER_CLASSIFICATION_SUMMARY_PATH.replace("_summary.csv", "_per_class_report.csv"),
    index=False,
)
classification_confusion_df.to_csv(MARKER_CLASSIFICATION_CONFUSION_PATH, index=False)

if len(marker_classification_improvement_df) > 0:
    marker_classification_improvement_df.to_csv(
        MARKER_CLASSIFICATION_SUMMARY_PATH.replace("_summary.csv", "_improvement.csv"),
        index=False,
    )

# Save human-readable report.
with open(MARKER_CLASSIFICATION_REPORT_PATH, "w") as f:
    f.write("MARKER-ONLY CELL-TYPE CLASSIFICATION REPORT\n")
    f.write("=" * 90 + "\n\n")
    f.write(f"RUN_NAME: {RUN_NAME}\n")
    f.write(f"RANDOM_STATE: {RANDOM_STATE}\n")
    f.write(f"TEST_SIZE: {TEST_SIZE}\n")
    f.write(f"n_marker_genes: {len(marker_genes_used)}\n")
    f.write(f"marker_genes_used: {marker_genes_used}\n\n")

    f.write("CLASSIFICATION SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(classification_summary_df.to_string(index=False))
    f.write("\n\n")

    if len(marker_classification_improvement_df) > 0:
        f.write("RAW VS LEARNED 9E COMPLETED IMPROVEMENT\n")
        f.write("-" * 90 + "\n")
        f.write(marker_classification_improvement_df.to_string(index=False))
        f.write("\n\n")

    f.write("INTERPRETATION GUIDE\n")
    f.write("-" * 90 + "\n")
    f.write("Good sign: Learned 9E completed counts improve balanced accuracy and macro-F1 over raw counts.\n")
    f.write("Balanced accuracy and macro-F1 are especially important because cell types are imbalanced.\n")
    f.write("This validates that known marker genes carry stronger cell-type signal after imputation.\n")

print("\nSaved marker-only classification outputs:")
print(f"  Summary             : {MARKER_CLASSIFICATION_SUMMARY_PATH}")
print(f"  Per-class report    : {MARKER_CLASSIFICATION_SUMMARY_PATH.replace('_summary.csv', '_per_class_report.csv')}")
print(f"  Confusion matrices  : {MARKER_CLASSIFICATION_CONFUSION_PATH}")
if len(marker_classification_improvement_df) > 0:
    print(f"  Improvement table   : {MARKER_CLASSIFICATION_SUMMARY_PATH.replace('_summary.csv', '_improvement.csv')}")
print(f"  Text report         : {MARKER_CLASSIFICATION_REPORT_PATH}")

print("\n" + "=" * 100)
print("CELL 4 COMPLETE — marker-only cell-type classification")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 5 — PCA/UMAP visualization of Raw vs Step4 vs Learned 9E
#
# Biological/statistical question:
#   Does cell-type separation visually improve from raw counts to Step4 and 9E?
#
# Main comparison:
#   Raw observed counts
#   Step4 denoised counts
#   Learned 9E completed counts
#
# Outputs:
#   1. PCA scatter plots
#   2. UMAP scatter plots
#   3. Embedding CSVs for reproducibility
#
# GPU not needed.
# ==============================================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

print("=" * 100)
print("CELL 5 — PCA/UMAP visualization of Raw vs Step4 vs Learned 9E")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "denoised_adata",
    "cell_type_labels",
    "cell_ids_step4",
    "X_raw_counts",
    "X_completed_9E",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

if "X_denoised" not in globals():
    raise NameError("Missing X_denoised. Load Step4 denoised matrix before running this cell.")

# ------------------------------------------------------------------------------
# 1. Imports / install Scanpy if needed
# ------------------------------------------------------------------------------

try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print(f"Scanpy import failed: {e}")
    print("Installing scanpy/leiden dependencies...")
    !pip install -q scanpy leidenalg igraph anndata
    import scanpy as sc
    import anndata as ad

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

VIS_DIR = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_visualizations_pca_umap")
os.makedirs(VIS_DIR, exist_ok=True)

PCA_FIG_PATH = os.path.join(VIS_DIR, f"{RUN_NAME}_raw_step4_9E_PCA_celltypes.png")
UMAP_FIG_PATH = os.path.join(VIS_DIR, f"{RUN_NAME}_raw_step4_9E_UMAP_celltypes.png")
EMBEDDINGS_CSV_PATH = os.path.join(VIS_DIR, f"{RUN_NAME}_raw_step4_9E_embeddings.csv")
SUMMARY_CSV_PATH = os.path.join(VIS_DIR, f"{RUN_NAME}_raw_step4_9E_visualization_summary.csv")

print(f"VIS_DIR: {VIS_DIR}")

# ------------------------------------------------------------------------------
# 3. Config
# ------------------------------------------------------------------------------

SAMPLE_SIZE = 50000
RANDOM_STATE = 42
N_PCS = 30
N_NEIGHBORS = 15

print(f"SAMPLE_SIZE: {SAMPLE_SIZE:,}")
print(f"RANDOM_STATE: {RANDOM_STATE}")
print(f"N_PCS: {N_PCS}")
print(f"N_NEIGHBORS: {N_NEIGHBORS}")

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def make_embedding_for_matrix(X_full, dataset_name, sample_idx):
    """
    Build AnnData for one matrix, normalize/log1p, compute PCA and UMAP.
    """
    print("\n" + "-" * 100)
    print(f"Computing PCA/UMAP for: {dataset_name}")
    print("-" * 100)

    X_full = ensure_dense(X_full).astype(np.float32)

    obs_sub = denoised_adata.obs.iloc[sample_idx].copy()
    var_sub = denoised_adata.var.copy()

    adata_tmp = ad.AnnData(
        X=X_full[sample_idx, :].copy(),
        obs=obs_sub,
        var=var_sub,
    )

    adata_tmp.var_names_make_unique()

    n_pcs_use = min(N_PCS, adata_tmp.n_vars - 1)

    print(f"  AnnData shape: {adata_tmp.shape}")
    print(f"  Total sampled counts: {adata_tmp.X.sum(dtype=np.float64):,.2f}")

    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    sc.pp.pca(
        adata_tmp,
        n_comps=n_pcs_use,
        random_state=RANDOM_STATE,
    )

    sc.pp.neighbors(
        adata_tmp,
        n_neighbors=N_NEIGHBORS,
        n_pcs=n_pcs_use,
    )

    sc.tl.umap(
        adata_tmp,
        random_state=RANDOM_STATE,
    )

    pca = adata_tmp.obsm["X_pca"][:, :2]
    umap = adata_tmp.obsm["X_umap"][:, :2]

    labels = np.asarray(cell_type_labels).astype(str)[sample_idx]
    cids = np.asarray(cell_ids_step4).astype(int)[sample_idx]

    emb_df = pd.DataFrame({
        "dataset": dataset_name,
        "cell_id": cids,
        "cell_type": labels,
        "PCA1": pca[:, 0],
        "PCA2": pca[:, 1],
        "UMAP1": umap[:, 0],
        "UMAP2": umap[:, 1],
    })

    summary = {
        "dataset": dataset_name,
        "n_cells": int(adata_tmp.n_obs),
        "n_genes": int(adata_tmp.n_vars),
        "n_pcs": int(n_pcs_use),
        "total_sampled_counts_before_normalization": float(X_full[sample_idx, :].sum(dtype=np.float64)),
    }

    del adata_tmp
    gc.collect()

    return emb_df, summary


def plot_embedding_grid(embedding_df, x_col, y_col, out_path, title):
    """
    Plot Raw / Step4 / 9E side by side, colored by cell type.
    """
    datasets = [
        "Raw observed counts",
        "Step4 denoised counts",
        "Learned 9E completed counts",
    ]

    cell_types = sorted(embedding_df["cell_type"].unique().tolist())

    # Use a stable categorical color map.
    cmap = plt.get_cmap("tab20")
    color_map = {
        ct: cmap(i % 20)
        for i, ct in enumerate(cell_types)
    }

    fig, axes = plt.subplots(1, 3, figsize=(21, 6), constrained_layout=True)

    for ax, dataset in zip(axes, datasets):
        sub = embedding_df[embedding_df["dataset"] == dataset].copy()

        for ct in cell_types:
            s = sub[sub["cell_type"] == ct]
            if len(s) == 0:
                continue

            ax.scatter(
                s[x_col],
                s[y_col],
                s=2,
                alpha=0.45,
                label=ct,
                color=color_map[ct],
                linewidths=0,
            )

        ax.set_title(dataset, fontsize=13)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_xticks([])
        ax.set_yticks([])

    # Put one legend outside.
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        fontsize=8,
        markerscale=4,
        frameon=False,
    )

    fig.suptitle(title, fontsize=16)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {out_path}")

# ------------------------------------------------------------------------------
# 5. Fixed sample
# ------------------------------------------------------------------------------

print("\nCreating fixed sampled cell set...")

n_cells = len(cell_ids_step4)
n_sample = min(SAMPLE_SIZE, n_cells)

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(n_cells, size=n_sample, replace=False)

print(f"Sampled cells: {n_sample:,} / {n_cells:,}")

# ------------------------------------------------------------------------------
# 6. Compute embeddings
# ------------------------------------------------------------------------------

embedding_dfs = []
summary_rows = []

datasets = {
    "Raw observed counts": X_raw_counts,
    "Step4 denoised counts": X_denoised,
    "Learned 9E completed counts": X_completed_9E,
}

for dataset_name, X in datasets.items():
    emb_df, summary = make_embedding_for_matrix(
        X_full=X,
        dataset_name=dataset_name,
        sample_idx=sample_idx,
    )

    embedding_dfs.append(emb_df)
    summary_rows.append(summary)

embedding_df = pd.concat(embedding_dfs, ignore_index=True)
visual_summary_df = pd.DataFrame(summary_rows)

print("\nEmbedding dataframe:")
display(embedding_df.head())

print("\nVisualization summary:")
display(visual_summary_df)

# ------------------------------------------------------------------------------
# 7. Save embeddings
# ------------------------------------------------------------------------------

embedding_df.to_csv(EMBEDDINGS_CSV_PATH, index=False)
visual_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

print(f"\nSaved embeddings:")
print(f"  {EMBEDDINGS_CSV_PATH}")

print(f"Saved visualization summary:")
print(f"  {SUMMARY_CSV_PATH}")

# ------------------------------------------------------------------------------
# 8. Plot PCA and UMAP
# ------------------------------------------------------------------------------

plot_embedding_grid(
    embedding_df=embedding_df,
    x_col="PCA1",
    y_col="PCA2",
    out_path=PCA_FIG_PATH,
    title="PCA visualization: Raw vs Step4 vs Learned 9E",
)

plot_embedding_grid(
    embedding_df=embedding_df,
    x_col="UMAP1",
    y_col="UMAP2",
    out_path=UMAP_FIG_PATH,
    title="UMAP visualization: Raw vs Step4 vs Learned 9E",
)

print("\n" + "=" * 100)
print("CELL 5 COMPLETE — PCA/UMAP visualization")
print("=" * 100)

In [ ]:
# ==============================================================================
# CELL 6 — Spatial map visualization for selected genes/cells
#
# Biological/visual question:
#   Where are observed and imputed molecules placed inside real cells?
#
# Main comparison:
#   Observed molecules vs Learned 9E imputed molecules vs 3 empirical baselines
#
# Outputs:
#   Per selected gene/cell figure panels showing molecule dots and cell/nucleus shape.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 6 — Spatial map visualization for selected genes/cells")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "gene_emp_imputed_df",
    "ct_gene_emp_imputed_df",
    "spatial_knn_emp_imputed_df",
    "shared_genes",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

SPATIAL_VIS_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_spatial_molecule_maps"
)

os.makedirs(SPATIAL_VIS_DIR, exist_ok=True)

SELECTION_CSV_PATH = os.path.join(
    SPATIAL_VIS_DIR,
    f"{RUN_NAME}_selected_gene_cell_pairs_for_spatial_maps.csv"
)

SUMMARY_CSV_PATH = os.path.join(
    SPATIAL_VIS_DIR,
    f"{RUN_NAME}_spatial_map_visualization_summary.csv"
)

print(f"SPATIAL_VIS_DIR: {SPATIAL_VIS_DIR}")

# ------------------------------------------------------------------------------
# 2. Load cell/nucleus polygons if available
# ------------------------------------------------------------------------------

CELL_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "cell_polygons.pkl")
NUC_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_polygons.pkl")

if "cell_polygons" not in globals():
    if os.path.exists(CELL_POLYGONS_PATH):
        print(f"Loading cell polygons: {CELL_POLYGONS_PATH}")
        with open(CELL_POLYGONS_PATH, "rb") as f:
            cell_polygons = pickle.load(f)
    else:
        cell_polygons = {}
        print("WARNING: cell_polygons.pkl not found. Plots will not show cell boundaries.")

if "nuc_polygons" not in globals():
    if os.path.exists(NUC_POLYGONS_PATH):
        print(f"Loading nucleus polygons: {NUC_POLYGONS_PATH}")
        with open(NUC_POLYGONS_PATH, "rb") as f:
            nuc_polygons = pickle.load(f)
    else:
        nuc_polygons = {}
        print("WARNING: nuc_polygons.pkl not found. Plots will not show nucleus boundaries.")

print(f"cell_polygons entries: {len(cell_polygons):,}")
print(f"nuc_polygons entries : {len(nuc_polygons):,}")

# ------------------------------------------------------------------------------
# 3. Configuration: selected genes
# ------------------------------------------------------------------------------

candidate_genes = [
    # Tumor / epithelial
    "ERBB2", "EPCAM", "KRT7", "KRT8", "KRT18", "GATA3", "FOXA1", "CCND1",

    # Stromal / ECM
    "LUM", "POSTN", "CXCL12", "DPT", "FBLN1",

    # Immune
    "CD3D", "CD68", "TYROBP", "MNDA", "TNFRSF17",

    # Endothelial / vascular
    "CAV1", "NOSTRIN",
]

available_genes = set(str(g) for g in shared_genes)
selected_genes = [g for g in candidate_genes if g in available_genes]

print(f"Selected genes present in panel: {selected_genes}")

if len(selected_genes) == 0:
    raise RuntimeError("None of the candidate genes are present in shared_genes.")

MAX_GENE_CELL_PAIRS = 12

# These thresholds control how visually informative selected examples are.
MIN_OBSERVED_FOR_PAIR = 2
MIN_IMPUTED_FOR_PAIR = 2

# Plotting controls.
MAX_POINTS_PER_PANEL = 300
POINT_SIZE = 16
ALPHA_POINTS = 0.75

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def _plot_polygon_boundary(ax, geom, linestyle="-", linewidth=1.0, alpha=0.9):
    """
    Plot shapely Polygon or MultiPolygon boundary.
    """
    if geom is None:
        return

    try:
        if geom.geom_type == "Polygon":
            x, y = geom.exterior.xy
            ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

    except Exception:
        pass


def _get_geom_bounds(cid, fallback_df=None, padding=5.0):
    """
    Get plot bounds from cell polygon if available. Otherwise use molecule coordinates.
    """
    cid = int(cid)

    if cid in cell_polygons:
        try:
            minx, miny, maxx, maxy = cell_polygons[cid].bounds
            return minx - padding, maxx + padding, miny - padding, maxy + padding
        except Exception:
            pass

    if fallback_df is not None and len(fallback_df) > 0:
        x = pd.to_numeric(fallback_df["x"], errors="coerce")
        y = pd.to_numeric(fallback_df["y"], errors="coerce")

        if x.notna().sum() > 0 and y.notna().sum() > 0:
            return (
                float(x.min()) - padding,
                float(x.max()) + padding,
                float(y.min()) - padding,
                float(y.max()) + padding,
            )

    return None


def get_pair_counts(df, label):
    """
    Count molecules per (cell_id, gene_id).
    """
    d = df[["cell_id", "gene_id"]].copy()
    d["cell_id"] = d["cell_id"].astype(int)
    d["gene_id"] = d["gene_id"].astype(str)

    out = (
        d.groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name=label)
    )

    return out


def select_gene_cell_pairs():
    """
    Select gene/cell pairs that have both observed and learned-imputed molecules.
    These pairs make the clearest visualization panels.
    """
    print("\nSelecting gene/cell pairs for spatial visualization...")

    observed_sub = mol_observed[
        mol_observed["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    learned_sub = learned_imputed_df[
        learned_imputed_df["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    observed_counts = get_pair_counts(observed_sub, "n_observed")
    learned_counts = get_pair_counts(learned_sub, "n_learned_imputed")

    merged = observed_counts.merge(
        learned_counts,
        on=["cell_id", "gene_id"],
        how="inner",
    )

    merged = merged[
        (merged["n_observed"] >= MIN_OBSERVED_FOR_PAIR)
        & (merged["n_learned_imputed"] >= MIN_IMPUTED_FOR_PAIR)
    ].copy()

    if len(merged) == 0:
        print("No pairs found with strict thresholds.")
        print("Relaxing thresholds to at least one observed and one learned-imputed molecule.")

        merged = observed_counts.merge(
            learned_counts,
            on=["cell_id", "gene_id"],
            how="inner",
        )

        merged = merged[
            (merged["n_observed"] >= 1)
            & (merged["n_learned_imputed"] >= 1)
        ].copy()

    if len(merged) == 0:
        raise RuntimeError("Could not find any gene/cell pair with both observed and learned-imputed molecules.")

    merged["total_pair_molecules"] = merged["n_observed"] + merged["n_learned_imputed"]

    selected_rows = []

    # Pick one strong example per gene when possible.
    for gene in selected_genes:
        sub = merged[merged["gene_id"] == gene].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values(
            ["n_learned_imputed", "n_observed", "total_pair_molecules"],
            ascending=False,
        )

        selected_rows.append(sub.iloc[0])

        if len(selected_rows) >= MAX_GENE_CELL_PAIRS:
            break

    selected = pd.DataFrame(selected_rows)

    if len(selected) < min(MAX_GENE_CELL_PAIRS, len(merged)):
        remaining = merged.merge(
            selected[["cell_id", "gene_id"]],
            on=["cell_id", "gene_id"],
            how="left",
            indicator=True,
        )

        remaining = remaining[remaining["_merge"] == "left_only"].drop(columns=["_merge"])
        remaining = remaining.sort_values(
            ["n_learned_imputed", "n_observed", "total_pair_molecules"],
            ascending=False,
        )

        needed = MAX_GENE_CELL_PAIRS - len(selected)
        selected = pd.concat([selected, remaining.head(needed)], ignore_index=True)

    selected = selected.head(MAX_GENE_CELL_PAIRS).reset_index(drop=True)

    return selected


def get_pair_molecules(df, cid, gid):
    """
    Get molecules for one cell-gene pair.
    """
    sub = df[
        (df["cell_id"].astype(int) == int(cid))
        & (df["gene_id"].astype(str) == str(gid))
    ].copy()

    return sub


def downsample_points(df, max_points=300, seed=42):
    """
    Downsample points for clean visualization.
    """
    if len(df) <= max_points:
        return df

    return df.sample(n=max_points, random_state=seed).copy()


def plot_one_pair(cid, gid, out_prefix):
    """
    Make one multi-panel plot for a selected cell-gene pair.
    """
    cid = int(cid)
    gid = str(gid)

    source_tables = {
        "Observed": mol_observed,
        "Learned 9E": learned_imputed_df,
        "Gene empirical": gene_emp_imputed_df,
        "Cell-type gene empirical": ct_gene_emp_imputed_df,
        "Spatial-kNN empirical": spatial_knn_emp_imputed_df,
    }

    pair_data = {}

    for label, df in source_tables.items():
        sub = get_pair_molecules(df, cid, gid)
        pair_data[label] = sub

    combined_for_bounds = pd.concat(
        [d for d in pair_data.values() if len(d) > 0],
        ignore_index=True,
    ) if any(len(d) > 0 for d in pair_data.values()) else pd.DataFrame()

    bounds = _get_geom_bounds(cid, fallback_df=combined_for_bounds, padding=5.0)

    n_panels = len(source_tables)
    fig, axes = plt.subplots(
        1,
        n_panels,
        figsize=(4.2 * n_panels, 4.4),
        sharex=True,
        sharey=True,
    )

    if n_panels == 1:
        axes = [axes]

    for ax, (label, sub) in zip(axes, pair_data.items()):
        # Plot cell/nucleus boundaries first.
        if cid in cell_polygons:
            _plot_polygon_boundary(
                ax,
                cell_polygons[cid],
                linestyle="-",
                linewidth=1.3,
                alpha=0.9,
            )

        if cid in nuc_polygons:
            _plot_polygon_boundary(
                ax,
                nuc_polygons[cid],
                linestyle="--",
                linewidth=1.1,
                alpha=0.9,
            )

        sub_plot = downsample_points(
            sub,
            max_points=MAX_POINTS_PER_PANEL,
            seed=42,
        )

        if len(sub_plot) > 0:
            ax.scatter(
                pd.to_numeric(sub_plot["x"], errors="coerce"),
                pd.to_numeric(sub_plot["y"], errors="coerce"),
                s=POINT_SIZE,
                alpha=ALPHA_POINTS,
                marker="o",
                label=label,
            )

        ax.set_title(
            f"{label}\n n={len(sub):,}",
            fontsize=10,
        )

        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, linewidth=0.3, alpha=0.4)

        if bounds is not None:
            xmin, xmax, ymin, ymax = bounds
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.suptitle(
        f"Cell {cid} — Gene {gid}\nObserved vs learned 9E vs empirical baseline imputed molecule locations",
        fontsize=13,
        y=1.04,
    )

    plt.tight_layout()

    png_path = f"{out_prefix}.png"
    pdf_path = f"{out_prefix}.pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    summary_rows = []

    for label, sub in pair_data.items():
        if len(sub) == 0:
            summary_rows.append({
                "cell_id": cid,
                "gene_id": gid,
                "source": label,
                "n_molecules": 0,
                "mean_r_norm": np.nan,
                "mean_z_rel": np.nan,
                "mean_p_nuclear": np.nan,
                "mean_x": np.nan,
                "mean_y": np.nan,
            })
        else:
            summary_rows.append({
                "cell_id": cid,
                "gene_id": gid,
                "source": label,
                "n_molecules": int(len(sub)),
                "mean_r_norm": float(pd.to_numeric(sub["r_norm"], errors="coerce").mean()) if "r_norm" in sub.columns else np.nan,
                "mean_z_rel": float(pd.to_numeric(sub["z_rel"], errors="coerce").mean()) if "z_rel" in sub.columns else np.nan,
                "mean_p_nuclear": float(pd.to_numeric(sub["p_nuclear"], errors="coerce").mean()) if "p_nuclear" in sub.columns else np.nan,
                "mean_x": float(pd.to_numeric(sub["x"], errors="coerce").mean()),
                "mean_y": float(pd.to_numeric(sub["y"], errors="coerce").mean()),
            })

    return pd.DataFrame(summary_rows), png_path, pdf_path


# ------------------------------------------------------------------------------
# 5. Select examples
# ------------------------------------------------------------------------------

selected_pairs_df = select_gene_cell_pairs()

print("\nSelected gene/cell pairs:")
display(selected_pairs_df)

selected_pairs_df.to_csv(SELECTION_CSV_PATH, index=False)

print(f"Saved selected pairs:")
print(f"  {SELECTION_CSV_PATH}")

# ------------------------------------------------------------------------------
# 6. Generate spatial maps
# ------------------------------------------------------------------------------

print("\nGenerating spatial molecule maps...")

all_summary_rows = []
figure_records = []

for i, row in selected_pairs_df.iterrows():
    cid = int(row["cell_id"])
    gid = str(row["gene_id"])

    safe_gid = gid.replace("/", "_").replace(" ", "_").replace("+", "plus")
    out_prefix = os.path.join(
        SPATIAL_VIS_DIR,
        f"{RUN_NAME}_cell_{cid}_gene_{safe_gid}"
    )

    print(f"\n[{i+1}/{len(selected_pairs_df)}] Plotting cell_id={cid}, gene_id={gid}")

    summary_df, png_path, pdf_path = plot_one_pair(
        cid=cid,
        gid=gid,
        out_prefix=out_prefix,
    )

    all_summary_rows.append(summary_df)

    figure_records.append({
        "cell_id": cid,
        "gene_id": gid,
        "png_path": png_path,
        "pdf_path": pdf_path,
        "n_observed_selected": int(row.get("n_observed", -1)),
        "n_learned_imputed_selected": int(row.get("n_learned_imputed", -1)),
    })

# ------------------------------------------------------------------------------
# 7. Save summary tables
# ------------------------------------------------------------------------------

spatial_summary_df = pd.concat(all_summary_rows, ignore_index=True)
figure_manifest_df = pd.DataFrame(figure_records)

spatial_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

FIGURE_MANIFEST_PATH = os.path.join(
    SPATIAL_VIS_DIR,
    f"{RUN_NAME}_spatial_map_figure_manifest.csv"
)

figure_manifest_df.to_csv(FIGURE_MANIFEST_PATH, index=False)

print("\nSpatial map summary:")
display(spatial_summary_df.head(30))

print("\nFigure manifest:")
display(figure_manifest_df)

print("\nSaved spatial visualization outputs:")
print(f"  Selected pairs CSV : {SELECTION_CSV_PATH}")
print(f"  Summary CSV        : {SUMMARY_CSV_PATH}")
print(f"  Figure manifest    : {FIGURE_MANIFEST_PATH}")
print(f"  Figure directory   : {SPATIAL_VIS_DIR}")

print("\n" + "=" * 100)
print("CELL 6 COMPLETE — Spatial map visualization")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 6B — Raw vs completed molecule dot maps
#
# Figure layout:
#   Panel 1: Raw observed
#   Panel 2: Raw + learned 9E
#   Panel 3: Raw + gene empirical
#   Panel 4: Raw + cell-type gene empirical
#   Panel 5: Raw + spatial-kNN empirical
#
# Biological/visual question:
#   How did the molecule map look before imputation, and how does it look after
#   learned 9E / empirical baseline imputation?
#
# GPU not needed.
# ==============================================================================

import os
import gc
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 6B — Raw vs completed molecule dot maps")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "gene_emp_imputed_df",
    "ct_gene_emp_imputed_df",
    "spatial_knn_emp_imputed_df",
    "shared_genes",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

RAW_COMPLETED_VIS_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_raw_vs_completed_dot_maps"
)

os.makedirs(RAW_COMPLETED_VIS_DIR, exist_ok=True)

SELECTION_CSV_PATH = os.path.join(
    RAW_COMPLETED_VIS_DIR,
    f"{RUN_NAME}_selected_gene_cell_pairs_raw_vs_completed.csv"
)

SUMMARY_CSV_PATH = os.path.join(
    RAW_COMPLETED_VIS_DIR,
    f"{RUN_NAME}_raw_vs_completed_dot_map_summary.csv"
)

FIGURE_MANIFEST_PATH = os.path.join(
    RAW_COMPLETED_VIS_DIR,
    f"{RUN_NAME}_raw_vs_completed_dot_map_figure_manifest.csv"
)

print(f"RAW_COMPLETED_VIS_DIR: {RAW_COMPLETED_VIS_DIR}")

# ------------------------------------------------------------------------------
# 2. Load cell/nucleus polygons if available
# ------------------------------------------------------------------------------

CELL_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "cell_polygons.pkl")
NUC_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_polygons.pkl")

if "cell_polygons" not in globals():
    if os.path.exists(CELL_POLYGONS_PATH):
        print(f"Loading cell polygons: {CELL_POLYGONS_PATH}")
        with open(CELL_POLYGONS_PATH, "rb") as f:
            cell_polygons = pickle.load(f)
    else:
        cell_polygons = {}
        print("WARNING: cell_polygons.pkl not found. Plots will not show cell boundaries.")

if "nuc_polygons" not in globals():
    if os.path.exists(NUC_POLYGONS_PATH):
        print(f"Loading nucleus polygons: {NUC_POLYGONS_PATH}")
        with open(NUC_POLYGONS_PATH, "rb") as f:
            nuc_polygons = pickle.load(f)
    else:
        nuc_polygons = {}
        print("WARNING: nuc_polygons.pkl not found. Plots will not show nucleus boundaries.")

print(f"cell_polygons entries: {len(cell_polygons):,}")
print(f"nuc_polygons entries : {len(nuc_polygons):,}")

# ------------------------------------------------------------------------------
# 3. Configuration
# ------------------------------------------------------------------------------

candidate_genes = [
    # Tumor / epithelial
    "ERBB2", "EPCAM", "KRT7", "KRT8", "KRT18", "GATA3", "FOXA1", "CCND1",

    # Stromal / ECM
    "LUM", "POSTN", "CXCL12", "DPT", "FBLN1",

    # Immune
    "CD3D", "CD68", "TYROBP", "MNDA", "TNFRSF17",

    # Endothelial / vascular
    "CAV1", "NOSTRIN",
]

available_genes = set(str(g) for g in shared_genes)
selected_genes = [g for g in candidate_genes if g in available_genes]

print(f"Selected genes present in panel: {selected_genes}")

if len(selected_genes) == 0:
    raise RuntimeError("None of the candidate genes are present in shared_genes.")

MAX_GENE_CELL_PAIRS = 12

# To make visually useful figures, choose cell-gene pairs where the raw cell has
# some observed molecules and the learned model added some molecules.
MIN_OBSERVED_FOR_PAIR = 1
MIN_LEARNED_IMPUTED_FOR_PAIR = 2

# Plotting controls.
MAX_RAW_POINTS_PER_PANEL = 400
MAX_IMPUTED_POINTS_PER_PANEL = 400
RAW_POINT_SIZE = 18
IMPUTED_POINT_SIZE = 22
RAW_ALPHA = 0.75
IMPUTED_ALPHA = 0.85

# ------------------------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------------------------

def _plot_polygon_boundary(ax, geom, linestyle="-", linewidth=1.0, alpha=0.9):
    """
    Plot shapely Polygon or MultiPolygon boundary.
    """
    if geom is None:
        return

    try:
        if geom.geom_type == "Polygon":
            x, y = geom.exterior.xy
            ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                x, y = poly.exterior.xy
                ax.plot(x, y, linestyle=linestyle, linewidth=linewidth, alpha=alpha)

    except Exception:
        pass


def _get_geom_bounds(cid, fallback_df=None, padding=5.0):
    """
    Get plot bounds from cell polygon if available. Otherwise use molecule coordinates.
    """
    cid = int(cid)

    if cid in cell_polygons:
        try:
            minx, miny, maxx, maxy = cell_polygons[cid].bounds
            return minx - padding, maxx + padding, miny - padding, maxy + padding
        except Exception:
            pass

    if fallback_df is not None and len(fallback_df) > 0:
        x = pd.to_numeric(fallback_df["x"], errors="coerce")
        y = pd.to_numeric(fallback_df["y"], errors="coerce")

        if x.notna().sum() > 0 and y.notna().sum() > 0:
            return (
                float(x.min()) - padding,
                float(x.max()) + padding,
                float(y.min()) - padding,
                float(y.max()) + padding,
            )

    return None


def get_pair_counts(df, label):
    """
    Count molecules per (cell_id, gene_id).
    """
    d = df[["cell_id", "gene_id"]].copy()
    d["cell_id"] = d["cell_id"].astype(int)
    d["gene_id"] = d["gene_id"].astype(str)

    return (
        d.groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name=label)
    )


def get_pair_molecules(df, cid, gid):
    """
    Get molecules for one cell-gene pair.
    """
    return df[
        (df["cell_id"].astype(int) == int(cid))
        & (df["gene_id"].astype(str) == str(gid))
    ].copy()


def downsample_points(df, max_points=400, seed=42):
    """
    Downsample points for clean visualization.
    """
    if len(df) <= max_points:
        return df.copy()

    return df.sample(n=max_points, random_state=seed).copy()


def select_gene_cell_pairs_for_raw_completed():
    """
    Select visually useful gene/cell pairs.
    Preference:
      - gene in selected_genes
      - at least MIN_OBSERVED_FOR_PAIR raw observed molecules
      - at least MIN_LEARNED_IMPUTED_FOR_PAIR learned 9E imputed molecules
    """
    print("\nSelecting gene/cell pairs for raw-vs-completed visualization...")

    observed_sub = mol_observed[
        mol_observed["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    learned_sub = learned_imputed_df[
        learned_imputed_df["gene_id"].astype(str).isin(selected_genes)
    ].copy()

    observed_counts = get_pair_counts(observed_sub, "n_observed")
    learned_counts = get_pair_counts(learned_sub, "n_learned_imputed")

    merged = observed_counts.merge(
        learned_counts,
        on=["cell_id", "gene_id"],
        how="inner",
    )

    merged = merged[
        (merged["n_observed"] >= MIN_OBSERVED_FOR_PAIR)
        & (merged["n_learned_imputed"] >= MIN_LEARNED_IMPUTED_FOR_PAIR)
    ].copy()

    if len(merged) == 0:
        print("No pairs found with the current thresholds.")
        print("Relaxing to at least one observed and one learned-imputed molecule.")

        merged = observed_counts.merge(
            learned_counts,
            on=["cell_id", "gene_id"],
            how="inner",
        )

        merged = merged[
            (merged["n_observed"] >= 1)
            & (merged["n_learned_imputed"] >= 1)
        ].copy()

    if len(merged) == 0:
        raise RuntimeError("Could not find any gene/cell pair with both raw and learned-imputed molecules.")

    merged["total_raw_plus_learned"] = merged["n_observed"] + merged["n_learned_imputed"]

    # Prefer examples with enough imputed molecules to make the after-panel visible,
    # but still not too dense.
    selected_rows = []

    for gene in selected_genes:
        sub = merged[merged["gene_id"] == gene].copy()

        if len(sub) == 0:
            continue

        sub = sub.sort_values(
            ["n_learned_imputed", "n_observed", "total_raw_plus_learned"],
            ascending=False,
        )

        selected_rows.append(sub.iloc[0])

        if len(selected_rows) >= MAX_GENE_CELL_PAIRS:
            break

    selected = pd.DataFrame(selected_rows)

    if len(selected) < min(MAX_GENE_CELL_PAIRS, len(merged)):
        remaining = merged.merge(
            selected[["cell_id", "gene_id"]],
            on=["cell_id", "gene_id"],
            how="left",
            indicator=True,
        )

        remaining = remaining[remaining["_merge"] == "left_only"].drop(columns=["_merge"])

        remaining = remaining.sort_values(
            ["n_learned_imputed", "n_observed", "total_raw_plus_learned"],
            ascending=False,
        )

        needed = MAX_GENE_CELL_PAIRS - len(selected)

        selected = pd.concat(
            [selected, remaining.head(needed)],
            ignore_index=True,
        )

    selected = selected.head(MAX_GENE_CELL_PAIRS).reset_index(drop=True)

    return selected


def plot_raw_plus_imputed_panel(
    ax,
    cid,
    gid,
    raw_df,
    imputed_df=None,
    title="",
    show_legend=True,
):
    """
    Plot one panel:
      - cell/nucleus boundary
      - raw observed molecules
      - optional imputed molecules overlaid
    """
    cid = int(cid)
    gid = str(gid)

    raw_pair = get_pair_molecules(raw_df, cid, gid)

    if imputed_df is not None:
        imp_pair = get_pair_molecules(imputed_df, cid, gid)
    else:
        imp_pair = pd.DataFrame(columns=raw_pair.columns)

    combined = pd.concat(
        [raw_pair, imp_pair],
        ignore_index=True,
    ) if len(imp_pair) > 0 else raw_pair

    # Plot cell and nucleus boundaries.
    if cid in cell_polygons:
        _plot_polygon_boundary(
            ax,
            cell_polygons[cid],
            linestyle="-",
            linewidth=1.3,
            alpha=0.9,
        )

    if cid in nuc_polygons:
        _plot_polygon_boundary(
            ax,
            nuc_polygons[cid],
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )

    # Downsample only for visual clarity.
    raw_plot = downsample_points(
        raw_pair,
        max_points=MAX_RAW_POINTS_PER_PANEL,
        seed=42,
    )

    imp_plot = downsample_points(
        imp_pair,
        max_points=MAX_IMPUTED_POINTS_PER_PANEL,
        seed=43,
    )

    # Raw observed molecules.
    if len(raw_plot) > 0:
        ax.scatter(
            pd.to_numeric(raw_plot["x"], errors="coerce"),
            pd.to_numeric(raw_plot["y"], errors="coerce"),
            s=RAW_POINT_SIZE,
            alpha=RAW_ALPHA,
            marker="o",
            label=f"raw observed n={len(raw_pair):,}",
        )

    # Imputed molecules.
    if len(imp_plot) > 0:
        ax.scatter(
            pd.to_numeric(imp_plot["x"], errors="coerce"),
            pd.to_numeric(imp_plot["y"], errors="coerce"),
            s=IMPUTED_POINT_SIZE,
            alpha=IMPUTED_ALPHA,
            marker="x",
            label=f"imputed n={len(imp_pair):,}",
        )

    bounds = _get_geom_bounds(cid, fallback_df=combined, padding=5.0)

    if bounds is not None:
        xmin, xmax, ymin, ymax = bounds
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linewidth=0.3, alpha=0.35)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    if show_legend:
        ax.legend(fontsize=7, loc="best", frameon=True)

    return {
        "n_raw_observed": int(len(raw_pair)),
        "n_imputed": int(len(imp_pair)),
        "n_completed": int(len(raw_pair) + len(imp_pair)),
    }


def plot_one_raw_completed_comparison(cid, gid, out_prefix):
    """
    Make a 5-panel figure:

      Panel 1: Raw observed
      Panel 2: Raw + learned 9E
      Panel 3: Raw + gene empirical
      Panel 4: Raw + cell-type gene empirical
      Panel 5: Raw + spatial-kNN empirical
    """
    cid = int(cid)
    gid = str(gid)

    panels = [
        {
            "title": "Raw observed",
            "imputed_df": None,
            "method": "raw_observed",
        },
        {
            "title": "Raw + learned 9E",
            "imputed_df": learned_imputed_df,
            "method": "learned_9E",
        },
        {
            "title": "Raw + gene empirical",
            "imputed_df": gene_emp_imputed_df,
            "method": "gene_emp",
        },
        {
            "title": "Raw + cell-type gene empirical",
            "imputed_df": ct_gene_emp_imputed_df,
            "method": "ct_gene_emp",
        },
        {
            "title": "Raw + spatial-kNN empirical",
            "imputed_df": spatial_knn_emp_imputed_df,
            "method": "spatial_knn_emp",
        },
    ]

    fig, axes = plt.subplots(
        1,
        len(panels),
        figsize=(4.3 * len(panels), 4.6),
        sharex=True,
        sharey=True,
    )

    summary_rows = []

    for ax, panel in zip(axes, panels):
        counts = plot_raw_plus_imputed_panel(
            ax=ax,
            cid=cid,
            gid=gid,
            raw_df=mol_observed,
            imputed_df=panel["imputed_df"],
            title=panel["title"],
            show_legend=True,
        )

        summary_rows.append({
            "cell_id": cid,
            "gene_id": gid,
            "method": panel["method"],
            "panel_title": panel["title"],
            **counts,
        })

    fig.suptitle(
        f"Raw vs completed molecule maps\nCell {cid} — Gene {gid}",
        fontsize=14,
        y=1.05,
    )

    plt.tight_layout()

    png_path = f"{out_prefix}.png"
    pdf_path = f"{out_prefix}.pdf"

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(summary_rows), png_path, pdf_path


# ------------------------------------------------------------------------------
# 5. Select cell-gene examples
# ------------------------------------------------------------------------------

selected_pairs_df = select_gene_cell_pairs_for_raw_completed()

print("\nSelected gene/cell pairs for raw-vs-completed dot maps:")
display(selected_pairs_df)

selected_pairs_df.to_csv(SELECTION_CSV_PATH, index=False)

print(f"Saved selected pairs:")
print(f"  {SELECTION_CSV_PATH}")

# ------------------------------------------------------------------------------
# 6. Generate figures
# ------------------------------------------------------------------------------

print("\nGenerating raw-vs-completed dot map figures...")

all_summary_rows = []
figure_records = []

for i, row in selected_pairs_df.iterrows():
    cid = int(row["cell_id"])
    gid = str(row["gene_id"])

    safe_gid = (
        gid.replace("/", "_")
        .replace(" ", "_")
        .replace("+", "plus")
        .replace(":", "_")
    )

    out_prefix = os.path.join(
        RAW_COMPLETED_VIS_DIR,
        f"{RUN_NAME}_raw_vs_completed_cell_{cid}_gene_{safe_gid}"
    )

    print(f"\n[{i + 1}/{len(selected_pairs_df)}] Plotting cell_id={cid}, gene_id={gid}")

    summary_df, png_path, pdf_path = plot_one_raw_completed_comparison(
        cid=cid,
        gid=gid,
        out_prefix=out_prefix,
    )

    all_summary_rows.append(summary_df)

    figure_records.append({
        "cell_id": cid,
        "gene_id": gid,
        "png_path": png_path,
        "pdf_path": pdf_path,
        "n_observed_selected": int(row.get("n_observed", -1)),
        "n_learned_imputed_selected": int(row.get("n_learned_imputed", -1)),
    })

# ------------------------------------------------------------------------------
# 7. Save summaries
# ------------------------------------------------------------------------------

raw_completed_summary_df = pd.concat(all_summary_rows, ignore_index=True)
figure_manifest_df = pd.DataFrame(figure_records)

raw_completed_summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
figure_manifest_df.to_csv(FIGURE_MANIFEST_PATH, index=False)

print("\nRaw-vs-completed dot map summary:")
display(raw_completed_summary_df.head(40))

print("\nFigure manifest:")
display(figure_manifest_df)

print("\nSaved raw-vs-completed dot map outputs:")
print(f"  Selected pairs CSV : {SELECTION_CSV_PATH}")
print(f"  Summary CSV        : {SUMMARY_CSV_PATH}")
print(f"  Figure manifest    : {FIGURE_MANIFEST_PATH}")
print(f"  Figure directory   : {RAW_COMPLETED_VIS_DIR}")

print("\n" + "=" * 100)
print("CELL 6B COMPLETE — Raw vs completed molecule dot maps")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 6C — Global raw vs completed molecule map
#
# Figure layout:
#   Panel 1: Raw observed molecules only
#   Panel 2: Completed molecules = raw observed + learned 9E imputed
#
# Purpose:
#   Tissue-level before/after visualization of molecule-level imputation.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 6C — Global raw vs completed molecule map")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from previous cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_maps"
)

os.makedirs(GLOBAL_MAP_DIR, exist_ok=True)

GLOBAL_RAW_COMPLETED_PNG = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_molecule_map.png"
)

GLOBAL_RAW_COMPLETED_PDF = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_molecule_map.pdf"
)

GLOBAL_MAP_SUMMARY_CSV = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_molecule_map_summary.csv"
)

print(f"GLOBAL_MAP_DIR: {GLOBAL_MAP_DIR}")

# ------------------------------------------------------------------------------
# 2. Plotting configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42

# Increase these if your runtime can handle it.
MAX_RAW_POINTS = 600_000
MAX_IMPUTED_POINTS = 300_000

# For completed panel, raw and imputed are plotted separately.
RAW_POINT_SIZE = 0.15
IMPUTED_POINT_SIZE = 0.25

RAW_ALPHA = 0.45
IMPUTED_ALPHA = 0.65

# If True, use gene colors. If False, raw/imputed use simple colors.
COLOR_BY_GENE = False

# For gene coloring, use only top genes to avoid too many random colors.
TOP_N_GENES_FOR_COLOR = 20

print(f"MAX_RAW_POINTS: {MAX_RAW_POINTS:,}")
print(f"MAX_IMPUTED_POINTS: {MAX_IMPUTED_POINTS:,}")
print(f"COLOR_BY_GENE: {COLOR_BY_GENE}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def sample_molecules_for_plot(df, max_points, seed=42):
    """
    Downsample molecule table for visualization.
    """
    if len(df) <= max_points:
        return df.copy()

    return df.sample(n=max_points, random_state=seed).copy()


def clean_xy(df):
    """
    Keep only rows with valid x/y.
    """
    out = df.copy()
    out["x"] = pd.to_numeric(out["x"], errors="coerce")
    out["y"] = pd.to_numeric(out["y"], errors="coerce")
    out = out.dropna(subset=["x", "y"])
    return out


def get_global_bounds(*dfs, padding_frac=0.03):
    """
    Use all provided dataframes to compute common x/y plot bounds.
    """
    xs = []
    ys = []

    for df in dfs:
        if df is None or len(df) == 0:
            continue

        xs.append(pd.to_numeric(df["x"], errors="coerce"))
        ys.append(pd.to_numeric(df["y"], errors="coerce"))

    x_all = pd.concat(xs, ignore_index=True).dropna()
    y_all = pd.concat(ys, ignore_index=True).dropna()

    xmin, xmax = float(x_all.min()), float(x_all.max())
    ymin, ymax = float(y_all.min()), float(y_all.max())

    xpad = (xmax - xmin) * padding_frac
    ypad = (ymax - ymin) * padding_frac

    return xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad


def build_gene_color_map(df, top_n=20):
    """
    Make a simple gene-to-color map for top genes.
    Other genes become gray.
    """
    top_genes = (
        df["gene_id"]
        .astype(str)
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )

    cmap = plt.cm.get_cmap("tab20", len(top_genes))

    gene_to_color = {
        gene: cmap(i)
        for i, gene in enumerate(top_genes)
    }

    return gene_to_color, top_genes


def plot_gene_colored_points(ax, df, gene_to_color, default_color="lightgray",
                             point_size=0.2, alpha=0.5, label_prefix=""):
    """
    Plot points colored by top genes.
    """
    d = df.copy()
    d["gene_id"] = d["gene_id"].astype(str)

    # Plot non-top genes first.
    top_genes = set(gene_to_color.keys())
    other = d[~d["gene_id"].isin(top_genes)]

    if len(other) > 0:
        ax.scatter(
            other["x"],
            other["y"],
            s=point_size,
            alpha=alpha * 0.4,
            c=default_color,
            linewidths=0,
            label=f"{label_prefix}other genes",
        )

    # Plot top genes.
    for gene, color in gene_to_color.items():
        sub = d[d["gene_id"] == gene]

        if len(sub) == 0:
            continue

        ax.scatter(
            sub["x"],
            sub["y"],
            s=point_size,
            alpha=alpha,
            c=[color],
            linewidths=0,
            label=f"{label_prefix}{gene}",
        )

# ------------------------------------------------------------------------------
# 4. Prepare sampled raw and imputed molecules
# ------------------------------------------------------------------------------

print("\nPreparing sampled molecule tables...")

raw_for_plot = mol_observed[
    mol_observed["status"].astype(str) == "observed"
].copy()

imputed_for_plot = learned_imputed_df.copy()

raw_for_plot = clean_xy(raw_for_plot)
imputed_for_plot = clean_xy(imputed_for_plot)

print(f"Raw observed molecules available : {len(raw_for_plot):,}")
print(f"Learned 9E imputed available     : {len(imputed_for_plot):,}")

raw_sample = sample_molecules_for_plot(
    raw_for_plot,
    max_points=MAX_RAW_POINTS,
    seed=RANDOM_STATE,
)

imputed_sample = sample_molecules_for_plot(
    imputed_for_plot,
    max_points=MAX_IMPUTED_POINTS,
    seed=RANDOM_STATE + 1,
)

completed_sample = pd.concat(
    [
        raw_sample.assign(plot_source="raw_observed"),
        imputed_sample.assign(plot_source="learned_9E_imputed"),
    ],
    ignore_index=True,
)

print(f"Raw sample      : {len(raw_sample):,}")
print(f"Imputed sample  : {len(imputed_sample):,}")
print(f"Completed sample: {len(completed_sample):,}")

xmin, xmax, ymin, ymax = get_global_bounds(raw_sample, imputed_sample)

print(f"Plot bounds: x=[{xmin:.1f}, {xmax:.1f}], y=[{ymin:.1f}, {ymax:.1f}]")

# ------------------------------------------------------------------------------
# 5. Plot raw vs completed
# ------------------------------------------------------------------------------

print("\nPlotting global raw vs completed molecule map...")

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 8),
    sharex=True,
    sharey=True,
)

# Panel 1: raw observed only.
ax = axes[0]

if COLOR_BY_GENE:
    gene_to_color, top_genes = build_gene_color_map(raw_sample, TOP_N_GENES_FOR_COLOR)
    plot_gene_colored_points(
        ax,
        raw_sample,
        gene_to_color=gene_to_color,
        point_size=RAW_POINT_SIZE,
        alpha=RAW_ALPHA,
        label_prefix="raw: ",
    )
else:
    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

ax.set_title(
    f"Raw observed molecules\nsample n={len(raw_sample):,} / total {len(raw_for_plot):,}",
    fontsize=13,
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(False)

# Panel 2: completed = raw + learned imputed.
ax = axes[1]

if COLOR_BY_GENE:
    gene_to_color, top_genes = build_gene_color_map(completed_sample, TOP_N_GENES_FOR_COLOR)

    # Raw in faint background.
    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA * 0.5,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

    # Imputed colored by gene.
    plot_gene_colored_points(
        ax,
        imputed_sample,
        gene_to_color=gene_to_color,
        point_size=IMPUTED_POINT_SIZE,
        alpha=IMPUTED_ALPHA,
        label_prefix="imputed: ",
    )
else:
    ax.scatter(
        raw_sample["x"],
        raw_sample["y"],
        s=RAW_POINT_SIZE,
        alpha=RAW_ALPHA * 0.45,
        linewidths=0,
        label=f"raw observed sample n={len(raw_sample):,}",
    )

    ax.scatter(
        imputed_sample["x"],
        imputed_sample["y"],
        s=IMPUTED_POINT_SIZE,
        alpha=IMPUTED_ALPHA,
        marker="x",
        linewidths=0.25,
        label=f"9E imputed sample n={len(imputed_sample):,}",
    )

ax.set_title(
    f"Completed molecule map: raw + learned 9E imputed\n"
    f"sample n={len(completed_sample):,} / total {len(raw_for_plot) + len(imputed_for_plot):,}",
    fontsize=13,
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(False)
ax.legend(loc="best", fontsize=8, markerscale=4)

fig.suptitle(
    "Global tissue-level molecule map before and after learned 9E imputation",
    fontsize=16,
    y=1.02,
)

plt.tight_layout()

plt.savefig(GLOBAL_RAW_COMPLETED_PNG, dpi=300, bbox_inches="tight")
plt.savefig(GLOBAL_RAW_COMPLETED_PDF, bbox_inches="tight")
plt.show()

print(f"Saved PNG: {GLOBAL_RAW_COMPLETED_PNG}")
print(f"Saved PDF: {GLOBAL_RAW_COMPLETED_PDF}")

# ------------------------------------------------------------------------------
# 6. Save summary
# ------------------------------------------------------------------------------

summary = {
    "raw_total_molecules": int(len(raw_for_plot)),
    "learned_9E_imputed_total_molecules": int(len(imputed_for_plot)),
    "completed_total_molecules": int(len(raw_for_plot) + len(imputed_for_plot)),
    "raw_sampled_molecules": int(len(raw_sample)),
    "imputed_sampled_molecules": int(len(imputed_sample)),
    "completed_sampled_molecules": int(len(completed_sample)),
    "max_raw_points": int(MAX_RAW_POINTS),
    "max_imputed_points": int(MAX_IMPUTED_POINTS),
    "random_state": int(RANDOM_STATE),
    "color_by_gene": bool(COLOR_BY_GENE),
    "png_path": GLOBAL_RAW_COMPLETED_PNG,
    "pdf_path": GLOBAL_RAW_COMPLETED_PDF,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(GLOBAL_MAP_SUMMARY_CSV, index=False)

print("\nGlobal map summary:")
display(summary_df)

print(f"Saved summary CSV: {GLOBAL_MAP_SUMMARY_CSV}")

print("\n" + "=" * 100)
print("CELL 6C COMPLETE — Global raw vs completed molecule map")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 6D_FIXED — All-molecule raw vs completed Datashader map + side-by-side image
#
# This cell replaces separate 6D and 6E cells.
#
# It plots ALL molecules, not sampled molecules:
#   Panel 1: Raw observed molecules
#   Panel 2: Completed = raw observed + learned 9E imputed molecules
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd

from PIL import Image, ImageDraw

print("=" * 100)
print("CELL 6D_FIXED — All-molecule raw vs completed Datashader map + side-by-side image")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_maps"
)

os.makedirs(GLOBAL_MAP_DIR, exist_ok=True)

RAW_ALL_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_observed_datashader"
)

COMPLETED_ALL_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_completed_raw_plus_9E_datashader"
)

raw_png = RAW_ALL_PATH + ".png"
completed_png = COMPLETED_ALL_PATH + ".png"

combined_png = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_datashader_side_by_side.png"
)

SUMMARY_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_datashader_summary.csv"
)

print(f"GLOBAL_MAP_DIR : {GLOBAL_MAP_DIR}")
print(f"Raw image      : {raw_png}")
print(f"Completed image: {completed_png}")
print(f"Combined image : {combined_png}")

# ------------------------------------------------------------------------------
# 3. Prepare all molecule coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

raw_all = mol_observed[
    mol_observed["status"].astype(str) == "observed"
][["x", "y"]].copy()

imputed_all = learned_imputed_df[["x", "y"]].copy()

# Convert x/y to numeric and reduce memory.
raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

raw_all["source"] = "raw"
imputed_all["source"] = "imputed"

# Datashader count_cat needs categorical dtype.
raw_all["source"] = raw_all["source"].astype("category")
imputed_all["source"] = imputed_all["source"].astype("category")

completed_all = pd.concat([raw_all, imputed_all], ignore_index=True)
completed_all["source"] = completed_all["source"].astype("category")

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Render all-molecule maps using Datashader
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

# Raw-only image.
print("\nRendering raw observed all-molecule image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    RAW_ALL_PATH,
    fmt=".png",
)

print(f"Saved raw image:")
print(f"  {raw_png}")

# Completed raw + imputed image.
print("\nRendering completed raw + learned 9E imputed all-molecule image...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count_cat("source"),
)

img_completed = tf.shade(
    agg_completed,
    color_key={
        "raw": "lightgray",
        "imputed": "red",
    },
    how="eq_hist",
)

export_image(
    img_completed,
    COMPLETED_ALL_PATH,
    fmt=".png",
)

print(f"Saved completed image:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 6. Combine side by side
# ------------------------------------------------------------------------------

print("\nCombining raw and completed images side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 100
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text((30, 58), f"n = {len(raw_all):,}", fill="gray")
draw.text((w + gap + 30, 58), f"raw n = {len(raw_all):,}, imputed n = {len(imputed_all):,}", fill="gray")

combined.save(combined_png)

print(f"Saved side-by-side image:")
print(f"  {combined_png}")

display(combined)

# ------------------------------------------------------------------------------
# 7. Save summary
# ------------------------------------------------------------------------------

summary = {
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "raw_image": raw_png,
    "completed_image": completed_png,
    "combined_image": combined_png,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\nSummary:")
display(summary_df)

print(f"Saved summary:")
print(f"  {SUMMARY_PATH}")

# ------------------------------------------------------------------------------
# 8. Cleanup
# ------------------------------------------------------------------------------

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("CELL 6D_FIXED COMPLETE — all-molecule raw vs completed map saved")
print("=" * 100)

In [ ]:
# ==============================================================================
# CELL 6D_BW — All-molecule raw vs completed black/white Datashader map
#
# Panel 1: Raw observed molecules
# Panel 2: Completed = raw observed + learned 9E imputed
#
# Both panels use black background + white molecule density.
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("CELL 6D_BW — Black/white all-molecule raw vs completed Datashader map")
print("=" * 100)

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

GLOBAL_MAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_maps"
)
os.makedirs(GLOBAL_MAP_DIR, exist_ok=True)

RAW_BW_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_observed_BW_datashader"
)

COMPLETED_BW_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_completed_raw_plus_9E_BW_datashader"
)

COMBINED_BW_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_BW_side_by_side.png"
)

SUMMARY_BW_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_BW_summary.csv"
)

print("\nPreparing coordinates...")

raw_all = mol_observed[
    mol_observed["status"].astype(str) == "observed"
][["x", "y"]].copy()

imputed_all = learned_imputed_df[["x", "y"]].copy()

raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")
imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

# Completed table = raw + imputed coordinates.
completed_all = pd.concat(
    [
        raw_all,
        imputed_all,
    ],
    ignore_index=True,
)

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

print("\nRendering raw observed black/white image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    RAW_BW_PATH,
    fmt=".png",
)

print(f"Saved raw BW image: {RAW_BW_PATH}.png")

print("\nRendering completed black/white image...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_completed = tf.shade(
    agg_completed,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_completed,
    COMPLETED_BW_PATH,
    fmt=".png",
)

print(f"Saved completed BW image: {COMPLETED_BW_PATH}.png")

# Combine side-by-side.
raw_png = RAW_BW_PATH + ".png"
completed_png = COMPLETED_BW_PATH + ".png"

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    img2 = img2.resize(img1.size)

w, h = img1.size
title_h = 100
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text((30, 58), f"n = {len(raw_all):,}", fill="gray")
draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}, imputed n = {len(imputed_all):,}",
    fill="gray",
)

combined.save(COMBINED_BW_PATH)

display(combined)

print(f"\nSaved side-by-side BW image:")
print(f"  {COMBINED_BW_PATH}")

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "raw_bw_image": raw_png,
    "completed_bw_image": completed_png,
    "combined_bw_image": COMBINED_BW_PATH,
    "plot_width": WIDTH,
    "plot_height": HEIGHT,
}])

summary_df.to_csv(SUMMARY_BW_PATH, index=False)

print(f"Saved summary:")
print(f"  {SUMMARY_BW_PATH}")

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("CELL 6D_BW COMPLETE")
print("=" * 100)

In [ ]:
# ==============================================================================
# CELL 6D_OPTION2 — All-molecule raw vs completed grayscale Datashader map
#
# Option 2:
#   Panel 1: Raw observed molecules
#            black background + white raw density
#
#   Panel 2: Completed = raw observed + learned 9E imputed
#            black background
#            raw molecules     = dim gray
#            imputed molecules = white
#
# Purpose:
#   Tissue-level before/after visualization of molecule-level imputation,
#   while still distinguishing raw vs imputed molecules without using pink/red.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("CELL 6D_OPTION2 — All-molecule raw vs completed grayscale Datashader map")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_maps"
)

os.makedirs(GLOBAL_MAP_DIR, exist_ok=True)

RAW_OPTION2_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_observed_option2_BW_datashader"
)

COMPLETED_OPTION2_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_completed_raw_dimgray_imputed_white_datashader"
)

COMBINED_OPTION2_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_dimgray_white_side_by_side.png"
)

SUMMARY_OPTION2_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_dimgray_white_summary.csv"
)

print(f"GLOBAL_MAP_DIR       : {GLOBAL_MAP_DIR}")
print(f"Raw image path       : {RAW_OPTION2_PATH}.png")
print(f"Completed image path : {COMPLETED_OPTION2_PATH}.png")
print(f"Combined image path  : {COMBINED_OPTION2_PATH}")

# ------------------------------------------------------------------------------
# 3. Prepare all raw and imputed molecule coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

raw_all = mol_observed[
    mol_observed["status"].astype(str) == "observed"
][["x", "y"]].copy()

imputed_all = learned_imputed_df[["x", "y"]].copy()

# Convert x/y to numeric float32 to reduce memory.
raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

# Remove invalid coordinate rows.
raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

# Add source labels for completed panel.
# This is the key difference from the pure black/white density version.
raw_all["source"] = "raw"
imputed_all["source"] = "imputed"

completed_all = pd.concat(
    [
        raw_all,
        imputed_all,
    ],
    ignore_index=True,
)

# Datashader count_cat needs categorical dtype.
completed_all["source"] = completed_all["source"].astype("category")

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")
print(f"Completed points    : {len(completed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

xmin = float(completed_all["x"].min())
xmax = float(completed_all["x"].max())
ymin = float(completed_all["y"].min())
ymax = float(completed_all["y"].max())

xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03

x_range = (xmin - xpad, xmax + xpad)
y_range = (ymin - ypad, ymax + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Canvas settings
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

# ------------------------------------------------------------------------------
# 6. Render Panel 1: raw observed black/white density
# ------------------------------------------------------------------------------

print("\nRendering Panel 1: raw observed black/white density image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    RAW_OPTION2_PATH,
    fmt=".png",
)

raw_png = RAW_OPTION2_PATH + ".png"

print(f"Saved raw image:")
print(f"  {raw_png}")

# ------------------------------------------------------------------------------
# 7. Render Panel 2: completed grayscale raw=dimgray, imputed=white
# ------------------------------------------------------------------------------

print("\nRendering Panel 2: completed grayscale image with raw=dimgray and imputed=white...")

agg_completed = canvas.points(
    completed_all,
    x="x",
    y="y",
    agg=ds.count_cat("source"),
)

img_completed = tf.shade(
    agg_completed,
    color_key={
        "raw": "dimgray",
        "imputed": "white",
    },
    how="eq_hist",
)

export_image(
    img_completed,
    COMPLETED_OPTION2_PATH,
    fmt=".png",
)

completed_png = COMPLETED_OPTION2_PATH + ".png"

print(f"Saved completed image:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 8. Combine side by side
# ------------------------------------------------------------------------------

print("\nCombining raw and completed images side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    print(f"WARNING: image sizes differ: raw={img1.size}, completed={img2.size}")
    print("Resizing completed image to match raw image size.")
    img2 = img2.resize(img1.size)

w, h = img1.size

title_h = 110
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text(
    (30, 58),
    f"raw observed n = {len(raw_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}; imputed n = {len(imputed_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 82),
    "raw = dim gray; imputed = white",
    fill="gray",
)

combined.save(COMBINED_OPTION2_PATH)

print(f"Saved side-by-side image:")
print(f"  {COMBINED_OPTION2_PATH}")

display(combined)

# ------------------------------------------------------------------------------
# 9. Save summary
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(completed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "raw_image": raw_png,
    "completed_image": completed_png,
    "combined_image": COMBINED_OPTION2_PATH,
    "completed_panel_raw_color": "dimgray",
    "completed_panel_imputed_color": "white",
    "background_color": "black",
}])

summary_df.to_csv(SUMMARY_OPTION2_PATH, index=False)

print("\nSummary:")
display(summary_df)

print(f"Saved summary:")
print(f"  {SUMMARY_OPTION2_PATH}")

# ------------------------------------------------------------------------------
# 10. Cleanup
# ------------------------------------------------------------------------------

del completed_all
gc.collect()

print("\n" + "=" * 100)
print("CELL 6D_OPTION2 COMPLETE — grayscale raw/imputed all-molecule map saved")
print("=" * 100)

In [ ]:
# ==============================================================================
# CELL 6D_OPTION4 — All-molecule raw vs completed map
#
# Panel 1:
#   Raw observed molecules
#   black background + white raw density
#
# Panel 2:
#   Completed = raw observed + learned 9E imputed
#   black background
#   white raw density, rendered exactly like Panel 1
#   imputed molecules overlaid in red
#
# Important:
#   Panel 2 is NOT using color_key raw=white/imputed=red directly.
#   Instead:
#     1. render raw density as black/white image
#     2. render imputed density as red transparent layer
#     3. alpha-composite red imputed layer on top of raw black/white layer
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

print("=" * 100)
print("CELL 6D_OPTION4 — Raw density + red imputed overlay")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import Datashader
# ------------------------------------------------------------------------------

try:
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image
except Exception:
    print("Installing datashader...")
    !pip install -q datashader colorcet
    import datashader as ds
    import datashader.transfer_functions as tf
    from datashader.utils import export_image

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable: {v}")

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

GLOBAL_MAP_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_global_raw_vs_completed_maps"
)
os.makedirs(GLOBAL_MAP_DIR, exist_ok=True)

RAW_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_observed_option4_BW_datashader"
)

RAW_FOR_COMPLETED_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_density_for_completed_option4_BW_datashader"
)

IMPUTED_RED_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_imputed_red_overlay_option4_datashader"
)

COMPLETED_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_completed_raw_BW_plus_imputed_red_overlay"
)

COMBINED_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_BW_red_overlay_side_by_side.png"
)

SUMMARY_OPTION4_PATH = os.path.join(
    GLOBAL_MAP_DIR,
    f"{RUN_NAME}_ALL_raw_vs_completed_BW_red_overlay_summary.csv"
)

print(f"GLOBAL_MAP_DIR       : {GLOBAL_MAP_DIR}")
print(f"Panel 1 raw image    : {RAW_OPTION4_PATH}.png")
print(f"Panel 2 completed    : {COMPLETED_OPTION4_PATH}.png")
print(f"Combined image       : {COMBINED_OPTION4_PATH}")

# ------------------------------------------------------------------------------
# 3. Prepare all raw and imputed coordinates
# ------------------------------------------------------------------------------

print("\nPreparing all raw and imputed molecule coordinates...")

raw_all = mol_observed[
    mol_observed["status"].astype(str) == "observed"
][["x", "y"]].copy()

imputed_all = learned_imputed_df[["x", "y"]].copy()

raw_all["x"] = pd.to_numeric(raw_all["x"], errors="coerce").astype("float32")
raw_all["y"] = pd.to_numeric(raw_all["y"], errors="coerce").astype("float32")

imputed_all["x"] = pd.to_numeric(imputed_all["x"], errors="coerce").astype("float32")
imputed_all["y"] = pd.to_numeric(imputed_all["y"], errors="coerce").astype("float32")

raw_all = raw_all.dropna(subset=["x", "y"])
imputed_all = imputed_all.dropna(subset=["x", "y"])

print(f"Raw observed points : {len(raw_all):,}")
print(f"9E imputed points   : {len(imputed_all):,}")

# ------------------------------------------------------------------------------
# 4. Shared plot bounds
# ------------------------------------------------------------------------------

all_x_min = min(float(raw_all["x"].min()), float(imputed_all["x"].min()))
all_x_max = max(float(raw_all["x"].max()), float(imputed_all["x"].max()))
all_y_min = min(float(raw_all["y"].min()), float(imputed_all["y"].min()))
all_y_max = max(float(raw_all["y"].max()), float(imputed_all["y"].max()))

xpad = (all_x_max - all_x_min) * 0.03
ypad = (all_y_max - all_y_min) * 0.03

x_range = (all_x_min - xpad, all_x_max + xpad)
y_range = (all_y_min - ypad, all_y_max + ypad)

print(f"x_range: {x_range}")
print(f"y_range: {y_range}")

# ------------------------------------------------------------------------------
# 5. Canvas settings
# ------------------------------------------------------------------------------

WIDTH = 1600
HEIGHT = 1200

canvas = ds.Canvas(
    plot_width=WIDTH,
    plot_height=HEIGHT,
    x_range=x_range,
    y_range=y_range,
)

# ------------------------------------------------------------------------------
# 6. Render raw density image for Panel 1
# ------------------------------------------------------------------------------

print("\nRendering Panel 1: raw observed black/white density image...")

agg_raw = canvas.points(
    raw_all,
    x="x",
    y="y",
    agg=ds.count(),
)

img_raw = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw,
    RAW_OPTION4_PATH,
    fmt=".png",
)

raw_png = RAW_OPTION4_PATH + ".png"

print(f"Saved Panel 1 raw image:")
print(f"  {raw_png}")

# ------------------------------------------------------------------------------
# 7. Render raw density image again for Panel 2 background
# ------------------------------------------------------------------------------

print("\nRendering Panel 2 background: raw black/white density image...")

# Use the same agg_raw and same shading so raw density in Panel 2 matches Panel 1.
img_raw_for_completed = tf.shade(
    agg_raw,
    cmap=["black", "white"],
    how="eq_hist",
)

export_image(
    img_raw_for_completed,
    RAW_FOR_COMPLETED_OPTION4_PATH,
    fmt=".png",
)

raw_for_completed_png = RAW_FOR_COMPLETED_OPTION4_PATH + ".png"

print(f"Saved raw background for completed panel:")
print(f"  {raw_for_completed_png}")

# ------------------------------------------------------------------------------
# 8. Render imputed molecules as red overlay
# ------------------------------------------------------------------------------

print("\nRendering imputed molecules as red overlay...")

agg_imputed = canvas.points(
    imputed_all,
    x="x",
    y="y",
    agg=ds.count(),
)

# Red density on transparent/black-compatible layer.
# span controls sensitivity; lower max can make red more visible.
img_imputed_red = tf.shade(
    agg_imputed,
    cmap=["black", "red"],
    how="eq_hist",
)

export_image(
    img_imputed_red,
    IMPUTED_RED_OPTION4_PATH,
    fmt=".png",
)

imputed_red_png = IMPUTED_RED_OPTION4_PATH + ".png"

print(f"Saved red imputed overlay image:")
print(f"  {imputed_red_png}")

# ------------------------------------------------------------------------------
# 9. Alpha-composite red imputed layer over raw black/white density
# ------------------------------------------------------------------------------

print("\nCompositing completed panel: raw black/white density + red imputed overlay...")

raw_bg = Image.open(raw_for_completed_png).convert("RGBA")
red_layer = Image.open(imputed_red_png).convert("RGBA")

if raw_bg.size != red_layer.size:
    red_layer = red_layer.resize(raw_bg.size)

# Convert red_layer so black pixels become transparent, red pixels stay visible.
red_arr = np.array(red_layer).astype(np.uint8)

r = red_arr[:, :, 0].astype(np.int16)
g = red_arr[:, :, 1].astype(np.int16)
b = red_arr[:, :, 2].astype(np.int16)

# Identify red-like pixels. Datashader creates black background and red density.
red_strength = np.maximum(r - np.maximum(g, b), 0)

# Alpha controls red overlay strength.
# Increase multiplier if red is too faint; decrease if red is too dominant.
alpha = np.clip(red_strength * 2.5, 0, 255).astype(np.uint8)

red_overlay = np.zeros_like(red_arr)
red_overlay[:, :, 0] = 255
red_overlay[:, :, 1] = 0
red_overlay[:, :, 2] = 0
red_overlay[:, :, 3] = alpha

red_overlay_img = Image.fromarray(red_overlay, mode="RGBA")

completed_img = Image.alpha_composite(raw_bg, red_overlay_img).convert("RGB")
completed_img.save(COMPLETED_OPTION4_PATH + ".png")

completed_png = COMPLETED_OPTION4_PATH + ".png"

print(f"Saved completed panel:")
print(f"  {completed_png}")

# ------------------------------------------------------------------------------
# 10. Combine Panel 1 and Panel 2 side by side
# ------------------------------------------------------------------------------

print("\nCombining Panel 1 and Panel 2 side by side...")

if not os.path.exists(raw_png):
    raise FileNotFoundError(f"Raw image not found after export: {raw_png}")

if not os.path.exists(completed_png):
    raise FileNotFoundError(f"Completed image not found after export: {completed_png}")

img1 = Image.open(raw_png).convert("RGB")
img2 = Image.open(completed_png).convert("RGB")

if img1.size != img2.size:
    print(f"WARNING: image sizes differ: raw={img1.size}, completed={img2.size}")
    print("Resizing completed image to match raw image size.")
    img2 = img2.resize(img1.size)

w, h = img1.size
title_h = 110
gap = 20

combined = Image.new("RGB", (2 * w + gap, h + title_h), "black")

combined.paste(img1, (0, title_h))
combined.paste(img2, (w + gap, title_h))

draw = ImageDraw.Draw(combined)

draw.text((30, 25), "Raw observed molecules", fill="white")
draw.text((w + gap + 30, 25), "Completed: raw + learned 9E imputed", fill="white")

draw.text(
    (30, 58),
    f"raw observed n = {len(raw_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 58),
    f"raw n = {len(raw_all):,}; imputed n = {len(imputed_all):,}",
    fill="gray",
)

draw.text(
    (w + gap + 30, 82),
    "raw density = white; imputed molecules = red",
    fill="gray",
)

combined.save(COMBINED_OPTION4_PATH)

print(f"Saved side-by-side image:")
print(f"  {COMBINED_OPTION4_PATH}")

display(combined)

# ------------------------------------------------------------------------------
# 11. Save summary
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([{
    "raw_observed_points": int(len(raw_all)),
    "learned_9E_imputed_points": int(len(imputed_all)),
    "completed_points": int(len(raw_all) + len(imputed_all)),
    "plot_width": int(WIDTH),
    "plot_height": int(HEIGHT),
    "x_min": float(x_range[0]),
    "x_max": float(x_range[1]),
    "y_min": float(y_range[0]),
    "y_max": float(y_range[1]),
    "panel1_raw_image": raw_png,
    "panel2_raw_background_image": raw_for_completed_png,
    "panel2_imputed_red_overlay_image": imputed_red_png,
    "panel2_completed_image": completed_png,
    "combined_image": COMBINED_OPTION4_PATH,
    "panel1_description": "raw observed, black background, white density",
    "panel2_description": "raw density same as panel 1 plus red imputed molecule overlay",
}])

summary_df.to_csv(SUMMARY_OPTION4_PATH, index=False)

print("\nSummary:")
display(summary_df)

print(f"Saved summary:")
print(f"  {SUMMARY_OPTION4_PATH}")

# ------------------------------------------------------------------------------
# 12. Cleanup
# ------------------------------------------------------------------------------

gc.collect()

print("\n" + "=" * 100)
print("CELL 6D_OPTION4 COMPLETE — raw density + red imputed overlay saved")
print("=" * 100)

========RAN TILL THIS========

In [ ]:
# ==============================================================================
# CELL 7 — Nuclear fraction / marker localization validation
#
# Biological question:
#   Are nuclear-associated marker genes placed in biologically plausible
#   nuclear/perinuclear locations after imputation?
#
# Main comparison:
#   Raw observed
#   Learned 9E imputed / completed
#   Gene empirical imputed / completed
#   Cell-type gene empirical imputed / completed
#   Spatial-kNN empirical imputed / completed
#
# Main metric:
#   nuclear_fraction = number of nuclear molecules / total molecules
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 7 — Nuclear fraction / marker localization validation")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "gene_emp_imputed_df",
    "ct_gene_emp_imputed_df",
    "spatial_knn_emp_imputed_df",
    "shared_genes",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from reload/downstream cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

NUCLEAR_VAL_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_nuclear_fraction_validation"
)
os.makedirs(NUCLEAR_VAL_DIR, exist_ok=True)

NUCLEAR_GENE_SUMMARY_PATH = os.path.join(
    NUCLEAR_VAL_DIR,
    f"{RUN_NAME}_nuclear_fraction_gene_summary.csv"
)

NUCLEAR_MARKER_SUMMARY_PATH = os.path.join(
    NUCLEAR_VAL_DIR,
    f"{RUN_NAME}_nuclear_fraction_marker_summary.csv"
)

NUCLEAR_METHOD_SUMMARY_PATH = os.path.join(
    NUCLEAR_VAL_DIR,
    f"{RUN_NAME}_nuclear_fraction_method_summary.csv"
)

NUCLEAR_HEATMAP_PATH = os.path.join(
    NUCLEAR_VAL_DIR,
    f"{RUN_NAME}_nuclear_fraction_marker_heatmap.png"
)

NUCLEAR_DELTA_HEATMAP_PATH = os.path.join(
    NUCLEAR_VAL_DIR,
    f"{RUN_NAME}_nuclear_fraction_delta_vs_raw_heatmap.png"
)

print(f"NUCLEAR_VAL_DIR: {NUCLEAR_VAL_DIR}")

# ------------------------------------------------------------------------------
# 2. Marker genes
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

# These are useful Xenium-panel nuclear-associated / transcription-factor-like
# marker genes for your breast cancer dataset.
candidate_nuclear_markers = [
    "ESR1", "PGR", "AR", "GATA3", "FOXA1",
    "MKI67", "TOP2A", "CCND1", "IRF7",
]

# Some broadly cytoplasmic/membrane/ECM-like genes to serve as comparison/background.
candidate_non_nuclear_reference = [
    "EPCAM", "KRT7", "KRT8", "KRT18", "ERBB2",
    "LUM", "POSTN", "CXCL12", "CAV1", "PECAM1",
    "VWF", "CD3D", "CD68", "TYROBP",
]

nuclear_marker_genes = [g for g in candidate_nuclear_markers if g in available_genes]
non_nuclear_reference_genes = [g for g in candidate_non_nuclear_reference if g in available_genes]

print(f"Nuclear-associated marker genes present: {nuclear_marker_genes}")
print(f"Reference/background genes present     : {non_nuclear_reference_genes}")

if len(nuclear_marker_genes) == 0:
    raise RuntimeError("No nuclear-associated marker genes were found in shared_genes.")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def get_nuclear_indicator(df):
    """
    Returns a numeric nuclear indicator for each molecule.

    Priority:
      1. p_nuclear column, if available
      2. overlaps_nucleus column, if available

    p_nuclear is allowed to be continuous, but in your tables it is often 0/1.
    """
    if "p_nuclear" in df.columns:
        return pd.to_numeric(df["p_nuclear"], errors="coerce").fillna(0.0).astype(np.float32)

    if "overlaps_nucleus" in df.columns:
        return pd.to_numeric(df["overlaps_nucleus"], errors="coerce").fillna(0.0).astype(np.float32)

    raise KeyError("DataFrame has neither 'p_nuclear' nor 'overlaps_nucleus'.")


def aggregate_nuclear_by_gene(df, method_name, source_type):
    """
    Aggregate nuclear statistics per gene for one molecule table.

    source_type:
      raw_observed
      learned_imputed
      baseline_imputed
      completed
    """
    required_cols = ["gene_id"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"{method_name} missing required columns: {missing}")

    temp = pd.DataFrame({
        "gene_id": df["gene_id"].astype(str).values,
        "nuclear_value": get_nuclear_indicator(df).values,
    })

    out = (
        temp
        .groupby("gene_id", observed=True)
        .agg(
            n_molecules=("nuclear_value", "size"),
            nuclear_sum=("nuclear_value", "sum"),
            nuclear_fraction=("nuclear_value", "mean"),
        )
        .reset_index()
    )

    out["method"] = method_name
    out["source_type"] = source_type

    return out


def combine_raw_and_imputed_gene_aggs(raw_agg, imp_agg, completed_method_name):
    """
    Build completed per-gene nuclear fraction without concatenating huge tables.

    completed = raw observed + imputed molecules
    """
    raw_sub = raw_agg[["gene_id", "n_molecules", "nuclear_sum"]].rename(
        columns={
            "n_molecules": "raw_n_molecules",
            "nuclear_sum": "raw_nuclear_sum",
        }
    )

    imp_sub = imp_agg[["gene_id", "n_molecules", "nuclear_sum"]].rename(
        columns={
            "n_molecules": "imputed_n_molecules",
            "nuclear_sum": "imputed_nuclear_sum",
        }
    )

    merged = raw_sub.merge(imp_sub, on="gene_id", how="outer").fillna(0)

    merged["n_molecules"] = merged["raw_n_molecules"] + merged["imputed_n_molecules"]
    merged["nuclear_sum"] = merged["raw_nuclear_sum"] + merged["imputed_nuclear_sum"]
    merged["nuclear_fraction"] = merged["nuclear_sum"] / merged["n_molecules"].replace(0, np.nan)

    out = merged[["gene_id", "n_molecules", "nuclear_sum", "nuclear_fraction"]].copy()
    out["method"] = completed_method_name
    out["source_type"] = "completed"

    return out


def add_marker_labels(df):
    df = df.copy()
    df["is_nuclear_marker"] = df["gene_id"].isin(nuclear_marker_genes)
    df["is_reference_gene"] = df["gene_id"].isin(non_nuclear_reference_genes)
    return df


# ------------------------------------------------------------------------------
# 4. Aggregate raw, imputed, and completed nuclear fractions
# ------------------------------------------------------------------------------

print("\nAggregating nuclear fractions by gene...")

raw_gene_agg = aggregate_nuclear_by_gene(
    mol_observed[mol_observed["status"].astype(str) == "observed"].copy(),
    method_name="Raw observed",
    source_type="raw_observed",
)

learned_imp_agg = aggregate_nuclear_by_gene(
    learned_imputed_df,
    method_name="Learned 9E imputed only",
    source_type="learned_imputed",
)

gene_emp_imp_agg = aggregate_nuclear_by_gene(
    gene_emp_imputed_df,
    method_name="Gene empirical imputed only",
    source_type="baseline_imputed",
)

ct_gene_emp_imp_agg = aggregate_nuclear_by_gene(
    ct_gene_emp_imputed_df,
    method_name="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
)

spatial_knn_imp_agg = aggregate_nuclear_by_gene(
    spatial_knn_emp_imputed_df,
    method_name="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
)

learned_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    learned_imp_agg,
    completed_method_name="Learned 9E completed",
)

gene_emp_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    gene_emp_imp_agg,
    completed_method_name="Gene empirical completed",
)

ct_gene_emp_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    ct_gene_emp_imp_agg,
    completed_method_name="Cell-type gene empirical completed",
)

spatial_knn_completed_agg = combine_raw_and_imputed_gene_aggs(
    raw_gene_agg,
    spatial_knn_imp_agg,
    completed_method_name="Spatial-kNN empirical completed",
)

nuclear_gene_summary_df = pd.concat(
    [
        raw_gene_agg,
        learned_imp_agg,
        gene_emp_imp_agg,
        ct_gene_emp_imp_agg,
        spatial_knn_imp_agg,
        learned_completed_agg,
        gene_emp_completed_agg,
        ct_gene_emp_completed_agg,
        spatial_knn_completed_agg,
    ],
    ignore_index=True,
)

nuclear_gene_summary_df = add_marker_labels(nuclear_gene_summary_df)

print(f"nuclear_gene_summary_df shape: {nuclear_gene_summary_df.shape}")
display(nuclear_gene_summary_df.head(20))

# ------------------------------------------------------------------------------
# 5. Marker-only table
# ------------------------------------------------------------------------------

marker_methods_order = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

nuclear_marker_summary_df = nuclear_gene_summary_df[
    nuclear_gene_summary_df["gene_id"].isin(nuclear_marker_genes)
].copy()

nuclear_marker_summary_df["method"] = pd.Categorical(
    nuclear_marker_summary_df["method"],
    categories=marker_methods_order,
    ordered=True,
)

nuclear_marker_summary_df = nuclear_marker_summary_df.sort_values(
    ["gene_id", "method"]
).reset_index(drop=True)

print("\nNuclear marker summary:")
display(nuclear_marker_summary_df)

# ------------------------------------------------------------------------------
# 6. Method-level summary: nuclear markers vs reference/background genes
# ------------------------------------------------------------------------------

method_summary_rows = []

for method in marker_methods_order:
    sub = nuclear_gene_summary_df[nuclear_gene_summary_df["method"].astype(str) == method].copy()

    nuc = sub[sub["gene_id"].isin(nuclear_marker_genes)]
    ref = sub[sub["gene_id"].isin(non_nuclear_reference_genes)]

    method_summary_rows.append({
        "method": method,
        "n_nuclear_marker_genes": int(len(nuc)),
        "n_reference_genes": int(len(ref)),

        "mean_nuclear_fraction_nuclear_markers": float(nuc["nuclear_fraction"].mean()) if len(nuc) else np.nan,
        "median_nuclear_fraction_nuclear_markers": float(nuc["nuclear_fraction"].median()) if len(nuc) else np.nan,

        "mean_nuclear_fraction_reference_genes": float(ref["nuclear_fraction"].mean()) if len(ref) else np.nan,
        "median_nuclear_fraction_reference_genes": float(ref["nuclear_fraction"].median()) if len(ref) else np.nan,
    })

nuclear_method_summary_df = pd.DataFrame(method_summary_rows)

nuclear_method_summary_df["mean_marker_minus_reference"] = (
    nuclear_method_summary_df["mean_nuclear_fraction_nuclear_markers"]
    - nuclear_method_summary_df["mean_nuclear_fraction_reference_genes"]
)

nuclear_method_summary_df["median_marker_minus_reference"] = (
    nuclear_method_summary_df["median_nuclear_fraction_nuclear_markers"]
    - nuclear_method_summary_df["median_nuclear_fraction_reference_genes"]
)

# Add deltas vs raw observed.
raw_method_row = nuclear_method_summary_df[
    nuclear_method_summary_df["method"] == "Raw observed"
].iloc[0]

for col in [
    "mean_nuclear_fraction_nuclear_markers",
    "median_nuclear_fraction_nuclear_markers",
    "mean_marker_minus_reference",
    "median_marker_minus_reference",
]:
    nuclear_method_summary_df[f"delta_vs_raw_{col}"] = (
        nuclear_method_summary_df[col] - raw_method_row[col]
    )

print("\nMethod-level nuclear localization summary:")
display(nuclear_method_summary_df)

# ------------------------------------------------------------------------------
# 7. Heatmap: nuclear fraction for marker genes
# ------------------------------------------------------------------------------

heatmap_df = nuclear_marker_summary_df.pivot_table(
    index="gene_id",
    columns="method",
    values="nuclear_fraction",
    aggfunc="mean",
)

# Keep only readable methods in main heatmap.
main_heatmap_methods = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
    "Learned 9E completed",
]

main_heatmap_methods = [m for m in main_heatmap_methods if m in heatmap_df.columns]
heatmap_df = heatmap_df[main_heatmap_methods]

fig_w = max(10, 1.4 * len(main_heatmap_methods))
fig_h = max(4, 0.5 * len(heatmap_df.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = heatmap_df.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)

ax.set_title("Nuclear fraction of nuclear-associated marker genes", fontsize=14, pad=14)
ax.set_xlabel("Dataset / method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index, fontsize=10)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color="white" if val > 0.55 else "black")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Nuclear fraction", fontsize=11)

plt.tight_layout()
plt.savefig(NUCLEAR_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved nuclear fraction heatmap:")
print(f"  {NUCLEAR_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 8. Delta heatmap: method - raw observed
# ------------------------------------------------------------------------------

raw_values = heatmap_df["Raw observed"] if "Raw observed" in heatmap_df.columns else None

if raw_values is not None:
    delta_heatmap_df = heatmap_df.copy()

    for col in delta_heatmap_df.columns:
        delta_heatmap_df[col] = delta_heatmap_df[col] - raw_values

    # Drop raw column because it is always zero.
    if "Raw observed" in delta_heatmap_df.columns:
        delta_heatmap_df = delta_heatmap_df.drop(columns=["Raw observed"])

    fig_w = max(10, 1.4 * len(delta_heatmap_df.columns))
    fig_h = max(4, 0.5 * len(delta_heatmap_df.index) + 2)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    arr = delta_heatmap_df.values.astype(float)
    finite_vals = arr[np.isfinite(arr)]

    vmax = np.nanpercentile(np.abs(finite_vals), 95) if len(finite_vals) else 0.2
    vmax = max(float(vmax), 0.05)

    im = ax.imshow(
        arr,
        aspect="auto",
        vmin=-vmax,
        vmax=vmax,
        cmap="coolwarm",
    )

    ax.set_title("Change in nuclear fraction vs raw observed", fontsize=14, pad=14)
    ax.set_xlabel("Dataset / method")
    ax.set_ylabel("Marker gene")

    ax.set_xticks(np.arange(len(delta_heatmap_df.columns)))
    ax.set_xticklabels(delta_heatmap_df.columns, rotation=45, ha="right", fontsize=9)

    ax.set_yticks(np.arange(len(delta_heatmap_df.index)))
    ax.set_yticklabels(delta_heatmap_df.index, fontsize=10)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=8, color="black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Δ nuclear fraction vs raw", fontsize=11)

    plt.tight_layout()
    plt.savefig(NUCLEAR_DELTA_HEATMAP_PATH, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved nuclear delta heatmap:")
    print(f"  {NUCLEAR_DELTA_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 9. Save tables
# ------------------------------------------------------------------------------

nuclear_gene_summary_df.to_csv(NUCLEAR_GENE_SUMMARY_PATH, index=False)
nuclear_marker_summary_df.to_csv(NUCLEAR_MARKER_SUMMARY_PATH, index=False)
nuclear_method_summary_df.to_csv(NUCLEAR_METHOD_SUMMARY_PATH, index=False)

print("\nSaved nuclear localization validation outputs:")
print(f"  Gene summary   : {NUCLEAR_GENE_SUMMARY_PATH}")
print(f"  Marker summary : {NUCLEAR_MARKER_SUMMARY_PATH}")
print(f"  Method summary : {NUCLEAR_METHOD_SUMMARY_PATH}")
print(f"  Heatmap        : {NUCLEAR_HEATMAP_PATH}")
print(f"  Delta heatmap  : {NUCLEAR_DELTA_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: nuclear-associated markers have higher nuclear_fraction than reference genes.")
print("  Good sign 2: Learned 9E imputed/completed nuclear fractions are biologically plausible.")
print("  Good sign 3: Learned 9E is not blindly more nuclear for every gene; check marker-specific behavior.")
print("  Caution: very high nuclear fraction for all genes may indicate over-central placement.")

print("\n" + "=" * 100)
print("CELL 7 COMPLETE — Nuclear fraction / marker localization validation")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 8 — Radial distribution comparison for marker genes
#
# Biological question:
#   Does learned 9E capture subcellular radial localization better than
#   empirical baselines?
#
# Main comparison:
#   Learned 9E imputed-only radial distribution
#   Gene empirical imputed-only radial distribution
#   Cell-type gene empirical imputed-only radial distribution
#   Spatial-kNN empirical imputed-only radial distribution
#
# Reference:
#   Raw observed radial distribution for the same gene.
#
# Main metrics:
#   mean r_norm
#   median r_norm
#   central fraction: r_norm <= 0.35
#   peripheral fraction: r_norm >= 0.75
#   Wasserstein distance to raw observed distribution
#   KS statistic to raw observed distribution
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wasserstein_distance, ks_2samp

print("=" * 100)
print("CELL 8 — Radial distribution comparison for marker genes")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "gene_emp_imputed_df",
    "ct_gene_emp_imputed_df",
    "spatial_knn_emp_imputed_df",
    "shared_genes",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from reload/downstream cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

RADIAL_VAL_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_radial_distribution_validation"
)
os.makedirs(RADIAL_VAL_DIR, exist_ok=True)

RADIAL_GENE_SUMMARY_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_radial_distribution_gene_summary.csv"
)

RADIAL_DISTANCE_SUMMARY_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_radial_distribution_distance_to_raw.csv"
)

RADIAL_METHOD_SUMMARY_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_radial_distribution_method_summary.csv"
)

RADIAL_WASSERSTEIN_HEATMAP_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_radial_wasserstein_to_raw_heatmap.png"
)

RADIAL_MEAN_HEATMAP_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_mean_r_norm_marker_heatmap.png"
)

RADIAL_HIST_DIR = os.path.join(
    RADIAL_VAL_DIR,
    "marker_radial_histograms"
)
os.makedirs(RADIAL_HIST_DIR, exist_ok=True)

print(f"RADIAL_VAL_DIR: {RADIAL_VAL_DIR}")

# ------------------------------------------------------------------------------
# 2. Marker genes
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

candidate_marker_genes = [
    # Nuclear-associated / TF-like
    "ESR1", "PGR", "AR", "GATA3", "FOXA1", "MKI67", "TOP2A", "CCND1", "IRF7",

    # Tumor / epithelial
    "ERBB2", "EPCAM", "KRT7", "KRT8", "KRT18",

    # Stromal / ECM
    "LUM", "POSTN", "CXCL12", "FBLN1",

    # Immune / macrophage / T-cell
    "CD3D", "CD68", "TYROBP", "MS4A1", "PTPRC",

    # Endothelial / vascular
    "PECAM1", "VWF", "CAV1", "NOSTRIN",
]

marker_genes = [g for g in candidate_marker_genes if g in available_genes]

print(f"Marker genes present for radial validation: {marker_genes}")

if len(marker_genes) == 0:
    raise RuntimeError("No marker genes found in shared_genes.")

# To avoid overly crowded figures, make histograms for these priority genes.
priority_hist_genes = [
    g for g in [
        "GATA3", "FOXA1", "ESR1", "ERBB2", "EPCAM",
        "KRT7", "LUM", "POSTN", "CD3D", "CD68",
        "PECAM1", "VWF"
    ]
    if g in marker_genes
]

MAX_HIST_GENES = 12
priority_hist_genes = priority_hist_genes[:MAX_HIST_GENES]

print(f"Priority genes for radial histogram plots: {priority_hist_genes}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def clean_rnorm_table(df, method_name, source_type, genes):
    """
    Keep gene_id and r_norm for selected genes only.
    """
    if "gene_id" not in df.columns:
        raise KeyError(f"{method_name} is missing gene_id column.")
    if "r_norm" not in df.columns:
        raise KeyError(f"{method_name} is missing r_norm column.")

    out = df[["gene_id", "r_norm"]].copy()
    out["gene_id"] = out["gene_id"].astype(str)
    out = out[out["gene_id"].isin(genes)].copy()

    out["r_norm"] = pd.to_numeric(out["r_norm"], errors="coerce")
    out = out.dropna(subset=["r_norm"])

    # r_norm should be in [0, 1]. Clip small numeric errors.
    out["r_norm"] = out["r_norm"].clip(0, 1).astype(np.float32)

    out["method"] = method_name
    out["source_type"] = source_type

    return out


def summarize_rnorm_by_gene(df):
    """
    Summarize radial distribution per method/gene.
    """
    rows = []

    for (method, source_type, gene), sub in df.groupby(["method", "source_type", "gene_id"], observed=True):
        vals = sub["r_norm"].to_numpy(dtype=np.float32)

        if len(vals) == 0:
            continue

        rows.append({
            "method": method,
            "source_type": source_type,
            "gene_id": gene,
            "n_molecules": int(len(vals)),
            "mean_r_norm": float(np.mean(vals)),
            "median_r_norm": float(np.median(vals)),
            "std_r_norm": float(np.std(vals)),
            "q25_r_norm": float(np.quantile(vals, 0.25)),
            "q75_r_norm": float(np.quantile(vals, 0.75)),
            "central_fraction_r_le_0_35": float(np.mean(vals <= 0.35)),
            "mid_fraction_0_35_lt_r_lt_0_75": float(np.mean((vals > 0.35) & (vals < 0.75))),
            "peripheral_fraction_r_ge_0_75": float(np.mean(vals >= 0.75)),
        })

    return pd.DataFrame(rows)


def compute_distance_to_raw(all_rnorm_df):
    """
    Compare each method/gene radial distribution against raw observed distribution
    for the same gene.
    """
    rows = []

    raw_df = all_rnorm_df[all_rnorm_df["method"] == "Raw observed"].copy()

    method_names = [
        "Learned 9E imputed only",
        "Gene empirical imputed only",
        "Cell-type gene empirical imputed only",
        "Spatial-kNN empirical imputed only",
    ]

    for gene in marker_genes:
        raw_vals = raw_df[raw_df["gene_id"] == gene]["r_norm"].to_numpy(dtype=np.float32)

        if len(raw_vals) < 5:
            continue

        for method in method_names:
            vals = all_rnorm_df[
                (all_rnorm_df["method"] == method)
                & (all_rnorm_df["gene_id"] == gene)
            ]["r_norm"].to_numpy(dtype=np.float32)

            if len(vals) < 5:
                rows.append({
                    "gene_id": gene,
                    "method": method,
                    "n_raw": int(len(raw_vals)),
                    "n_method": int(len(vals)),
                    "wasserstein_to_raw": np.nan,
                    "ks_stat_to_raw": np.nan,
                    "ks_pvalue_to_raw": np.nan,
                    "mean_r_norm_raw": float(np.mean(raw_vals)),
                    "mean_r_norm_method": np.nan,
                    "delta_mean_r_norm_method_minus_raw": np.nan,
                })
                continue

            ks = ks_2samp(raw_vals, vals)

            rows.append({
                "gene_id": gene,
                "method": method,
                "n_raw": int(len(raw_vals)),
                "n_method": int(len(vals)),
                "wasserstein_to_raw": float(wasserstein_distance(raw_vals, vals)),
                "ks_stat_to_raw": float(ks.statistic),
                "ks_pvalue_to_raw": float(ks.pvalue),
                "mean_r_norm_raw": float(np.mean(raw_vals)),
                "mean_r_norm_method": float(np.mean(vals)),
                "delta_mean_r_norm_method_minus_raw": float(np.mean(vals) - np.mean(raw_vals)),
            })

    return pd.DataFrame(rows)


def plot_radial_histograms_for_gene(all_rnorm_df, gene, out_path):
    """
    Plot radial histograms for one gene:
      raw observed vs learned 9E vs baselines.
    """
    methods = [
        "Raw observed",
        "Learned 9E imputed only",
        "Gene empirical imputed only",
        "Cell-type gene empirical imputed only",
        "Spatial-kNN empirical imputed only",
    ]

    bins = np.linspace(0, 1, 31)

    fig, ax = plt.subplots(figsize=(9, 5))

    for method in methods:
        vals = all_rnorm_df[
            (all_rnorm_df["method"] == method)
            & (all_rnorm_df["gene_id"] == gene)
        ]["r_norm"].to_numpy(dtype=np.float32)

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=bins,
            density=True,
            histtype="step",
            linewidth=1.8,
            label=f"{method} (n={len(vals):,})",
        )

    ax.set_title(f"Radial distribution for marker gene {gene}", fontsize=14)
    ax.set_xlabel("r_norm: 0 = nucleus/center, 1 = cell edge")
    ax.set_ylabel("Density")
    ax.set_xlim(0, 1)
    ax.grid(True, linewidth=0.3, alpha=0.4)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved radial histogram for {gene}: {out_path}")


# ------------------------------------------------------------------------------
# 4. Build radial tables
# ------------------------------------------------------------------------------

print("\nBuilding radial distribution tables...")

raw_rnorm = clean_rnorm_table(
    mol_observed[mol_observed["status"].astype(str) == "observed"].copy(),
    method_name="Raw observed",
    source_type="raw_observed",
    genes=marker_genes,
)

learned_rnorm = clean_rnorm_table(
    learned_imputed_df,
    method_name="Learned 9E imputed only",
    source_type="learned_imputed",
    genes=marker_genes,
)

gene_emp_rnorm = clean_rnorm_table(
    gene_emp_imputed_df,
    method_name="Gene empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

ct_gene_emp_rnorm = clean_rnorm_table(
    ct_gene_emp_imputed_df,
    method_name="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

spatial_knn_rnorm = clean_rnorm_table(
    spatial_knn_emp_imputed_df,
    method_name="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
    genes=marker_genes,
)

all_rnorm_df = pd.concat(
    [
        raw_rnorm,
        learned_rnorm,
        gene_emp_rnorm,
        ct_gene_emp_rnorm,
        spatial_knn_rnorm,
    ],
    ignore_index=True,
)

print(f"all_rnorm_df shape: {all_rnorm_df.shape}")
print("Molecules per method:")
display(
    all_rnorm_df
    .groupby("method")
    .size()
    .reset_index(name="n_marker_molecules")
)

# ------------------------------------------------------------------------------
# 5. Per-gene radial summaries
# ------------------------------------------------------------------------------

radial_gene_summary_df = summarize_rnorm_by_gene(all_rnorm_df)

print("\nRadial gene summary:")
display(radial_gene_summary_df.head(30))

# ------------------------------------------------------------------------------
# 6. Distance to raw observed distribution
# ------------------------------------------------------------------------------

radial_distance_df = compute_distance_to_raw(all_rnorm_df)

print("\nDistance of imputed radial distributions to raw observed distribution:")
display(radial_distance_df.head(30))

# Method-level summary: lower distance is better.
radial_method_summary_df = (
    radial_distance_df
    .groupby("method", observed=True)
    .agg(
        n_genes_evaluated=("gene_id", "nunique"),
        mean_wasserstein_to_raw=("wasserstein_to_raw", "mean"),
        median_wasserstein_to_raw=("wasserstein_to_raw", "median"),
        mean_ks_stat_to_raw=("ks_stat_to_raw", "mean"),
        median_ks_stat_to_raw=("ks_stat_to_raw", "median"),
        mean_abs_delta_mean_r_norm=("delta_mean_r_norm_method_minus_raw", lambda x: float(np.nanmean(np.abs(x)))),
        median_abs_delta_mean_r_norm=("delta_mean_r_norm_method_minus_raw", lambda x: float(np.nanmedian(np.abs(x)))),
    )
    .reset_index()
)

radial_method_summary_df = radial_method_summary_df.sort_values(
    "mean_wasserstein_to_raw",
    ascending=True,
).reset_index(drop=True)

print("\nMethod-level radial distribution summary:")
display(radial_method_summary_df)

# ------------------------------------------------------------------------------
# 7. Ranking: which method is closest to raw for each gene?
# ------------------------------------------------------------------------------

rank_df = radial_distance_df.copy()

rank_df["wasserstein_rank_for_gene"] = (
    rank_df
    .groupby("gene_id")["wasserstein_to_raw"]
    .rank(method="min", ascending=True)
)

rank_df["is_best_for_gene"] = rank_df["wasserstein_rank_for_gene"] == 1

best_counts_df = (
    rank_df
    .groupby("method", observed=True)
    .agg(
        n_best_genes=("is_best_for_gene", "sum"),
        mean_rank=("wasserstein_rank_for_gene", "mean"),
    )
    .reset_index()
    .sort_values(["n_best_genes", "mean_rank"], ascending=[False, True])
)

print("\nBest method counts by gene based on Wasserstein distance to raw:")
display(best_counts_df)

# ------------------------------------------------------------------------------
# 8. Heatmap: Wasserstein distance to raw observed
# ------------------------------------------------------------------------------

wasserstein_matrix = radial_distance_df.pivot_table(
    index="gene_id",
    columns="method",
    values="wasserstein_to_raw",
    aggfunc="mean",
)

method_order = [
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
]

method_order = [m for m in method_order if m in wasserstein_matrix.columns]
wasserstein_matrix = wasserstein_matrix[method_order]

# Order genes by learned 9E distance if available.
if "Learned 9E imputed only" in wasserstein_matrix.columns:
    gene_order = wasserstein_matrix["Learned 9E imputed only"].sort_values().index
    wasserstein_matrix = wasserstein_matrix.loc[gene_order]

fig_w = max(9, 1.8 * len(method_order))
fig_h = max(6, 0.35 * len(wasserstein_matrix.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = wasserstein_matrix.values.astype(float)
vmax = np.nanpercentile(arr, 95) if np.isfinite(arr).any() else 0.2
vmax = max(float(vmax), 0.05)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=vmax,
    cmap="magma_r",
)

ax.set_title("Radial Wasserstein distance to raw observed distribution\nLower is better", fontsize=14, pad=14)
ax.set_xlabel("Imputation method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(wasserstein_matrix.columns)))
ax.set_xticklabels(wasserstein_matrix.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(wasserstein_matrix.index)))
ax.set_yticklabels(wasserstein_matrix.index, fontsize=9)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7, color="white" if val > 0.5 * vmax else "black")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Wasserstein distance to raw", fontsize=11)

plt.tight_layout()
plt.savefig(RADIAL_WASSERSTEIN_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved radial Wasserstein heatmap:")
print(f"  {RADIAL_WASSERSTEIN_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 9. Heatmap: mean r_norm by method
# ------------------------------------------------------------------------------

mean_r_matrix = radial_gene_summary_df.pivot_table(
    index="gene_id",
    columns="method",
    values="mean_r_norm",
    aggfunc="mean",
)

mean_r_method_order = [
    "Raw observed",
    "Learned 9E imputed only",
    "Gene empirical imputed only",
    "Cell-type gene empirical imputed only",
    "Spatial-kNN empirical imputed only",
]

mean_r_method_order = [m for m in mean_r_method_order if m in mean_r_matrix.columns]
mean_r_matrix = mean_r_matrix[mean_r_method_order]

if "Raw observed" in mean_r_matrix.columns:
    gene_order = mean_r_matrix["Raw observed"].sort_values().index
    mean_r_matrix = mean_r_matrix.loc[gene_order]

fig_w = max(10, 1.6 * len(mean_r_method_order))
fig_h = max(6, 0.35 * len(mean_r_matrix.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = mean_r_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="viridis",
)

ax.set_title("Mean radial position of marker genes\n0 = nuclear/central, 1 = cell edge", fontsize=14, pad=14)
ax.set_xlabel("Dataset / method")
ax.set_ylabel("Marker gene")

ax.set_xticks(np.arange(len(mean_r_matrix.columns)))
ax.set_xticklabels(mean_r_matrix.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(mean_r_matrix.index)))
ax.set_yticklabels(mean_r_matrix.index, fontsize=9)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color="white" if val > 0.55 else "black")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Mean r_norm", fontsize=11)

plt.tight_layout()
plt.savefig(RADIAL_MEAN_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved mean r_norm heatmap:")
print(f"  {RADIAL_MEAN_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 10. Histogram plots for selected marker genes
# ------------------------------------------------------------------------------

print("\nGenerating radial histogram plots for priority genes...")

hist_records = []

for gene in priority_hist_genes:
    out_path = os.path.join(
        RADIAL_HIST_DIR,
        f"{RUN_NAME}_radial_histogram_{gene}.png"
    )

    plot_radial_histograms_for_gene(
        all_rnorm_df=all_rnorm_df,
        gene=gene,
        out_path=out_path,
    )

    hist_records.append({
        "gene_id": gene,
        "histogram_path": out_path,
    })

radial_hist_manifest_df = pd.DataFrame(hist_records)

RADIAL_HIST_MANIFEST_PATH = os.path.join(
    RADIAL_VAL_DIR,
    f"{RUN_NAME}_radial_histogram_manifest.csv"
)
radial_hist_manifest_df.to_csv(RADIAL_HIST_MANIFEST_PATH, index=False)

# ------------------------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------------------------

radial_gene_summary_df.to_csv(RADIAL_GENE_SUMMARY_PATH, index=False)
radial_distance_df.to_csv(RADIAL_DISTANCE_SUMMARY_PATH, index=False)
radial_method_summary_df.to_csv(RADIAL_METHOD_SUMMARY_PATH, index=False)

print("\nSaved radial distribution validation outputs:")
print(f"  Gene summary           : {RADIAL_GENE_SUMMARY_PATH}")
print(f"  Distance to raw        : {RADIAL_DISTANCE_SUMMARY_PATH}")
print(f"  Method summary         : {RADIAL_METHOD_SUMMARY_PATH}")
print(f"  Wasserstein heatmap    : {RADIAL_WASSERSTEIN_HEATMAP_PATH}")
print(f"  Mean r_norm heatmap    : {RADIAL_MEAN_HEATMAP_PATH}")
print(f"  Histogram manifest     : {RADIAL_HIST_MANIFEST_PATH}")
print(f"  Histogram directory    : {RADIAL_HIST_DIR}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E has lower mean/median Wasserstein distance to raw than baselines.")
print("  Good sign 2: Learned 9E mean r_norm is close to raw marker-specific mean r_norm.")
print("  Good sign 3: For nuclear markers, learned 9E should not be pushed too peripheral.")
print("  Good sign 4: For membrane/ECM markers, learned 9E should not collapse everything to the nucleus.")
print("  Caution: Raw observed molecules are sparse/noisy, so use this with held-out recovery and nuclear validation.")

print("\n" + "=" * 100)
print("CELL 8 COMPLETE — Radial distribution comparison for marker genes")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 9 — Per-gene / per-cell-type localization summaries
#
# Biological question:
#   Which genes and cell types show meaningful localization patterns after
#   learned 9E imputation, and how do they compare with empirical baselines?
#
# Main comparison:
#   Raw observed
#   Learned 9E imputed / completed
#   Gene empirical imputed / completed
#   Cell-type gene empirical imputed / completed
#   Spatial-kNN empirical imputed / completed
#
# Main outputs:
#   Per (gene, cell type, method) summaries of:
#     - number of molecules
#     - mean r_norm
#     - nuclear fraction
#     - mean z_rel
#     - central/peripheral fractions
#     - imputed fraction for completed tables
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 100)
print("CELL 9 — Per-gene / per-cell-type localization summaries")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "gene_emp_imputed_df",
    "ct_gene_emp_imputed_df",
    "spatial_knn_emp_imputed_df",
    "shared_genes",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from reload/downstream cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

LOC_SUMMARY_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_per_gene_celltype_localization_summary"
)
os.makedirs(LOC_SUMMARY_DIR, exist_ok=True)

LOC_IMPUTED_ONLY_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_localization_summary_imputed_only_by_gene_celltype.csv"
)

LOC_COMPLETED_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_localization_summary_completed_by_gene_celltype.csv"
)

LOC_DELTA_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_localization_delta_learned_vs_baselines.csv"
)

LOC_METHOD_SUMMARY_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_localization_method_summary.csv"
)

LOC_NUC_HEATMAP_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_learned_completed_nuclear_fraction_gene_celltype_heatmap.png"
)

LOC_R_HEATMAP_PATH = os.path.join(
    LOC_SUMMARY_DIR,
    f"{RUN_NAME}_learned_completed_mean_r_norm_gene_celltype_heatmap.png"
)

print(f"LOC_SUMMARY_DIR: {LOC_SUMMARY_DIR}")

# ------------------------------------------------------------------------------
# 2. Marker genes and cell-type column helpers
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)

candidate_summary_genes = [
    # Nuclear / TF-like / proliferation
    "ESR1", "PGR", "AR", "GATA3", "FOXA1", "MKI67", "TOP2A", "CCND1", "IRF7",

    # Epithelial / tumor
    "ERBB2", "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19",

    # Stromal / ECM
    "LUM", "POSTN", "CXCL12", "FBLN1", "DCN", "COL1A1", "COL1A2",

    # Immune
    "CD3D", "CD3E", "CD8A", "MS4A1", "CD79A", "PTPRC",
    "CD68", "CD163", "TYROBP", "LYZ", "LST1",

    # Endothelial
    "PECAM1", "VWF", "CAV1", "NOSTRIN", "CLDN5", "KDR",
]

summary_genes = [g for g in candidate_summary_genes if g in available_genes]

if len(summary_genes) == 0:
    print("WARNING: none of the candidate genes were found. Using all shared genes.")
    summary_genes = [str(g) for g in shared_genes]

print(f"Genes used for focused heatmaps/summaries: {len(summary_genes)}")
print(summary_genes[:50])


def get_celltype_col(df):
    """
    Find cell-type column in a molecule table.
    """
    candidates = [
        "Assigned_Xenium_Cell_Type",
        "cell_type",
        "ct_label",
        "celltype",
        "Cell_Type",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError(
        "Could not find a cell-type column. Expected one of: "
        f"{candidates}. Existing columns: {list(df.columns)}"
    )


def numeric_col_or_nan(df, col):
    """
    Return numeric column if present, otherwise all NaN.
    """
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").astype(np.float32)
    return pd.Series(np.nan, index=df.index, dtype=np.float32)


def nuclear_indicator(df):
    """
    Use p_nuclear if present; otherwise overlaps_nucleus.
    """
    if "p_nuclear" in df.columns:
        return pd.to_numeric(df["p_nuclear"], errors="coerce").fillna(0).astype(np.float32)

    if "overlaps_nucleus" in df.columns:
        return pd.to_numeric(df["overlaps_nucleus"], errors="coerce").fillna(0).astype(np.float32)

    return pd.Series(np.nan, index=df.index, dtype=np.float32)


# ------------------------------------------------------------------------------
# 3. Aggregation functions
# ------------------------------------------------------------------------------

def aggregate_localization(df, method, source_type, genes=None):
    """
    Aggregate localization statistics per gene and cell type.

    source_type examples:
      raw_observed
      learned_imputed
      baseline_imputed
    """
    if genes is not None:
        d = df[df["gene_id"].astype(str).isin(genes)].copy()
    else:
        d = df.copy()

    ct_col = get_celltype_col(d)

    temp = pd.DataFrame({
        "gene_id": d["gene_id"].astype(str).values,
        "cell_type": d[ct_col].astype(str).values,
        "r_norm": numeric_col_or_nan(d, "r_norm").values,
        "z_rel": numeric_col_or_nan(d, "z_rel").values,
        "p_nuclear": nuclear_indicator(d).values,
    })

    temp["central"] = temp["r_norm"] <= 0.35
    temp["peripheral"] = temp["r_norm"] >= 0.75

    out = (
        temp
        .groupby(["gene_id", "cell_type"], observed=True)
        .agg(
            n_molecules=("gene_id", "size"),

            mean_r_norm=("r_norm", "mean"),
            median_r_norm=("r_norm", "median"),
            std_r_norm=("r_norm", "std"),

            mean_z_rel=("z_rel", "mean"),
            median_z_rel=("z_rel", "median"),
            std_z_rel=("z_rel", "std"),

            nuclear_fraction=("p_nuclear", "mean"),
            central_fraction_r_le_0_35=("central", "mean"),
            peripheral_fraction_r_ge_0_75=("peripheral", "mean"),
        )
        .reset_index()
    )

    out["method"] = method
    out["source_type"] = source_type

    return out


def combine_raw_and_imputed_localization(raw_agg, imp_agg, completed_method):
    """
    Combine raw and imputed aggregate tables into completed aggregate table
    using weighted means.

    This avoids concatenating huge raw + imputed molecule tables.
    """
    keys = ["gene_id", "cell_type"]

    raw = raw_agg.copy()
    imp = imp_agg.copy()

    raw = raw.rename(columns={c: f"raw_{c}" for c in raw.columns if c not in keys})
    imp = imp.rename(columns={c: f"imp_{c}" for c in imp.columns if c not in keys})

    merged = raw.merge(imp, on=keys, how="outer")

    count_cols = ["raw_n_molecules", "imp_n_molecules"]
    for c in count_cols:
        if c not in merged.columns:
            merged[c] = 0
        merged[c] = merged[c].fillna(0).astype(float)

    raw_n = merged["raw_n_molecules"].to_numpy(dtype=float)
    imp_n = merged["imp_n_molecules"].to_numpy(dtype=float)
    total_n = raw_n + imp_n

    out = merged[keys].copy()
    out["n_molecules"] = total_n.astype(int)
    out["n_raw_molecules"] = raw_n.astype(int)
    out["n_imputed_molecules"] = imp_n.astype(int)
    out["imputed_fraction"] = np.divide(
        imp_n,
        np.maximum(total_n, 1),
    )

    weighted_metrics = [
        "mean_r_norm",
        "mean_z_rel",
        "nuclear_fraction",
        "central_fraction_r_le_0_35",
        "peripheral_fraction_r_ge_0_75",
    ]

    for m in weighted_metrics:
        raw_col = f"raw_{m}"
        imp_col = f"imp_{m}"

        raw_vals = merged[raw_col].to_numpy(dtype=float) if raw_col in merged.columns else np.full(len(merged), np.nan)
        imp_vals = merged[imp_col].to_numpy(dtype=float) if imp_col in merged.columns else np.full(len(merged), np.nan)

        raw_vals = np.nan_to_num(raw_vals, nan=0.0)
        imp_vals = np.nan_to_num(imp_vals, nan=0.0)

        out[m] = np.divide(
            raw_vals * raw_n + imp_vals * imp_n,
            np.maximum(total_n, 1),
        )

    # Median/std cannot be combined exactly from aggregate summaries.
    out["median_r_norm"] = np.nan
    out["std_r_norm"] = np.nan
    out["median_z_rel"] = np.nan
    out["std_z_rel"] = np.nan

    out["method"] = completed_method
    out["source_type"] = "completed"

    return out


# ------------------------------------------------------------------------------
# 4. Build imputed-only and completed summaries
# ------------------------------------------------------------------------------

print("\nAggregating raw and imputed localization summaries...")

raw_obs = mol_observed[mol_observed["status"].astype(str) == "observed"].copy()

raw_agg = aggregate_localization(
    raw_obs,
    method="Raw observed",
    source_type="raw_observed",
    genes=summary_genes,
)

learned_imp_agg = aggregate_localization(
    learned_imputed_df,
    method="Learned 9E imputed only",
    source_type="learned_imputed",
    genes=summary_genes,
)

gene_emp_imp_agg = aggregate_localization(
    gene_emp_imputed_df,
    method="Gene empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

ct_gene_emp_imp_agg = aggregate_localization(
    ct_gene_emp_imputed_df,
    method="Cell-type gene empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

spatial_knn_imp_agg = aggregate_localization(
    spatial_knn_emp_imputed_df,
    method="Spatial-kNN empirical imputed only",
    source_type="baseline_imputed",
    genes=summary_genes,
)

imputed_only_summary_df = pd.concat(
    [
        learned_imp_agg,
        gene_emp_imp_agg,
        ct_gene_emp_imp_agg,
        spatial_knn_imp_agg,
    ],
    ignore_index=True,
)

learned_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    learned_imp_agg,
    completed_method="Learned 9E completed",
)

gene_emp_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    gene_emp_imp_agg,
    completed_method="Gene empirical completed",
)

ct_gene_emp_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    ct_gene_emp_imp_agg,
    completed_method="Cell-type gene empirical completed",
)

spatial_knn_completed_agg = combine_raw_and_imputed_localization(
    raw_agg,
    spatial_knn_imp_agg,
    completed_method="Spatial-kNN empirical completed",
)

completed_summary_df = pd.concat(
    [
        raw_agg,
        learned_completed_agg,
        gene_emp_completed_agg,
        ct_gene_emp_completed_agg,
        spatial_knn_completed_agg,
    ],
    ignore_index=True,
)

print(f"Raw summary rows          : {len(raw_agg):,}")
print(f"Imputed-only summary rows : {len(imputed_only_summary_df):,}")
print(f"Completed summary rows    : {len(completed_summary_df):,}")

print("\nPreview completed summary:")
display(completed_summary_df.head(20))

# ------------------------------------------------------------------------------
# 5. Learned 9E vs empirical baselines deltas
# ------------------------------------------------------------------------------

print("\nComputing learned 9E vs baseline localization deltas...")

keys = ["gene_id", "cell_type"]

learned_comp = learned_completed_agg.copy()
learned_comp = learned_comp.rename(
    columns={c: f"learned_{c}" for c in learned_comp.columns if c not in keys}
)

delta_rows = []

baseline_completed_tables = {
    "Gene empirical completed": gene_emp_completed_agg,
    "Cell-type gene empirical completed": ct_gene_emp_completed_agg,
    "Spatial-kNN empirical completed": spatial_knn_completed_agg,
}

metrics_for_delta = [
    "mean_r_norm",
    "mean_z_rel",
    "nuclear_fraction",
    "central_fraction_r_le_0_35",
    "peripheral_fraction_r_ge_0_75",
    "imputed_fraction",
]

for baseline_name, base_df in baseline_completed_tables.items():
    base = base_df.copy()
    base = base.rename(columns={c: f"baseline_{c}" for c in base.columns if c not in keys})

    merged = learned_comp.merge(base, on=keys, how="inner")
    merged["baseline_method"] = baseline_name

    for m in metrics_for_delta:
        lc = f"learned_{m}"
        bc = f"baseline_{m}"

        if lc in merged.columns and bc in merged.columns:
            merged[f"delta_{m}_learned_minus_baseline"] = merged[lc] - merged[bc]

    delta_rows.append(merged)

loc_delta_df = pd.concat(delta_rows, ignore_index=True)

print(f"loc_delta_df shape: {loc_delta_df.shape}")
display(loc_delta_df.head(20))

# ------------------------------------------------------------------------------
# 6. Method-level summary
# ------------------------------------------------------------------------------

method_summary_df = (
    completed_summary_df
    .groupby("method", observed=True)
    .agg(
        n_gene_celltype_pairs=("gene_id", "size"),
        total_molecules=("n_molecules", "sum"),
        mean_r_norm=("mean_r_norm", "mean"),
        mean_z_rel=("mean_z_rel", "mean"),
        mean_nuclear_fraction=("nuclear_fraction", "mean"),
        mean_central_fraction=("central_fraction_r_le_0_35", "mean"),
        mean_peripheral_fraction=("peripheral_fraction_r_ge_0_75", "mean"),
    )
    .reset_index()
)

print("\nMethod-level localization summary:")
display(method_summary_df)

# ------------------------------------------------------------------------------
# 7. Heatmaps for learned 9E completed
# ------------------------------------------------------------------------------

# To avoid overly huge heatmaps, keep gene/cell-type pairs with enough molecules.
MIN_MOLECULES_HEATMAP = 20

learned_heat = learned_completed_agg[
    learned_completed_agg["n_molecules"] >= MIN_MOLECULES_HEATMAP
].copy()

# Keep selected marker genes only.
learned_heat = learned_heat[learned_heat["gene_id"].isin(summary_genes)].copy()

# Order genes by marker list, cell types alphabetically.
gene_order = [g for g in summary_genes if g in learned_heat["gene_id"].unique()]
celltype_order = sorted(learned_heat["cell_type"].unique().tolist())

def plot_gene_celltype_heatmap(df, value_col, title, cbar_label, out_path, vmin=None, vmax=None, cmap="viridis"):
    matrix = df.pivot_table(
        index="gene_id",
        columns="cell_type",
        values=value_col,
        aggfunc="mean",
    )

    matrix = matrix.reindex(index=gene_order, columns=celltype_order)

    # Drop all-empty rows/cols.
    matrix = matrix.dropna(axis=0, how="all").dropna(axis=1, how="all")

    fig_w = max(12, 0.65 * len(matrix.columns) + 4)
    fig_h = max(7, 0.35 * len(matrix.index) + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    arr = matrix.values.astype(float)

    im = ax.imshow(
        arr,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )

    ax.set_title(title, fontsize=14, pad=14)
    ax.set_xlabel("Cell type")
    ax.set_ylabel("Gene")

    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right", fontsize=8)

    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index, fontsize=8)

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label(cbar_label, fontsize=11)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap: {out_path}")

    return matrix

if len(learned_heat) > 0:
    nuc_matrix = plot_gene_celltype_heatmap(
        learned_heat,
        value_col="nuclear_fraction",
        title="Learned 9E completed: nuclear fraction by gene and cell type",
        cbar_label="Nuclear fraction",
        out_path=LOC_NUC_HEATMAP_PATH,
        vmin=0,
        vmax=1,
        cmap="viridis",
    )

    r_matrix = plot_gene_celltype_heatmap(
        learned_heat,
        value_col="mean_r_norm",
        title="Learned 9E completed: mean radial position by gene and cell type",
        cbar_label="Mean r_norm",
        out_path=LOC_R_HEATMAP_PATH,
        vmin=0,
        vmax=1,
        cmap="viridis",
    )
else:
    print("WARNING: no gene/cell-type pairs passed MIN_MOLECULES_HEATMAP.")

# ------------------------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------------------------

imputed_only_summary_df.to_csv(LOC_IMPUTED_ONLY_PATH, index=False)
completed_summary_df.to_csv(LOC_COMPLETED_PATH, index=False)
loc_delta_df.to_csv(LOC_DELTA_PATH, index=False)
method_summary_df.to_csv(LOC_METHOD_SUMMARY_PATH, index=False)

print("\nSaved per-gene/per-cell-type localization summary outputs:")
print(f"  Imputed-only summary : {LOC_IMPUTED_ONLY_PATH}")
print(f"  Completed summary    : {LOC_COMPLETED_PATH}")
print(f"  Learned-vs-baselines : {LOC_DELTA_PATH}")
print(f"  Method summary       : {LOC_METHOD_SUMMARY_PATH}")
print(f"  Nuclear heatmap      : {LOC_NUC_HEATMAP_PATH}")
print(f"  r_norm heatmap       : {LOC_R_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Known nuclear/TF genes have higher nuclear_fraction in expected cell types.")
print("  Good sign 2: Stromal/ECM or membrane-associated genes do not collapse fully to the nucleus.")
print("  Good sign 3: Learned 9E differs from baselines in biologically plausible ways, not randomly.")
print("  Caution: this is descriptive; use it with nuclear/radial statistical summaries.")

print("\n" + "=" * 100)
print("CELL 9 COMPLETE — Per-gene / per-cell-type localization summaries")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 10 — Spatial domain / tissue-region signal strengthening
#
# Biological question:
#   Does completed 9E count data make tissue-region / spatial biological
#   patterns clearer than raw observed counts?
#
# Main comparison:
#   Raw observed counts
#   Learned 9E completed counts
#
# Optional comparison:
#   Step4 denoised counts, if X_denoised is available
#
# Main metrics:
#   - Moran's I on cell-coordinate kNN graph
#   - neighbor smoothness / neighbor correlation
#   - expected-domain contrast for marker modules
#   - AUROC of module score for expected tissue/cell-type domain
#
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

print("=" * 100)
print("CELL 10 — Spatial domain / tissue-region signal strengthening")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "denoised_adata",
    "X_raw_counts",
    "X_completed_9E",
    "shared_genes",
    "cell_type_labels",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from reload/downstream cells: {v}")

# ------------------------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------------------------

SPATIAL_DOMAIN_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_spatial_domain_signal_strengthening"
)
os.makedirs(SPATIAL_DOMAIN_DIR, exist_ok=True)

SPATIAL_MODULE_METRICS_PATH = os.path.join(
    SPATIAL_DOMAIN_DIR,
    f"{RUN_NAME}_spatial_domain_module_metrics.csv"
)

SPATIAL_MODULE_GAIN_PATH = os.path.join(
    SPATIAL_DOMAIN_DIR,
    f"{RUN_NAME}_spatial_domain_module_gain_vs_raw.csv"
)

SPATIAL_MODULE_HEATMAP_PATH = os.path.join(
    SPATIAL_DOMAIN_DIR,
    f"{RUN_NAME}_spatial_domain_metric_gain_heatmap.png"
)

SPATIAL_MODULE_MAP_DIR = os.path.join(
    SPATIAL_DOMAIN_DIR,
    "module_spatial_maps"
)
os.makedirs(SPATIAL_MODULE_MAP_DIR, exist_ok=True)

print(f"SPATIAL_DOMAIN_DIR: {SPATIAL_DOMAIN_DIR}")

# ------------------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42
SAMPLE_SIZE = 50_000

# kNN graph for spatial autocorrelation.
SPATIAL_K = 12

# Plotting sample. Use same cells as metrics if possible.
MAX_MAP_POINTS = 50_000

print(f"SAMPLE_SIZE: {SAMPLE_SIZE:,}")
print(f"SPATIAL_K: {SPATIAL_K}")

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_xy_from_adata(adata):
    """
    Get cell x/y coordinates from AnnData.obs.
    """
    x_candidates = ["x_centroid", "x", "center_x", "cell_x"]
    y_candidates = ["y_centroid", "y", "center_y", "cell_y"]

    x_col = None
    y_col = None

    for c in x_candidates:
        if c in adata.obs.columns:
            x_col = c
            break

    for c in y_candidates:
        if c in adata.obs.columns:
            y_col = c
            break

    if x_col is None or y_col is None:
        raise KeyError(
            "Could not find x/y centroid columns in denoised_adata.obs. "
            f"obs columns: {list(adata.obs.columns)}"
        )

    xy = adata.obs[[x_col, y_col]].copy()
    xy.columns = ["x", "y"]

    xy["x"] = pd.to_numeric(xy["x"], errors="coerce")
    xy["y"] = pd.to_numeric(xy["y"], errors="coerce")

    return xy.to_numpy(dtype=np.float32), x_col, y_col


def log_norm_counts(X):
    """
    Library-size normalize counts and log1p transform.
    """
    X = ensure_dense(X).astype(np.float32)

    lib = X.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0)

    X_norm = X / lib * 1e4
    X_log = np.log1p(X_norm).astype(np.float32)

    return X_log


def safe_auroc(y_true, scores):
    try:
        y_true = np.asarray(y_true).astype(int)
        scores = np.asarray(scores).astype(float)

        if len(np.unique(y_true)) < 2:
            return np.nan

        return float(roc_auc_score(y_true, scores))
    except Exception:
        return np.nan


def build_spatial_knn_edges(xy, k=12):
    """
    Build directed kNN edges from spatial coordinates.
    Returns source indices and neighbor indices.
    """
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="ball_tree")
    nbrs.fit(xy)

    distances, indices = nbrs.kneighbors(xy)

    # Drop self neighbor at column 0.
    neigh = indices[:, 1:]
    src = np.repeat(np.arange(xy.shape[0]), k)
    dst = neigh.reshape(-1)

    return src.astype(np.int64), dst.astype(np.int64)


def morans_i_knn(values, src, dst):
    """
    Approximate Moran's I using unweighted directed kNN edges.

    Higher positive value means nearby cells have more similar module scores.
    """
    x = np.asarray(values, dtype=np.float64)
    x = x - np.nanmean(x)

    denom = np.nansum(x ** 2)

    if denom <= 1e-12:
        return np.nan

    w = len(src)
    n = len(x)

    num = np.nansum(x[src] * x[dst])

    return float((n / w) * (num / denom))


def neighbor_correlation(values, src, dst):
    """
    Pearson correlation between each cell's score and its neighbor's score.
    """
    a = np.asarray(values[src], dtype=np.float64)
    b = np.asarray(values[dst], dtype=np.float64)

    if np.nanstd(a) <= 1e-12 or np.nanstd(b) <= 1e-12:
        return np.nan

    return float(np.corrcoef(a, b)[0, 1])


def neighbor_smoothness(values, src, dst):
    """
    Mean absolute score difference between neighboring cells.
    Lower is smoother.
    """
    v = np.asarray(values, dtype=np.float64)
    return float(np.nanmean(np.abs(v[src] - v[dst])))


def module_score(X_log, genes, gene_to_col):
    """
    Average log-normalized expression over available module genes.
    """
    cols = [gene_to_col[g] for g in genes if g in gene_to_col]

    if len(cols) == 0:
        return None, []

    score = X_log[:, cols].mean(axis=1)
    return np.asarray(score, dtype=np.float32), [g for g in genes if g in gene_to_col]


def expected_domain_mask(cell_types, expected_celltypes):
    """
    Boolean mask for expected cell types.
    Allows partial matching if exact cell type is not found.
    """
    ct = pd.Series(np.asarray(cell_types).astype(str))

    mask = np.zeros(len(ct), dtype=bool)

    for expected in expected_celltypes:
        exact = (ct == expected).to_numpy()

        if exact.sum() > 0:
            mask |= exact
        else:
            # Fallback partial matching.
            mask |= ct.str.contains(expected, case=False, regex=False).to_numpy()

    return mask


# ------------------------------------------------------------------------------
# 4. Define biological modules and expected domains
# ------------------------------------------------------------------------------

available_genes = set(str(g) for g in shared_genes)
gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}

module_defs = {
    "Tumor_Epithelial": {
        "genes": ["EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "ERBB2", "GATA3", "FOXA1", "CCND1"],
        "expected_celltypes": ["DCIS_1", "DCIS_2", "Invasive_Tumor", "Prolif_Invasive_Tumor"],
    },
    "Stromal_ECM": {
        "genes": ["LUM", "POSTN", "CXCL12", "FBLN1", "DCN", "COL1A1", "COL1A2"],
        "expected_celltypes": ["Stromal"],
    },
    "Endothelial_Vascular": {
        "genes": ["PECAM1", "VWF", "CAV1", "NOSTRIN", "CLDN5", "KDR"],
        "expected_celltypes": ["Endothelial"],
    },
    "T_Cell": {
        "genes": ["CD3D", "CD3E", "CD8A", "CD8B", "IL7R", "LTB", "PTPRC"],
        "expected_celltypes": ["CD4+_T_Cells", "CD8+_T_Cells", "T_Cells"],
    },
    "B_Cell": {
        "genes": ["MS4A1", "CD79A", "CD79B", "BANK1", "TNFRSF17", "PTPRC"],
        "expected_celltypes": ["B_Cells"],
    },
    "Macrophage_Myeloid": {
        "genes": ["CD68", "CD163", "TYROBP", "LYZ", "LST1", "MNDA", "PTPRC"],
        "expected_celltypes": ["Macrophages_1", "Macrophages_2", "Macrophages"],
    },
    "Proliferation": {
        "genes": ["MKI67", "TOP2A", "CCND1"],
        "expected_celltypes": ["Prolif_Invasive_Tumor"],
    },
}

# Keep only modules with at least 2 genes present.
filtered_modules = {}

for module_name, spec in module_defs.items():
    present = [g for g in spec["genes"] if g in available_genes]

    if len(present) >= 2:
        filtered_modules[module_name] = {
            "genes": present,
            "expected_celltypes": spec["expected_celltypes"],
        }

print("\nModules with available genes:")
for m, spec in filtered_modules.items():
    print(f"  {m:25s}: {spec['genes']}")

if len(filtered_modules) == 0:
    raise RuntimeError("No modules have at least 2 available genes.")

# ------------------------------------------------------------------------------
# 5. Sample cells and prepare spatial graph
# ------------------------------------------------------------------------------

print("\nPreparing cell coordinates and sample...")

xy_all, x_col, y_col = get_xy_from_adata(denoised_adata)

valid_xy = np.isfinite(xy_all).all(axis=1)
all_indices = np.where(valid_xy)[0]

rng = np.random.default_rng(RANDOM_STATE)

if len(all_indices) > SAMPLE_SIZE:
    sample_idx = rng.choice(all_indices, size=SAMPLE_SIZE, replace=False)
else:
    sample_idx = all_indices.copy()

sample_idx = np.sort(sample_idx)

xy_sample = xy_all[sample_idx, :]
cell_types_sample = np.asarray(cell_type_labels).astype(str)[sample_idx]

print(f"Coordinate columns: {x_col}, {y_col}")
print(f"Valid coordinate cells: {len(all_indices):,}")
print(f"Sample cells used: {len(sample_idx):,}")

print("\nBuilding spatial kNN graph...")
src, dst = build_spatial_knn_edges(xy_sample, k=SPATIAL_K)
print(f"kNN edges: {len(src):,}")

# ------------------------------------------------------------------------------
# 6. Build datasets
# ------------------------------------------------------------------------------

datasets = {
    "Raw observed counts": X_raw_counts,
    "Learned 9E completed counts": X_completed_9E,
}

if "X_denoised" in globals():
    datasets["Step4 denoised counts"] = X_denoised

print("\nDatasets to evaluate:")
for name, X in datasets.items():
    print(f"  {name:30s}: {ensure_dense(X).shape}")

# ------------------------------------------------------------------------------
# 7. Compute module spatial-domain metrics
# ------------------------------------------------------------------------------

print("\nComputing spatial-domain metrics...")

metric_rows = []
module_score_cache = {}

for dataset_name, X_counts in datasets.items():
    print(f"\nProcessing dataset: {dataset_name}")

    X_sub = ensure_dense(X_counts)[sample_idx, :]
    X_log = log_norm_counts(X_sub)

    for module_name, spec in filtered_modules.items():
        score, genes_used = module_score(X_log, spec["genes"], gene_to_col)

        if score is None:
            continue

        domain_mask = expected_domain_mask(cell_types_sample, spec["expected_celltypes"])

        in_domain = score[domain_mask]
        out_domain = score[~domain_mask]

        domain_mean = float(np.mean(in_domain)) if len(in_domain) else np.nan
        background_mean = float(np.mean(out_domain)) if len(out_domain) else np.nan
        domain_contrast = domain_mean - background_mean

        domain_ratio = (domain_mean + 1e-6) / (background_mean + 1e-6)

        y_true = domain_mask.astype(int)
        auc = safe_auroc(y_true, score)

        moran = morans_i_knn(score, src, dst)
        neigh_corr = neighbor_correlation(score, src, dst)
        neigh_smooth = neighbor_smoothness(score, src, dst)

        metric_rows.append({
            "dataset": dataset_name,
            "module": module_name,
            "genes_used": ",".join(genes_used),
            "n_genes_used": int(len(genes_used)),
            "expected_celltypes": ",".join(spec["expected_celltypes"]),

            "n_domain_cells": int(domain_mask.sum()),
            "n_background_cells": int((~domain_mask).sum()),

            "mean_score_domain": domain_mean,
            "mean_score_background": background_mean,
            "domain_contrast_domain_minus_background": float(domain_contrast),
            "domain_ratio_domain_over_background": float(domain_ratio),
            "AUROC_domain_vs_background": auc,

            "morans_I_spatial_autocorrelation": moran,
            "neighbor_correlation": neigh_corr,
            "neighbor_smoothness_absdiff": neigh_smooth,
        })

        module_score_cache[(dataset_name, module_name)] = score.astype(np.float32)

    del X_sub, X_log
    gc.collect()

spatial_module_metrics_df = pd.DataFrame(metric_rows)

print("\nSpatial module metrics:")
display(spatial_module_metrics_df)

# ------------------------------------------------------------------------------
# 8. Raw vs completed gain table
# ------------------------------------------------------------------------------

print("\nComputing raw-vs-completed spatial-domain gains...")

raw_df = spatial_module_metrics_df[
    spatial_module_metrics_df["dataset"] == "Raw observed counts"
].copy()

completed_df = spatial_module_metrics_df[
    spatial_module_metrics_df["dataset"] == "Learned 9E completed counts"
].copy()

merge_keys = ["module"]

raw_renamed = raw_df.rename(columns={
    c: f"raw_{c}" for c in raw_df.columns if c not in merge_keys
})

comp_renamed = completed_df.rename(columns={
    c: f"completed_{c}" for c in completed_df.columns if c not in merge_keys
})

gain_df = raw_renamed.merge(
    comp_renamed,
    on=merge_keys,
    how="inner",
)

gain_metrics = [
    "domain_contrast_domain_minus_background",
    "domain_ratio_domain_over_background",
    "AUROC_domain_vs_background",
    "morans_I_spatial_autocorrelation",
    "neighbor_correlation",
    "neighbor_smoothness_absdiff",
]

for m in gain_metrics:
    raw_col = f"raw_{m}"
    comp_col = f"completed_{m}"

    if raw_col in gain_df.columns and comp_col in gain_df.columns:
        gain_df[f"delta_{m}_completed_minus_raw"] = gain_df[comp_col] - gain_df[raw_col]

# For neighbor smoothness, lower is better, so define improvement as raw - completed.
if (
    "raw_neighbor_smoothness_absdiff" in gain_df.columns
    and "completed_neighbor_smoothness_absdiff" in gain_df.columns
):
    gain_df["improvement_neighbor_smoothness_raw_minus_completed"] = (
        gain_df["raw_neighbor_smoothness_absdiff"]
        - gain_df["completed_neighbor_smoothness_absdiff"]
    )

print("\nRaw vs learned 9E completed gain table:")
display(gain_df)

# ------------------------------------------------------------------------------
# 9. Heatmap of gains
# ------------------------------------------------------------------------------

heatmap_metrics = [
    "delta_domain_contrast_domain_minus_background_completed_minus_raw",
    "delta_AUROC_domain_vs_background_completed_minus_raw",
    "delta_morans_I_spatial_autocorrelation_completed_minus_raw",
    "delta_neighbor_correlation_completed_minus_raw",
    "improvement_neighbor_smoothness_raw_minus_completed",
]

heatmap_metrics = [m for m in heatmap_metrics if m in gain_df.columns]

heat_df = gain_df.set_index("module")[heatmap_metrics].copy()

# Rename columns for readability.
rename_map = {
    "delta_domain_contrast_domain_minus_background_completed_minus_raw": "Δ domain contrast",
    "delta_AUROC_domain_vs_background_completed_minus_raw": "Δ AUROC",
    "delta_morans_I_spatial_autocorrelation_completed_minus_raw": "Δ Moran's I",
    "delta_neighbor_correlation_completed_minus_raw": "Δ neighbor corr",
    "improvement_neighbor_smoothness_raw_minus_completed": "Smoothness improvement",
}
heat_df = heat_df.rename(columns=rename_map)

fig_w = max(10, 1.7 * len(heat_df.columns) + 3)
fig_h = max(5, 0.6 * len(heat_df.index) + 2)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

arr = heat_df.values.astype(float)
finite = arr[np.isfinite(arr)]

vmax = np.nanpercentile(np.abs(finite), 95) if len(finite) else 1.0
vmax = max(float(vmax), 0.01)

im = ax.imshow(
    arr,
    aspect="auto",
    vmin=-vmax,
    vmax=vmax,
    cmap="coolwarm",
)

ax.set_title("Spatial-domain signal gain: Learned 9E completed − Raw observed", fontsize=14, pad=14)
ax.set_xlabel("Metric gain")
ax.set_ylabel("Biological module")

ax.set_xticks(np.arange(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, rotation=45, ha="right", fontsize=9)

ax.set_yticks(np.arange(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=10)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center", fontsize=8, color="black")

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Gain vs raw", fontsize=11)

plt.tight_layout()
plt.savefig(SPATIAL_MODULE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved spatial-domain gain heatmap:")
print(f"  {SPATIAL_MODULE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 10. Spatial maps for selected modules
# ------------------------------------------------------------------------------

print("\nGenerating spatial maps for module scores...")

# Use a readable number of points for plotting.
if len(sample_idx) > MAX_MAP_POINTS:
    map_local_idx = rng.choice(np.arange(len(sample_idx)), size=MAX_MAP_POINTS, replace=False)
else:
    map_local_idx = np.arange(len(sample_idx))

map_x = xy_sample[map_local_idx, 0]
map_y = xy_sample[map_local_idx, 1]

map_records = []

modules_to_plot = list(filtered_modules.keys())

for module_name in modules_to_plot:
    raw_key = ("Raw observed counts", module_name)
    comp_key = ("Learned 9E completed counts", module_name)

    if raw_key not in module_score_cache or comp_key not in module_score_cache:
        continue

    raw_score = module_score_cache[raw_key][map_local_idx]
    comp_score = module_score_cache[comp_key][map_local_idx]
    delta_score = comp_score - raw_score

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

    panels = [
        ("Raw observed", raw_score, "viridis"),
        ("Learned 9E completed", comp_score, "viridis"),
        ("Completed − Raw", delta_score, "coolwarm"),
    ]

    for ax, (title, vals, cmap) in zip(axes, panels):
        if title == "Completed − Raw":
            finite_vals = vals[np.isfinite(vals)]
            vmax_delta = np.nanpercentile(np.abs(finite_vals), 95) if len(finite_vals) else 1.0
            vmax_delta = max(float(vmax_delta), 0.01)

            sca = ax.scatter(
                map_x,
                map_y,
                c=vals,
                s=2,
                cmap=cmap,
                vmin=-vmax_delta,
                vmax=vmax_delta,
                linewidths=0,
            )
        else:
            sca = ax.scatter(
                map_x,
                map_y,
                c=vals,
                s=2,
                cmap=cmap,
                linewidths=0,
            )

        ax.set_title(title, fontsize=12)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.grid(False)

        cbar = plt.colorbar(sca, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("Module score", fontsize=9)

    fig.suptitle(f"Spatial module map: {module_name}", fontsize=15, y=1.02)
    plt.tight_layout()

    out_path = os.path.join(
        SPATIAL_MODULE_MAP_DIR,
        f"{RUN_NAME}_spatial_module_map_{module_name}.png"
    )

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

    map_records.append({
        "module": module_name,
        "map_path": out_path,
    })

    print(f"Saved module map for {module_name}: {out_path}")

module_map_manifest_df = pd.DataFrame(map_records)

SPATIAL_MODULE_MAP_MANIFEST_PATH = os.path.join(
    SPATIAL_DOMAIN_DIR,
    f"{RUN_NAME}_spatial_module_map_manifest.csv"
)
module_map_manifest_df.to_csv(SPATIAL_MODULE_MAP_MANIFEST_PATH, index=False)

# ------------------------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------------------------

spatial_module_metrics_df.to_csv(SPATIAL_MODULE_METRICS_PATH, index=False)
gain_df.to_csv(SPATIAL_MODULE_GAIN_PATH, index=False)

print("\nSaved spatial-domain validation outputs:")
print(f"  Module metrics      : {SPATIAL_MODULE_METRICS_PATH}")
print(f"  Gain vs raw         : {SPATIAL_MODULE_GAIN_PATH}")
print(f"  Gain heatmap        : {SPATIAL_MODULE_HEATMAP_PATH}")
print(f"  Module map manifest : {SPATIAL_MODULE_MAP_MANIFEST_PATH}")
print(f"  Module map dir      : {SPATIAL_MODULE_MAP_DIR}")

print("\nInterpretation guide:")
print("  Good sign 1: completed data has higher domain contrast than raw.")
print("  Good sign 2: completed data has higher AUROC for expected tissue/cell-type domains.")
print("  Good sign 3: completed data has higher Moran's I or neighbor correlation.")
print("  Good sign 4: completed data has lower neighbor smoothness absdiff.")
print("  Caution: too much smoothing can inflate spatial autocorrelation; check AUROC and module maps together.")

print("\n" + "=" * 100)
print("CELL 10 COMPLETE — Spatial domain / tissue-region signal strengthening")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 11_OFFICIAL_SPRAWL — Run official SPRAWL directly
#
# Goal:
#   Compute official SPRAWL scores:
#       1. peripheral
#       2. central
#       3. punctate
#       4. radial
#
# Main comparison:
#   Raw observed vs Learned 9E completed
#
# Optional:
#   Also run empirical baseline completed molecule tables.
#
# Important:
#   Official SPRAWL is computationally expensive, especially punctate/radial.
#   Start with SAMPLE_CELLS = 500 or 1000 first.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

print("=" * 100)
print("CELL 11_OFFICIAL_SPRAWL — Run official SPRAWL directly")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Install/import official SPRAWL
# ------------------------------------------------------------------------------

def pip_install(cmd):
    print(f"Running: {cmd}")
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())

try:
    import sprawl
    from sprawl import hdf5, scoring
    SPRAWL_AVAILABLE = True
    print("Official SPRAWL imported successfully.")

except Exception as e1:
    print(f"Initial SPRAWL import failed: {e1}")
    print("Trying PyPI package: subcellular-sprawl")

    try:
        pip_install("install -q subcellular-sprawl")
        import sprawl
        from sprawl import hdf5, scoring
        SPRAWL_AVAILABLE = True
        print("Official SPRAWL imported successfully from PyPI.")

    except Exception as e2:
        print(f"PyPI install/import failed: {e2}")
        print("Trying GitHub install from salzman-lab/SPRAWL package subdirectory.")

        try:
            pip_install("install -q git+https://github.com/salzman-lab/SPRAWL.git#subdirectory=package")
            import sprawl
            from sprawl import hdf5, scoring
            SPRAWL_AVAILABLE = True
            print("Official SPRAWL imported successfully from GitHub.")

        except Exception as e3:
            SPRAWL_AVAILABLE = False
            raise ImportError(
                "Could not install/import official SPRAWL. "
                "Try restarting the runtime and running this cell again.\n\n"
                f"Initial error: {e1}\n"
                f"PyPI error: {e2}\n"
                f"GitHub error: {e3}"
            )

print("\nSPRAWL scoring metrics available:")
print(list(scoring.available_metrics.keys()) if hasattr(scoring, "available_metrics") else "Could not inspect scoring.available_metrics")

# ------------------------------------------------------------------------------
# 1. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "mol_observed",
    "learned_imputed_df",
    "cell_polygons",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Missing required variable from reload/downstream cells: {v}")

# ------------------------------------------------------------------------------
# 2. Output paths
# ------------------------------------------------------------------------------

OFFICIAL_SPRAWL_DIR = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_official_sprawl_validation"
)
os.makedirs(OFFICIAL_SPRAWL_DIR, exist_ok=True)

SPRAWL_H5_DIR = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    "sprawl_h5_inputs"
)
os.makedirs(SPRAWL_H5_DIR, exist_ok=True)

SPRAWL_CELL_SCORE_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_cell_gene_scores.csv"
)

SPRAWL_GENE_CELLTYPE_SCORE_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_gene_celltype_scores.csv"
)

SPRAWL_METHOD_SUMMARY_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_method_summary.csv"
)

SPRAWL_GAIN_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_raw_vs_learned_gain.csv"
)

SPRAWL_FIG_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_summary.png"
)

print(f"OFFICIAL_SPRAWL_DIR: {OFFICIAL_SPRAWL_DIR}")

# ------------------------------------------------------------------------------
# 3. Configuration
# ------------------------------------------------------------------------------

RANDOM_STATE = 42

# Start small. Official SPRAWL radial/punctate are expensive.
SAMPLE_CELLS = 750

# Minimum total molecules in selected cells.
MIN_TOTAL_MOLS_PER_CELL = 10

# Minimum molecules per gene per cell.
# SPRAWL radial/punctate internally remove genes with only 1 molecule.
# But using >=3 is more stable.
MIN_GENE_MOLS_FOR_INPUT = 3

# Official SPRAWL metrics to run.
SPRAWL_METRICS = ["peripheral", "central", "punctate", "radial"]

# For expensive permutation metrics.
# Increase to 1000 for final run if runtime is acceptable.
SPRAWL_NUM_ITERATIONS = 200
SPRAWL_NUM_PAIRS = 4

# Multiprocessing.
SPRAWL_PROCESSES = 2

# Run baselines too?
# Start False. Set True only after raw/learned works.
RUN_BASELINES_TOO = False

# Optional: restrict to marker genes only for a faster biological validation.
RESTRICT_TO_MARKER_GENES = True

candidate_marker_genes = [
    "ESR1", "PGR", "AR", "GATA3", "FOXA1", "MKI67", "TOP2A", "CCND1", "IRF7",
    "ERBB2", "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19",
    "LUM", "POSTN", "CXCL12", "FBLN1", "DCN", "COL1A1", "COL1A2",
    "CD3D", "CD3E", "CD8A", "MS4A1", "CD79A", "PTPRC",
    "CD68", "CD163", "TYROBP", "LYZ", "LST1",
    "PECAM1", "VWF", "CAV1", "NOSTRIN", "CLDN5", "KDR",
]

if "shared_genes" in globals():
    available_genes = set(str(g) for g in shared_genes)
    marker_genes = [g for g in candidate_marker_genes if g in available_genes]
else:
    marker_genes = candidate_marker_genes

print(f"SAMPLE_CELLS: {SAMPLE_CELLS:,}")
print(f"SPRAWL_METRICS: {SPRAWL_METRICS}")
print(f"SPRAWL_NUM_ITERATIONS: {SPRAWL_NUM_ITERATIONS}")
print(f"SPRAWL_NUM_PAIRS: {SPRAWL_NUM_PAIRS}")
print(f"SPRAWL_PROCESSES: {SPRAWL_PROCESSES}")
print(f"RUN_BASELINES_TOO: {RUN_BASELINES_TOO}")
print(f"RESTRICT_TO_MARKER_GENES: {RESTRICT_TO_MARKER_GENES}")
print(f"Marker genes used if restricted: {marker_genes}")

# ------------------------------------------------------------------------------
# 4. Helper class for writing SPRAWL-compatible HDF5
# ------------------------------------------------------------------------------

class SimpleSprawlCell:
    """
    Minimal object compatible with sprawl.hdf5.HDF5.write_cells().

    SPRAWL expects each cell to have:
      - cell_id
      - annotation
      - zslices
      - boundaries[zslice]
      - spot_coords[zslice]
      - spot_genes[zslice]
      - gene_counts
      - genes
      - gene_vars
    """

    def __init__(self, cell_id, annotation, boundary_xy, spot_xy, spot_genes):
        self.cell_id = str(cell_id)
        self.annotation = str(annotation)

        # Collapse Xenium molecules into one 2D z-slice.
        # This is acceptable for 2D SPRAWL-style scoring.
        self.zslices = ["z0"]

        self.boundaries = {
            "z0": np.asarray(boundary_xy, dtype=np.float32)
        }

        self.spot_coords = {
            "z0": np.asarray(spot_xy, dtype=np.float32)
        }

        self.spot_genes = {
            "z0": np.asarray([str(g) for g in spot_genes])
        }

        self.gene_counts = Counter([str(g) for g in spot_genes])
        self.genes = sorted(list(self.gene_counts.keys()))

        # SPRAWL peripheral/central calculate theoretical vars if needed,
        # so this can start empty.
        self.gene_vars = {}

        self.n_per_z = {
            "z0": len(spot_genes)
        }

        self.n = len(spot_genes)


        self.ranked = False
        self.spot_ranks = {"z0": []}
        self.spot_values = {"z0": []}
        self.gene_med_ranks = {}

    def filter_genes_by_count(self, min_gene_spots=1, max_gene_spots=None):
        """
        SPRAWL's radial/punctate metrics may call this method.
        We implement it so this object behaves like sprawl.cell.Cell.
        """
        if max_gene_spots is None:
            max_gene_spots = max(self.gene_counts.values()) if self.gene_counts else 0

        keep_genes = {
            g for g, c in self.gene_counts.items()
            if min_gene_spots <= c <= max_gene_spots
        }

        new_spot_genes = []
        new_spot_coords = []

        for gene, xy in zip(self.spot_genes["z0"], self.spot_coords["z0"]):
            if gene in keep_genes:
                new_spot_genes.append(gene)
                new_spot_coords.append(xy)

        self.spot_genes["z0"] = np.asarray(new_spot_genes)
        self.spot_coords["z0"] = np.asarray(new_spot_coords, dtype=np.float32)

        self.gene_counts = Counter([str(g) for g in self.spot_genes["z0"]])
        self.genes = sorted(list(self.gene_counts.keys()))
        self.gene_vars = {}
        self.n = len(self.spot_genes["z0"])
        self.n_per_z = {"z0": self.n}

        if self.n == 0:
            self.zslices = []
            self.boundaries = {}
            self.spot_coords = {}
            self.spot_genes = {}
            self.n_per_z = {}

        return self


# ------------------------------------------------------------------------------
# 5. Helper functions
# ------------------------------------------------------------------------------

def install_and_import_sprawl():
    """
    Install and import official SPRAWL.

    Package name:
      pip package: subcellular-sprawl
      import name: sprawl
    """
    try:
        import sprawl
        from sprawl import hdf5, scoring
        print("Official SPRAWL imported successfully.")
        return sprawl, hdf5, scoring

    except Exception as e:
        print(f"Initial SPRAWL import failed: {e}")
        print("Installing official SPRAWL package: subcellular-sprawl")

        !pip install -q subcellular-sprawl

        import sprawl
        from sprawl import hdf5, scoring
        print("Official SPRAWL imported successfully after installation.")
        return sprawl, hdf5, scoring


def get_celltype_col(df):
    """
    Find cell-type column in molecule table.
    """
    candidates = [
        "Assigned_Xenium_Cell_Type",
        "cell_type",
        "ct_label",
        "celltype",
        "Cell_Type",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return None


def polygon_to_boundary_xy(poly):
    """
    Convert shapely Polygon/MultiPolygon to one boundary coordinate array.

    If MultiPolygon, use the largest component.
    """
    if poly is None:
        return None

    try:
        if poly.geom_type == "Polygon":
            return np.asarray(poly.exterior.coords, dtype=np.float32)

        if poly.geom_type == "MultiPolygon":
            largest = max(poly.geoms, key=lambda p: p.area)
            return np.asarray(largest.exterior.coords, dtype=np.float32)

    except Exception:
        return None

    return None


def prepare_sprawl_molecule_table(df, sample_cids, marker_genes=None):
    """
    Keep columns required for SPRAWL:
      cell_id, gene_id, x, y, cell_type
    """
    required_cols = ["cell_id", "gene_id", "x", "y"]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise KeyError(f"Molecule table missing required columns for SPRAWL: {missing}")

    ct_col = get_celltype_col(df)

    keep_cols = ["cell_id", "gene_id", "x", "y"]
    if ct_col is not None:
        keep_cols.append(ct_col)

    d = df.loc[df["cell_id"].astype(int).isin(sample_cids), keep_cols].copy()

    d["cell_id"] = d["cell_id"].astype(int)
    d["gene_id"] = d["gene_id"].astype(str)
    d["x"] = pd.to_numeric(d["x"], errors="coerce").astype(np.float32)
    d["y"] = pd.to_numeric(d["y"], errors="coerce").astype(np.float32)

    d = d.dropna(subset=["x", "y", "gene_id"])

    if marker_genes is not None:
        d = d[d["gene_id"].isin(marker_genes)].copy()

    if ct_col is not None:
        d = d.rename(columns={ct_col: "cell_type"})
        d["cell_type"] = d["cell_type"].astype(str)
    else:
        d["cell_type"] = "Unknown"

    return d


def build_sprawl_cells_from_molecules(mol_df, sample_cids, label):
    """
    Build SimpleSprawlCell objects from a molecule table.

    Each cell has:
      - one 2D z-slice named z0
      - cell boundary
      - molecule xy coordinates
      - molecule gene labels
    """
    cells = []

    grouped = mol_df.groupby("cell_id", observed=True)

    for cid, g in grouped:
        cid = int(cid)

        if cid not in sample_cids:
            continue

        if cid not in cell_polygons:
            continue

        boundary_xy = polygon_to_boundary_xy(cell_polygons[cid])

        if boundary_xy is None or len(boundary_xy) < 4:
            continue

        spot_xy = g[["x", "y"]].to_numpy(dtype=np.float32)
        spot_genes = g["gene_id"].astype(str).to_numpy()

        if len(spot_genes) < MIN_TOTAL_MOLS_PER_CELL:
            continue

        annotation = str(g["cell_type"].iloc[0]) if "cell_type" in g.columns else "Unknown"

        cells.append(
            SimpleSprawlCell(
                cell_id=cid,
                annotation=annotation,
                boundary_xy=boundary_xy,
                spot_xy=spot_xy,
                spot_genes=spot_genes,
            )
        )

    print(f"  {label}: built {len(cells):,} SPRAWL cells")
    return cells


def make_completed_sprawl_table(raw_df, imputed_df, sample_cids, marker_genes=None):
    """
    Build completed molecule table for SPRAWL:
      completed = raw observed + imputed
    only for sampled cells, to keep memory manageable.
    """
    raw_min = prepare_sprawl_molecule_table(
        raw_df,
        sample_cids=sample_cids,
        marker_genes=marker_genes,
    )

    imp_min = prepare_sprawl_molecule_table(
        imputed_df,
        sample_cids=sample_cids,
        marker_genes=marker_genes,
    )

    completed = pd.concat([raw_min, imp_min], ignore_index=True)
    return completed


def write_and_reload_sprawl_cells(cells, out_h5_path, hdf5_module):
    """
    Write SimpleSprawlCell objects to official SPRAWL HDF5,
    then read them back as official sprawl.cell.Cell objects.

    This makes the objects compatible with SPRAWL internals/multiprocessing.
    """
    if os.path.exists(out_h5_path):
        os.remove(out_h5_path)

    hdf5_module.HDF5.write_cells(cells, out_h5_path)

    h = hdf5_module.HDF5(out_h5_path)
    official_cells = h.cells()

    return official_cells


def run_sprawl_metric(cells, metric_name, scoring_module):
    """
    Run one official SPRAWL metric.
    """
    print(f"    Running official SPRAWL metric: {metric_name}")

    if metric_name in ["radial", "punctate"]:
        # These are permutation-heavy.
        # Start small. Increase NUM_ITERATIONS for final runs if needed.
        df = scoring_module.iter_scores(
            cells,
            metric=metric_name,
            processes=SPRAWL_PROCESSES,
            num_iterations=SPRAWL_NUM_ITERATIONS,
            num_pairs=SPRAWL_NUM_PAIRS,
        )
    else:
        df = scoring_module.iter_scores(
            cells,
            metric=metric_name,
            processes=SPRAWL_PROCESSES,
        )

    df["metric"] = metric_name

    return df


def run_all_sprawl_metrics_for_method(cells, method_name, scoring_module):
    """
    Run all four official SPRAWL metrics for one method.
    """
    metric_tables = []

    for metric_name in SPRAWL_METRICS:
        try:
            metric_df = run_sprawl_metric(
                cells=cells,
                metric_name=metric_name,
                scoring_module=scoring_module,
            )

            metric_df["method"] = method_name
            metric_tables.append(metric_df)

            print(f"      {metric_name}: {len(metric_df):,} rows")

        except Exception as e:
            print(f"      WARNING: SPRAWL metric {metric_name} failed for {method_name}: {e}")

    if len(metric_tables) == 0:
        return pd.DataFrame()

    out = pd.concat(metric_tables, ignore_index=True)
    return out


def add_sprawl_significance(score_df):
    """
    Add approximate z and p values using SPRAWL score and variance.

    SPRAWL returns score and variance per cell-gene pair.
    For per-cell pair screening:
        z = score / sqrt(variance)
    """
    df = score_df.copy()

    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df["variance"] = pd.to_numeric(df["variance"], errors="coerce")

    df["z"] = df["score"] / np.sqrt(df["variance"].replace(0, np.nan))
    df["approx_p_value"] = 2.0 * norm.sf(np.abs(df["z"]))

    df["is_significant"] = df["approx_p_value"] < SIG_THRESHOLD

    return df


def summarize_sprawl_by_method(score_df):
    """
    Method-level SPRAWL summary.
    """
    summary = (
        score_df
        .groupby(["method", "metric"], observed=True)
        .agg(
            n_scored_cell_gene_pairs=("gene", "size"),
            n_significant_pairs=("is_significant", "sum"),
            mean_score=("score", "mean"),
            median_score=("score", "median"),
            mean_num_gene_spots=("num_gene_spots", "mean"),
            median_num_gene_spots=("num_gene_spots", "median"),
        )
        .reset_index()
    )

    summary["fraction_significant"] = (
        summary["n_significant_pairs"]
        / summary["n_scored_cell_gene_pairs"].replace(0, np.nan)
    )

    return summary


def classify_sprawl_pattern(row):
    """
    Simple pattern label from metric score and significance.

    Note:
      peripheral and central are separate official metrics.
      radial and punctate are also separate official metrics.
    """
    if not bool(row["is_significant"]):
        return "non-significant"

    metric = str(row["metric"])
    score = float(row["score"])

    if metric == "peripheral":
        if score > 0:
            return "peripheral"
        return "anti-peripheral"

    if metric == "central":
        if score > 0:
            return "central"
        return "anti-central"

    if metric == "punctate":
        if score > 0:
            return "punctate"
        return "dispersed"

    if metric == "radial":
        if score > 0:
            return "radial"
        return "anti-radial"

    return "unknown"


# ------------------------------------------------------------------------------
# 6. Install/import official SPRAWL
# ------------------------------------------------------------------------------

sprawl, sprawl_hdf5, sprawl_scoring = install_and_import_sprawl()

# ------------------------------------------------------------------------------
# 7. Select cells and optional genes
# ------------------------------------------------------------------------------

print("\nSelecting sampled cells for official SPRAWL...")

raw_obs_full = mol_observed[mol_observed["status"].astype(str) == "observed"].copy()

valid_cids = sorted(
    set(raw_obs_full["cell_id"].dropna().astype(int).unique())
    & set(cell_polygons.keys())
)

rng = np.random.default_rng(RANDOM_STATE)

if len(valid_cids) > SAMPLE_CELLS:
    sample_cids = set(rng.choice(valid_cids, size=SAMPLE_CELLS, replace=False).tolist())
else:
    sample_cids = set(valid_cids)

print(f"Available valid cells: {len(valid_cids):,}")
print(f"Sampled cells        : {len(sample_cids):,}")

marker_genes = None

if RESTRICT_TO_MARKER_GENES:
    if "shared_genes" in globals():
        available_genes = set(str(g) for g in shared_genes)
        marker_genes = [g for g in candidate_marker_genes if g in available_genes]
    else:
        marker_genes = candidate_marker_genes

    print(f"Restricting to marker genes: {marker_genes}")

# ------------------------------------------------------------------------------
# 8. Build method-specific SPRAWL molecule tables
# ------------------------------------------------------------------------------

print("\nPreparing molecule tables for official SPRAWL...")

raw_sprawl_df = prepare_sprawl_molecule_table(
    raw_obs_full,
    sample_cids=sample_cids,
    marker_genes=marker_genes,
)

learned_completed_df = make_completed_sprawl_table(
    raw_obs_full,
    learned_imputed_df,
    sample_cids=sample_cids,
    marker_genes=marker_genes,
)

gene_emp_completed_df = make_completed_sprawl_table(
    raw_obs_full,
    gene_emp_imputed_df,
    sample_cids=sample_cids,
    marker_genes=marker_genes,
)

ct_gene_emp_completed_df = make_completed_sprawl_table(
    raw_obs_full,
    ct_gene_emp_imputed_df,
    sample_cids=sample_cids,
    marker_genes=marker_genes,
)

spatial_knn_completed_df = make_completed_sprawl_table(
    raw_obs_full,
    spatial_knn_emp_imputed_df,
    sample_cids=sample_cids,
    marker_genes=marker_genes,
)

method_molecule_tables = {
    "Raw observed": raw_sprawl_df,
    "Learned 9E completed": learned_completed_df,
    "Gene empirical completed": gene_emp_completed_df,
    "Cell-type gene empirical completed": ct_gene_emp_completed_df,
    "Spatial-kNN empirical completed": spatial_knn_completed_df,
}

for method_name, df in method_molecule_tables.items():
    print(f"  {method_name:35s}: {len(df):,} molecules")

# ------------------------------------------------------------------------------
# Compatibility alias for output directory naming
# ------------------------------------------------------------------------------

# Some parts of the cell may use SPRAWL_OFFICIAL_DIR,
# while the earlier path variable is OFFICIAL_SPRAWL_DIR.
# Make both names point to the same folder.
SPRAWL_OFFICIAL_DIR = OFFICIAL_SPRAWL_DIR

print(f"SPRAWL_OFFICIAL_DIR alias set to: {SPRAWL_OFFICIAL_DIR}")

# ------------------------------------------------------------------------------
# 9. Build/write/read official SPRAWL cells and run metrics
# ------------------------------------------------------------------------------

print("\nBuilding official SPRAWL HDF5 files and running metrics...")

all_score_tables = []
h5_manifest_rows = []

for method_name, mol_df in method_molecule_tables.items():
    print("\n" + "-" * 90)
    print(f"Method: {method_name}")
    print("-" * 90)

    simple_cells = build_sprawl_cells_from_molecules(
        mol_df=mol_df,
        sample_cids=sample_cids,
        label=method_name,
    )

    if len(simple_cells) == 0:
        print(f"  WARNING: no SPRAWL cells for {method_name}; skipping.")
        continue

    safe_method = (
        method_name
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
    )

    h5_path = os.path.join(
        SPRAWL_OFFICIAL_DIR,
        f"{RUN_NAME}_official_sprawl_{safe_method}.h5"
    )

    official_cells = write_and_reload_sprawl_cells(
        cells=simple_cells,
        out_h5_path=h5_path,
        hdf5_module=sprawl_hdf5,
    )

    print(f"  Wrote/read official SPRAWL HDF5: {h5_path}")
    print(f"  Official SPRAWL cells loaded: {len(official_cells):,}")

    h5_manifest_rows.append({
        "method": method_name,
        "h5_path": h5_path,
        "n_cells": len(official_cells),
        "n_molecules_in_input": int(len(mol_df)),
    })

    score_df = run_all_sprawl_metrics_for_method(
        cells=official_cells,
        method_name=method_name,
        scoring_module=sprawl_scoring,
    )

    if len(score_df) == 0:
        print(f"  WARNING: no SPRAWL scores generated for {method_name}.")
        continue

    all_score_tables.append(score_df)

    # Free method-specific objects.
    del simple_cells, official_cells, score_df
    gc.collect()

if len(all_score_tables) == 0:
    raise RuntimeError("Official SPRAWL produced no score tables. Check package/API/input format.")

official_sprawl_scores_df = pd.concat(all_score_tables, ignore_index=True)

print("\nOfficial SPRAWL raw score table:")
display(official_sprawl_scores_df.head(20))
print(f"official_sprawl_scores_df shape: {official_sprawl_scores_df.shape}")

# ------------------------------------------------------------------------------
# 10. Add approximate significance and pattern labels
# ------------------------------------------------------------------------------

official_sprawl_scores_df = add_sprawl_significance(official_sprawl_scores_df)
official_sprawl_scores_df["pattern_call"] = official_sprawl_scores_df.apply(
    classify_sprawl_pattern,
    axis=1,
)

print("\nOfficial SPRAWL score table with approximate significance:")
display(official_sprawl_scores_df.head(20))

# ------------------------------------------------------------------------------
# 11. Summaries
# ------------------------------------------------------------------------------

official_method_summary_df = summarize_sprawl_by_method(official_sprawl_scores_df)

print("\nOfficial SPRAWL method summary:")
display(official_method_summary_df)

pattern_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric", "pattern_call"], observed=True)
    .size()
    .reset_index(name="n_pairs")
)

pattern_summary_df["fraction_within_method_metric"] = (
    pattern_summary_df["n_pairs"]
    / pattern_summary_df.groupby(["method", "metric"])["n_pairs"].transform("sum")
)

print("\nOfficial SPRAWL pattern summary:")
display(pattern_summary_df)

# Gene-level gain: Learned 9E completed vs Raw observed.
raw_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Raw observed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

learned_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Learned 9E completed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

raw_gene_metric = (
    raw_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("raw_significant_cells")
)

learned_gene_metric = (
    learned_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("learned_significant_cells")
)

official_gene_gain_df = pd.concat(
    [raw_gene_metric, learned_gene_metric],
    axis=1,
).fillna(0).astype(int).reset_index()

official_gene_gain_df["gain_significant_cells"] = (
    official_gene_gain_df["learned_significant_cells"]
    - official_gene_gain_df["raw_significant_cells"]
)

official_gene_gain_df = official_gene_gain_df.sort_values(
    "gain_significant_cells",
    ascending=False,
).reset_index(drop=True)

print("\nTop genes by gain in significant official SPRAWL-localized cells:")
display(official_gene_gain_df.head(30))

h5_manifest_df = pd.DataFrame(h5_manifest_rows)

# ------------------------------------------------------------------------------
# 12. Visualization
# ------------------------------------------------------------------------------

print("\nGenerating official SPRAWL summary figure...")

plot_methods = [
    "Raw observed",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

plot_metrics = SPRAWL_METRICS

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for ax, metric in zip(axes, plot_metrics):
    sub = official_sprawl_scores_df[
        official_sprawl_scores_df["metric"] == metric
    ].copy()

    for method in plot_methods:
        vals = sub[sub["method"] == method]["score"].dropna()

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.6,
            label=method,
        )

    ax.axvline(0, linestyle="--", linewidth=1.0)
    ax.set_title(f"Official SPRAWL {metric} score distribution")
    ax.set_xlabel("SPRAWL score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved official SPRAWL figure:")
print(f"  {OFFICIAL_SPRAWL_FIG_PATH}")

# Heatmap of scorable pairs per method/metric.
scorable_matrix = official_method_summary_df.pivot_table(
    index="metric",
    columns="method",
    values="n_scored_cell_gene_pairs",
    aggfunc="sum",
).reindex(index=SPRAWL_METRICS, columns=plot_methods)

fig, ax = plt.subplots(figsize=(12, 5))

arr = scorable_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    cmap="viridis",
)

ax.set_title("Official SPRAWL: scorable cell-gene pairs")
ax.set_xlabel("Method")
ax.set_ylabel("Metric")

ax.set_xticks(np.arange(len(scorable_matrix.columns)))
ax.set_xticklabels(scorable_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(scorable_matrix.index)))
ax.set_yticklabels(scorable_matrix.index)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{int(val):,}", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Scored pairs")

plt.tight_layout()

OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH = os.path.join(
    SPRAWL_OFFICIAL_DIR,
    f"{RUN_NAME}_official_sprawl_scorable_pairs_heatmap.png"
)

plt.savefig(OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved scorable-pairs heatmap:")
print(f"  {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------------------------

official_sprawl_scores_df.to_csv(OFFICIAL_SPRAWL_SCORES_PATH, index=False)
official_method_summary_df.to_csv(OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH, index=False)
official_gene_gain_df.to_csv(OFFICIAL_SPRAWL_GENE_GAIN_PATH, index=False)
pattern_summary_df.to_csv(OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH, index=False)
h5_manifest_df.to_csv(OFFICIAL_SPRAWL_H5_MANIFEST_PATH, index=False)

print("\nSaved official SPRAWL outputs:")
print(f"  Cell-gene scores     : {OFFICIAL_SPRAWL_SCORES_PATH}")
print(f"  Method summary       : {OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH}")
print(f"  Gene gain table      : {OFFICIAL_SPRAWL_GENE_GAIN_PATH}")
print(f"  Pattern summary      : {OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH}")
print(f"  HDF5 manifest        : {OFFICIAL_SPRAWL_H5_MANIFEST_PATH}")
print(f"  Score figure         : {OFFICIAL_SPRAWL_FIG_PATH}")
print(f"  Scorable heatmap     : {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E completed has more scorable cell-gene pairs than raw.")
print("  Good sign 2: Learned 9E completed has more significant localized pairs than raw.")
print("  Good sign 3: Learned 9E score distributions are not collapsed to only one pattern.")
print("  Good sign 4: Compare learned 9E with empirical baselines across all four official SPRAWL metrics.")
print("  Caution 1: radial and punctate are permutation-heavy; increase SPRAWL_NUM_ITERATIONS for final reporting.")
print("  Caution 2: Xenium z is collapsed to one 2D z-slice here, so this is a 2D SPRAWL analysis.")

print("\n" + "=" * 100)
print("CELL COMPLETE — Official SPRAWL four-metric localization validation")
print("=" * 100)

gc.collect()

In [ ]:
# ==============================================================================
# CELL 11B — Continue official SPRAWL post-processing after norm import error
#
# Use this if official_sprawl_scores_df already exists in memory.
# It fixes:
#   NameError: name 'norm' is not defined
#
# Do NOT rerun official SPRAWL scoring if official_sprawl_scores_df already exists.
# GPU not needed.
# ==============================================================================

import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

print("=" * 100)
print("CELL 11B — Continue official SPRAWL post-processing")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "CHECKPOINT_DIR",
    "RUN_NAME",
    "official_sprawl_scores_df",
]

for v in required_vars:
    if v not in globals():
        raise NameError(
            f"Missing required variable: {v}. "
            "If official_sprawl_scores_df is missing, you need to rerun the official SPRAWL cell."
        )

# ------------------------------------------------------------------------------
# 1. Robust output paths
# ------------------------------------------------------------------------------

# Your cell used OFFICIAL_SPRAWL_DIR.
# Some previous code snippets also used SPRAWL_OFFICIAL_DIR.
# Keep both aliases valid.
if "OFFICIAL_SPRAWL_DIR" not in globals():
    OFFICIAL_SPRAWL_DIR = os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_official_sprawl_validation"
    )

SPRAWL_OFFICIAL_DIR = OFFICIAL_SPRAWL_DIR
os.makedirs(OFFICIAL_SPRAWL_DIR, exist_ok=True)

OFFICIAL_SPRAWL_SCORES_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_cell_gene_scores.csv"
)

OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_method_summary.csv"
)

OFFICIAL_SPRAWL_GENE_GAIN_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_gene_gain_raw_vs_learned.csv"
)

OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_pattern_summary.csv"
)

OFFICIAL_SPRAWL_H5_MANIFEST_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_h5_manifest.csv"
)

OFFICIAL_SPRAWL_FIG_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_score_distributions.png"
)

OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH = os.path.join(
    OFFICIAL_SPRAWL_DIR,
    f"{RUN_NAME}_official_sprawl_scorable_pairs_heatmap.png"
)

print(f"OFFICIAL_SPRAWL_DIR: {OFFICIAL_SPRAWL_DIR}")
print(f"official_sprawl_scores_df shape: {official_sprawl_scores_df.shape}")

# ------------------------------------------------------------------------------
# 2. Configuration defaults
# ------------------------------------------------------------------------------

if "SIG_THRESHOLD" not in globals():
    SIG_THRESHOLD = 0.05

if "SPRAWL_METRICS" not in globals():
    SPRAWL_METRICS = ["peripheral", "central", "punctate", "radial"]

plot_methods = [
    "Raw observed",
    "Learned 9E completed",
    "Gene empirical completed",
    "Cell-type gene empirical completed",
    "Spatial-kNN empirical completed",
]

print(f"SIG_THRESHOLD: {SIG_THRESHOLD}")
print(f"SPRAWL_METRICS: {SPRAWL_METRICS}")

# ------------------------------------------------------------------------------
# 3. Clean/check official SPRAWL score table
# ------------------------------------------------------------------------------

required_cols = [
    "metric",
    "cell_id",
    "annotation",
    "num_spots",
    "gene",
    "num_gene_spots",
    "score",
    "variance",
    "method",
]

missing_cols = [c for c in required_cols if c not in official_sprawl_scores_df.columns]

if missing_cols:
    raise KeyError(f"official_sprawl_scores_df is missing required columns: {missing_cols}")

official_sprawl_scores_df = official_sprawl_scores_df.copy()

official_sprawl_scores_df["metric"] = official_sprawl_scores_df["metric"].astype(str)
official_sprawl_scores_df["gene"] = official_sprawl_scores_df["gene"].astype(str)
official_sprawl_scores_df["method"] = official_sprawl_scores_df["method"].astype(str)

official_sprawl_scores_df["score"] = pd.to_numeric(
    official_sprawl_scores_df["score"],
    errors="coerce",
)

official_sprawl_scores_df["variance"] = pd.to_numeric(
    official_sprawl_scores_df["variance"],
    errors="coerce",
)

official_sprawl_scores_df["num_gene_spots"] = pd.to_numeric(
    official_sprawl_scores_df["num_gene_spots"],
    errors="coerce",
)

official_sprawl_scores_df["num_spots"] = pd.to_numeric(
    official_sprawl_scores_df["num_spots"],
    errors="coerce",
)

# ------------------------------------------------------------------------------
# 4. Add approximate z-score, p-value, and significance
# ------------------------------------------------------------------------------

print("\nAdding approximate significance values...")

# Avoid division by zero or negative variances.
safe_variance = official_sprawl_scores_df["variance"].where(
    official_sprawl_scores_df["variance"] > 0,
    np.nan,
)

official_sprawl_scores_df["z"] = (
    official_sprawl_scores_df["score"]
    / np.sqrt(safe_variance)
)

official_sprawl_scores_df["approx_p_value"] = 2.0 * norm.sf(
    np.abs(official_sprawl_scores_df["z"].astype(float))
)

official_sprawl_scores_df["is_significant"] = (
    official_sprawl_scores_df["approx_p_value"] < SIG_THRESHOLD
)

print("Added columns:")
print("  z")
print("  approx_p_value")
print("  is_significant")

display(official_sprawl_scores_df.head(20))

# ------------------------------------------------------------------------------
# 5. Pattern classification
# ------------------------------------------------------------------------------

def classify_sprawl_pattern(row):
    """
    Simple pattern label from official SPRAWL metric score and approximate significance.
    """
    if not bool(row["is_significant"]):
        return "non-significant"

    metric = str(row["metric"])
    score = float(row["score"])

    if metric == "peripheral":
        return "peripheral" if score > 0 else "anti-peripheral"

    if metric == "central":
        return "central" if score > 0 else "anti-central"

    if metric == "punctate":
        return "punctate" if score > 0 else "dispersed"

    if metric == "radial":
        return "radial" if score > 0 else "anti-radial"

    return "unknown"

official_sprawl_scores_df["pattern_call"] = official_sprawl_scores_df.apply(
    classify_sprawl_pattern,
    axis=1,
)

print("\nPattern-call preview:")
display(
    official_sprawl_scores_df[
        ["method", "metric", "cell_id", "gene", "score", "variance", "approx_p_value", "is_significant", "pattern_call"]
    ].head(20)
)

# ------------------------------------------------------------------------------
# 6. Method-level summary
# ------------------------------------------------------------------------------

official_method_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric"], observed=True)
    .agg(
        n_scored_cell_gene_pairs=("gene", "size"),
        n_significant_pairs=("is_significant", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        mean_abs_score=("score", lambda x: float(np.nanmean(np.abs(x)))),
        median_abs_score=("score", lambda x: float(np.nanmedian(np.abs(x)))),
        mean_num_gene_spots=("num_gene_spots", "mean"),
        median_num_gene_spots=("num_gene_spots", "median"),
    )
    .reset_index()
)

official_method_summary_df["fraction_significant"] = (
    official_method_summary_df["n_significant_pairs"]
    / official_method_summary_df["n_scored_cell_gene_pairs"].replace(0, np.nan)
)

print("\nOfficial SPRAWL method summary:")
display(official_method_summary_df)

# ------------------------------------------------------------------------------
# 7. Pattern summary
# ------------------------------------------------------------------------------

pattern_summary_df = (
    official_sprawl_scores_df
    .groupby(["method", "metric", "pattern_call"], observed=True)
    .size()
    .reset_index(name="n_pairs")
)

pattern_summary_df["fraction_within_method_metric"] = (
    pattern_summary_df["n_pairs"]
    / pattern_summary_df.groupby(["method", "metric"])["n_pairs"].transform("sum")
)

print("\nOfficial SPRAWL pattern summary:")
display(pattern_summary_df)

# ------------------------------------------------------------------------------
# 8. Gene-level gain: Learned 9E completed vs Raw observed
# ------------------------------------------------------------------------------

raw_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Raw observed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

learned_sig = official_sprawl_scores_df[
    (official_sprawl_scores_df["method"] == "Learned 9E completed")
    & (official_sprawl_scores_df["is_significant"])
].copy()

raw_gene_metric = (
    raw_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("raw_significant_cells")
)

learned_gene_metric = (
    learned_sig
    .groupby(["metric", "gene"], observed=True)
    .size()
    .rename("learned_significant_cells")
)

official_gene_gain_df = pd.concat(
    [raw_gene_metric, learned_gene_metric],
    axis=1,
).fillna(0).astype(int).reset_index()

official_gene_gain_df["gain_significant_cells"] = (
    official_gene_gain_df["learned_significant_cells"]
    - official_gene_gain_df["raw_significant_cells"]
)

official_gene_gain_df = official_gene_gain_df.sort_values(
    "gain_significant_cells",
    ascending=False,
).reset_index(drop=True)

print("\nTop genes by gain in significant official SPRAWL-localized cells:")
display(official_gene_gain_df.head(30))

# ------------------------------------------------------------------------------
# 9. Optional HDF5 manifest
# ------------------------------------------------------------------------------

if "h5_manifest_rows" in globals():
    h5_manifest_df = pd.DataFrame(h5_manifest_rows)
else:
    h5_manifest_df = pd.DataFrame()

# ------------------------------------------------------------------------------
# 10. Visualization: score distributions
# ------------------------------------------------------------------------------

print("\nGenerating official SPRAWL score-distribution figure...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.ravel()

for ax, metric in zip(axes, SPRAWL_METRICS):
    sub = official_sprawl_scores_df[
        official_sprawl_scores_df["metric"] == metric
    ].copy()

    for method in plot_methods:
        vals = sub[sub["method"] == method]["score"].dropna()

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.6,
            label=method,
        )

    ax.axvline(0, linestyle="--", linewidth=1.0)
    ax.set_title(f"Official SPRAWL {metric} score distribution")
    ax.set_xlabel("SPRAWL score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved official SPRAWL score figure:")
print(f"  {OFFICIAL_SPRAWL_FIG_PATH}")

# ------------------------------------------------------------------------------
# 11. Heatmap: scorable pairs
# ------------------------------------------------------------------------------

scorable_matrix = official_method_summary_df.pivot_table(
    index="metric",
    columns="method",
    values="n_scored_cell_gene_pairs",
    aggfunc="sum",
)

scorable_matrix = scorable_matrix.reindex(
    index=SPRAWL_METRICS,
    columns=[m for m in plot_methods if m in scorable_matrix.columns],
)

fig, ax = plt.subplots(figsize=(12, 5))

arr = scorable_matrix.values.astype(float)

im = ax.imshow(
    arr,
    aspect="auto",
    cmap="viridis",
)

ax.set_title("Official SPRAWL: scorable cell-gene pairs")
ax.set_xlabel("Method")
ax.set_ylabel("Metric")

ax.set_xticks(np.arange(len(scorable_matrix.columns)))
ax.set_xticklabels(scorable_matrix.columns, rotation=45, ha="right")

ax.set_yticks(np.arange(len(scorable_matrix.index)))
ax.set_yticklabels(scorable_matrix.index)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{int(val):,}", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Scored pairs")

plt.tight_layout()
plt.savefig(OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved scorable-pairs heatmap:")
print(f"  {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

# ------------------------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------------------------

official_sprawl_scores_df.to_csv(OFFICIAL_SPRAWL_SCORES_PATH, index=False)
official_method_summary_df.to_csv(OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH, index=False)
official_gene_gain_df.to_csv(OFFICIAL_SPRAWL_GENE_GAIN_PATH, index=False)
pattern_summary_df.to_csv(OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH, index=False)

if len(h5_manifest_df) > 0:
    h5_manifest_df.to_csv(OFFICIAL_SPRAWL_H5_MANIFEST_PATH, index=False)

print("\nSaved official SPRAWL outputs:")
print(f"  Cell-gene scores     : {OFFICIAL_SPRAWL_SCORES_PATH}")
print(f"  Method summary       : {OFFICIAL_SPRAWL_METHOD_SUMMARY_PATH}")
print(f"  Gene gain table      : {OFFICIAL_SPRAWL_GENE_GAIN_PATH}")
print(f"  Pattern summary      : {OFFICIAL_SPRAWL_PATTERN_SUMMARY_PATH}")
if len(h5_manifest_df) > 0:
    print(f"  HDF5 manifest        : {OFFICIAL_SPRAWL_H5_MANIFEST_PATH}")
print(f"  Score figure         : {OFFICIAL_SPRAWL_FIG_PATH}")
print(f"  Scorable heatmap     : {OFFICIAL_SPRAWL_SCORABLE_HEATMAP_PATH}")

print("\nInterpretation guide:")
print("  Good sign 1: Learned 9E completed has more scorable SPRAWL cell-gene pairs than raw.")
print("  Good sign 2: Learned 9E completed has more significant localized pairs than raw.")
print("  Good sign 3: Learned 9E should not collapse all scores to only one pattern.")
print("  Good sign 4: Compare learned 9E with empirical baselines across peripheral, central, punctate, and radial.")
print("  Caution: approx_p_value is computed from score/variance; use it mainly for comparative screening.")

print("\n" + "=" * 100)
print("CELL 11B COMPLETE — Official SPRAWL post-processing fixed")
print("=" * 100)

gc.collect()

==========================THE END=============================